# Eğitim A/B testi: aynı sahne, aynı eğitici, tek tek değişiklik

App'teki **Spirula dataset + bizim trainer** yolunun eğiticisi (`feature/scene-composer` @ `163d96c`) birkaç varyantla eğitilir;
hepsi aynı kareler, aynı tohum ve **aynı saklanan fotoğraflarla** (her 10. kare, eğitimde hiç kullanılmaz) ölçülür.

| varyant | değişiklik |
|---|---|
| `baseline` | 28 Eylül IMG_5966 eğitimiyle aynı ayarlar (doğrulama: ~25,3 dB beklenir) |
| `lr_decay` | Gaussian konumlarının öğrenme oranı eğitim boyunca 100 kat azalır (standart 3DGS; eğiticimizde sabit) |
| `lr_decay_clean` | + eğitim kareleri 4K'dan 2x2 ortalamayla küçültülür ve hafifçe gürültüden arındırılır |
| `lr_decay_60k` | + 60k adım |

**Kullanım:** GPU (A100 önerilir, **High-RAM** açık) → **Run all**. Varyant başına ~10-30 dk.
Her varyant bitince sonucu Drive'a yazılır; kopma olursa Run all biten varyantları atlar.

In [ ]:
# @title Ayarlar
DATASET_ZIP = "/content/drive/MyDrive/GaussianTests/inputs/IMG_5966_dataset.zip"  # @param {type:"string"}
OUT_DIR = "/content/drive/MyDrive/GaussianTests/training_ab/IMG_5966"  # @param {type:"string"}
VARIANTS = "baseline,lr_decay,lr_decay_clean,lr_decay_60k"  # @param {type:"string"}
HOLDOUT_EVERY = 10   # @param {type:"integer"}
LONG_EDGE = 1920     # @param {type:"integer"}
NLM_H = 3.0          # @param {type:"number"}
STEPS_OVERRIDE = 0   # @param {type:"integer"}
SEED = 42
# STEPS_OVERRIDE > 0: hızlı deneme için her varyantın adım sayısı (60k varyantı bunun 2 katı).
# NLM_H: gürültü temizleme gücü (0 = sadece 2x2 ortalama).
VARIANTS = [v.strip() for v in VARIANTS.split(",") if v.strip()]

In [ ]:
# @title Kurulum
import os
os.environ["MAX_JOBS"] = "4"
!pip install -q "gsplat==1.5.3" "pytorch-msssim==1.0.0" plyfile ninja lpips psutil
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# @title Kodu yaz (app eğiticisi + test kodu)
import json, sys, types
from pathlib import Path

SOURCES = json.loads('{"app/backend/model/gaussian_model.py": "\\"\\"\\"Faz 4a — Eğitilebilir 3D Gaussian model.\\n\\nHer Gaussian\'ın parametreleri:\\n  - means (N, 3)        — 3D pozisyon\\n  - scales (N, 3)       — log-space, exp() ile pozitif olur\\n  - quats (N, 4)        — quaternion (w, x, y, z), forward\'da normalize\\n  - opacities (N, 1)    — logit-space, sigmoid ile [0, 1]\\n  - sh_dc (N, 1, 3)     — Spherical Harmonics DC (sabit renk bileşeni)\\n  - sh_rest (N, K, 3)   — SH yüksek frekans (K = (deg+1)² - 1)\\n\\"\\"\\"\\nfrom __future__ import annotations\\nimport torch\\nimport torch.nn as nn\\nimport numpy as np\\n\\n\\ndef _rgb_to_sh_dc(rgb: torch.Tensor) -> torch.Tensor:\\n    \\"\\"\\"[0, 1] RGB → SH DC bileşeni (degree 0).\\"\\"\\"\\n    C0 = 0.28209479177387814  # 1 / (2 * sqrt(pi))\\n    return (rgb - 0.5) / C0\\n\\n\\nclass GaussianModel(nn.Module):\\n    \\"\\"\\"Eğitilebilir 3D Gaussian Splatting modeli.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        init_points: torch.Tensor,\\n        init_colors: torch.Tensor | None = None,\\n        sh_degree: int = 3,\\n        fourier_K: int = 8,\\n    ):\\n        \\"\\"\\"\\n        Args:\\n            init_points: (N, 3) — COLMAP sparse point cloud\\n            init_colors: (N, 3) — [0, 1] başlangıç renkleri (None → gri)\\n            sh_degree:   Spherical Harmonics derecesi (0–3)\\n            fourier_K:   Per-gaussian Fourier trajectory frekans sayısı (0 = kapalı).\\n                         Default 8 → K frekans × 2 (sin/cos) × 3 axis = 48 float/gaussian.\\n                         4DGS paper (Yang et al 2024) SOTA mimari.\\n        \\"\\"\\"\\n        super().__init__()\\n        N = len(init_points)\\n        if N == 0:\\n            raise ValueError(\\"init_points boş — COLMAP rekonstrüksiyonu başarısız mı?\\")\\n\\n        self.sh_degree = sh_degree\\n        # Toplam SH katsayı sayısı (deg+1)²; rest = total - 1\\n        self.num_sh_rest = (sh_degree + 1) ** 2 - 1\\n        self.fourier_K = int(fourier_K)\\n\\n        # ----- Pozisyon -----\\n        self.means = nn.Parameter(init_points.float())                                # (N, 3)\\n\\n        # ----- Scale (log-space) -----\\n        # En yakın komşu mesafesinin yarısı ile başlat\\n        with torch.no_grad():\\n            init_scale = self._estimate_init_scale(init_points)\\n        log_scale = torch.log(init_scale).unsqueeze(-1).expand(-1, 3).clone()         # (N, 3)\\n        self.scales = nn.Parameter(log_scale)\\n\\n        # ----- Rotasyon (identity quaternion) -----\\n        quats = torch.zeros(N, 4)\\n        quats[:, 0] = 1.0\\n        self.quats = nn.Parameter(quats)                                              # (N, 4)\\n\\n        # ----- Opasite (logit-space; sigmoid(-2) ≈ 0.119) -----\\n        self.opacities = nn.Parameter(torch.full((N, 1), -2.0))                       # (N, 1)\\n\\n        # ----- Renk: SH -----\\n        if init_colors is None:\\n            init_colors = torch.full((N, 3), 0.5)\\n        sh_dc = _rgb_to_sh_dc(init_colors).unsqueeze(1)                               # (N, 1, 3)\\n        self.sh_dc = nn.Parameter(sh_dc)\\n        self.sh_rest = nn.Parameter(torch.zeros(N, self.num_sh_rest, 3))              # (N, K, 3)\\n\\n        # ----- Per-gaussian Fourier trajectory (v3.6 / Yol C) -----\\n        # Δpos(gaussian_i, t) = Σ_{k=1}^{K} [A_{ik} sin(2πkt) + B_{ik} cos(2πkt)]\\n        # Shape: (N, K, 2, 3) — son axes: [sin_coef, cos_coef] × [x, y, z]\\n        # Zero-init → başlangıçta motion yok, MLP\'ninki gibi identity davranış.\\n        if self.fourier_K > 0:\\n            self.fourier_pos_coeffs = nn.Parameter(\\n                torch.zeros(N, self.fourier_K, 2, 3)\\n            )\\n        else:\\n            # Dummy placeholder — kapalı mod için (backward compat)\\n            self.register_parameter(\\"fourier_pos_coeffs\\", None)\\n\\n        # ----- Phase 2.1 — Static/Dynamic explicit split -----\\n        # is_static=True olan Gaussian\'lar deformation BYPASS edilir\\n        # (renderda d_means=means, d_quats=quats, d_scales=get_scales).\\n        # Init: hepsi static (mask-based selection sonra promote eder).\\n        # register_buffer: nn.Module persistent ama gradient yok.\\n        self.register_buffer(\\"is_static\\", torch.ones(N, dtype=torch.bool))\\n\\n        # ----- Phase 2.4 — Background flag (distance-based split) -----\\n        # is_background=True olan Gaussian\'lar uzakta sayilir; deformation kapali,\\n        # densify pruning farkli (more permissive). flag_background_by_distance()\\n        # ile init sonrasi set edilir. Default: hepsi foreground (False).\\n        self.register_buffer(\\"is_background\\", torch.zeros(N, dtype=torch.bool))\\n\\n    # ------------------------------------------------------------------\\n    # Yardımcı: başlangıç scale\'ini KNN ile tahmin et\\n    # ------------------------------------------------------------------\\n    @staticmethod\\n    def _estimate_init_scale(points: torch.Tensor, k: int = 3) -> torch.Tensor:\\n        \\"\\"\\"En yakın k komşu ortalama mesafesi → scale (N,).\\"\\"\\"\\n        N = points.shape[0]\\n        if N <= k + 1:\\n            return torch.full((N,), 0.01)\\n\\n        # Büyük nokta bulutları için referans seti küçültülür: ölçek tahmini\\n        # yaklaşık olabilir, exact KNN gerekmez. Bellek bütçesi: hedef ~256 MB.\\n        # ref_size × 4 bytes × chunk ≤ 256 MB → ref_size ≤ 16k for chunk=4096.\\n        ref_size = min(N, 16_000)\\n        if ref_size < N:\\n            ref_idx = torch.randperm(N)[:ref_size]\\n            ref_points = points[ref_idx]\\n        else:\\n            ref_points = points\\n\\n        # Bellek dostu chunked nearest-neighbor (büyük N için).\\n        # cdist(chunk, ref_size) → float32 matrix; hedef max ~256 MB per chunk.\\n        max_bytes = 256 * 1024 * 1024  # 256 MB\\n        chunk = max(1, min(4096, max_bytes // (ref_size * 4)))\\n        dists = []\\n        for s in range(0, N, chunk):\\n            e = min(s + chunk, N)\\n            d = torch.cdist(points[s:e], ref_points)    # (chunk, ref_size)\\n            d, _ = torch.topk(d, k=k + 1, largest=False)  # +1: kendisi\\n            dists.append(d[:, 1:].mean(dim=1))           # kendini at\\n        mean_dist = torch.cat(dists, dim=0)              # (N,)\\n        # Scale = mesafenin yarısı (komşu Gaussian\'larla hafif örtüşme)\\n        return torch.clamp(mean_dist * 0.5, min=1e-6, max=1.0)\\n\\n    # ------------------------------------------------------------------\\n    # Aktive edilmiş paramlara erişim\\n    # ------------------------------------------------------------------\\n    @property\\n    def get_scales(self) -> torch.Tensor:\\n        return torch.exp(self.scales)              # (N, 3) > 0\\n\\n    @property\\n    def get_opacities(self) -> torch.Tensor:\\n        return torch.sigmoid(self.opacities)       # (N, 1) ∈ [0, 1]\\n\\n    @property\\n    def get_quats(self) -> torch.Tensor:\\n        return torch.nn.functional.normalize(self.quats, dim=-1)\\n\\n    @property\\n    def get_colors(self) -> torch.Tensor:\\n        \\"\\"\\"SH katsayıları (N, K_total, 3).\\"\\"\\"\\n        return torch.cat([self.sh_dc, self.sh_rest], dim=1)\\n\\n    @property\\n    def num_points(self) -> int:\\n        return int(self.means.shape[0])\\n\\n    # ------------------------------------------------------------------\\n    # Phase 2.1 — Static/Dynamic API\\n    # ------------------------------------------------------------------\\n    @torch.no_grad()\\n    def promote_dynamic(self, dynamic_mask: torch.Tensor) -> int:\\n        \\"\\"\\"Verilen bool mask\'i (N,) True olan Gaussian\'lari dynamic flag\'le.\\n\\n        Returns:\\n            promoted: dynamic\'e gecirilen Gaussian sayisi\\n        \\"\\"\\"\\n        if not hasattr(self, \\"is_static\\"):\\n            return 0\\n        if dynamic_mask.shape[0] != self.num_points:\\n            raise ValueError(\\n                f\\"dynamic_mask boyutu ({dynamic_mask.shape[0]}) num_points ({self.num_points}) ile uyumsuz\\"\\n            )\\n        prev_static = self.is_static.clone()\\n        self.is_static = self.is_static & ~dynamic_mask.to(self.is_static.device)\\n        promoted = int((prev_static & ~self.is_static).sum().item())\\n        return promoted\\n\\n    @property\\n    def num_static(self) -> int:\\n        return int(self.is_static.sum().item()) if hasattr(self, \\"is_static\\") else self.num_points\\n\\n    @property\\n    def num_dynamic(self) -> int:\\n        return self.num_points - self.num_static\\n\\n    # ------------------------------------------------------------------\\n    # Phase 2.4 — Background API\\n    # ------------------------------------------------------------------\\n    @torch.no_grad()\\n    def flag_background_by_distance(\\n        self, scene_center: torch.Tensor, scene_extent: float, ratio: float = 2.0\\n    ) -> int:\\n        \\"\\"\\"Scene merkezinden \'ratio * scene_extent\' uzakta olanlari background flag\'le.\\n\\n        Args:\\n            scene_center: (3,) world-space scene center\\n            scene_extent: scene scale (look-points\'ten tahmin)\\n            ratio:        threshold = ratio * scene_extent (default 2.0)\\n\\n        Returns:\\n            n_bg: background olarak isaretlenen Gaussian sayisi\\n        \\"\\"\\"\\n        if not hasattr(self, \\"is_background\\"):\\n            return 0\\n        sc = scene_center.to(self.means.device).reshape(1, 3)\\n        d = (self.means.detach() - sc).norm(dim=-1)  # (N,)\\n        threshold = ratio * scene_extent\\n        bg_mask = d > threshold\\n        self.is_background = bg_mask.to(self.is_background.device)\\n        return int(bg_mask.sum().item())\\n\\n    @property\\n    def num_background(self) -> int:\\n        return int(self.is_background.sum().item()) if hasattr(self, \\"is_background\\") else 0\\n\\n    # ------------------------------------------------------------------\\n    # Density control: prune\\n    # ------------------------------------------------------------------\\n    @torch.no_grad()\\n    def prune_gaussians(self, min_opacity: float = 0.005, max_scale: float = 0.1) -> int:\\n        \\"\\"\\"Düşük opasite veya çok büyük Gaussian\'ları sil. Silinen sayıyı döner.\\"\\"\\"\\n        keep = (\\n            (self.get_opacities.squeeze(-1) > min_opacity) &\\n            (self.get_scales.max(dim=-1).values < max_scale)\\n        )\\n        n_pruned = int((~keep).sum().item())\\n        if n_pruned > 0:\\n            self._apply_mask(keep)\\n        return n_pruned\\n\\n    @torch.no_grad()\\n    def _apply_mask(self, mask: torch.Tensor) -> None:\\n        \\"\\"\\"Bool maskesi ile tüm parametreleri filtrele (in-place yeniden bağla).\\"\\"\\"\\n        self.means     = nn.Parameter(self.means[mask].contiguous())\\n        self.scales    = nn.Parameter(self.scales[mask].contiguous())\\n        self.quats     = nn.Parameter(self.quats[mask].contiguous())\\n        self.opacities = nn.Parameter(self.opacities[mask].contiguous())\\n        self.sh_dc     = nn.Parameter(self.sh_dc[mask].contiguous())\\n        self.sh_rest   = nn.Parameter(self.sh_rest[mask].contiguous())\\n        # v3.6: fourier trajectory senkronu\\n        if self.fourier_pos_coeffs is not None:\\n            self.fourier_pos_coeffs = nn.Parameter(\\n                self.fourier_pos_coeffs[mask].contiguous()\\n            )\\n        # Phase 2.1 — is_static buffer senkronu (prune)\\n        if hasattr(self, \\"is_static\\"):\\n            self.is_static = self.is_static[mask].contiguous()\\n        # Phase 2.4 — is_background buffer senkronu\\n        if hasattr(self, \\"is_background\\"):\\n            self.is_background = self.is_background[mask].contiguous()\\n\\n    # ------------------------------------------------------------------\\n    # Optimizer-aware density ops (T4 — perf fix)\\n    # ------------------------------------------------------------------\\n    # Per-gauss parametrelerin listesi. _apply_mask_keep_optimizer ve\\n    # _append_keep_optimizer bu listeyi tarar. Fourier dahil edilirse runtime\\n    # eklenir.\\n    _PER_GAUSS_ATTRS = (\\"means\\", \\"scales\\", \\"quats\\", \\"opacities\\", \\"sh_dc\\", \\"sh_rest\\")\\n\\n    def _per_gauss_attrs(self) -> tuple[str, ...]:\\n        attrs = list(self._PER_GAUSS_ATTRS)\\n        if self.fourier_pos_coeffs is not None:\\n            attrs.append(\\"fourier_pos_coeffs\\")\\n        return tuple(attrs)\\n\\n    @staticmethod\\n    def _swap_param_in_optimizer(optimizer, old_p: nn.Parameter, new_p: nn.Parameter) -> dict | None:\\n        \\"\\"\\"Pop optimizer state for `old_p`, return it; replace old_p with new_p in\\n        param_groups. Caller is responsible for re-attaching the (possibly mutated)\\n        state under `optimizer.state[new_p]` after mutating buffer shapes.\\"\\"\\"\\n        state = optimizer.state.pop(old_p, None)\\n        for group in optimizer.param_groups:\\n            for i, p in enumerate(group[\\"params\\"]):\\n                if p is old_p:\\n                    group[\\"params\\"][i] = new_p\\n        return state\\n\\n    @torch.no_grad()\\n    def _apply_mask_keep_optimizer(self, mask: torch.Tensor, optimizer) -> None:\\n        \\"\\"\\"Per-gauss param\'ları mask ile filtrele AND optimizer Adam state\'ini\\n        ayni mask ile filtrele.\\n\\n        INRIA 3DGS reference impl pattern: density step\'lerde Adam\'ın\\n        exp_avg / exp_avg_sq buffer\'ları parametre boyutuyla in-sync kalmali\\n        ki momentum kaybolmasin. Onceki kod _build_optimizer() ile her density\\n        step\'te optimizer\'i sifirdan kuruyordu — Adam momentum 200+ kez\\n        sifirlaniyor → convergence ciddi yavasliyordu.\\n        \\"\\"\\"\\n        for attr in self._per_gauss_attrs():\\n            old_p = getattr(self, attr)\\n            new_data = old_p.data[mask].contiguous()\\n            new_p = nn.Parameter(new_data)\\n            state = self._swap_param_in_optimizer(optimizer, old_p, new_p)\\n            if state is not None:\\n                if \\"exp_avg\\" in state:\\n                    state[\\"exp_avg\\"] = state[\\"exp_avg\\"][mask].contiguous()\\n                if \\"exp_avg_sq\\" in state:\\n                    state[\\"exp_avg_sq\\"] = state[\\"exp_avg_sq\\"][mask].contiguous()\\n                optimizer.state[new_p] = state\\n            setattr(self, attr, new_p)\\n\\n        # Buffer\'lar (gradient yok ama N ile sync olmali)\\n        if hasattr(self, \\"is_static\\"):\\n            self.is_static = self.is_static[mask].contiguous()\\n        if hasattr(self, \\"is_background\\"):\\n            self.is_background = self.is_background[mask].contiguous()\\n\\n    @torch.no_grad()\\n    def _append_keep_optimizer(\\n        self,\\n        optimizer,\\n        new_means: torch.Tensor,\\n        new_scales: torch.Tensor,\\n        new_quats: torch.Tensor,\\n        new_opacities: torch.Tensor,\\n        new_sh_dc: torch.Tensor,\\n        new_sh_rest: torch.Tensor,\\n        new_fourier_coeffs: torch.Tensor | None = None,\\n    ) -> None:\\n        \\"\\"\\"Yeni gauss\'ları concat AND optimizer Adam state\'i icin sifir-padded uzat.\\"\\"\\"\\n        device = self.means.device\\n        new_data_map = {\\n            \\"means\\": new_means,\\n            \\"scales\\": new_scales,\\n            \\"quats\\": new_quats,\\n            \\"opacities\\": new_opacities,\\n            \\"sh_dc\\": new_sh_dc,\\n            \\"sh_rest\\": new_sh_rest,\\n        }\\n        if self.fourier_pos_coeffs is not None:\\n            if new_fourier_coeffs is None:\\n                n_new = new_means.shape[0]\\n                new_fourier_coeffs = torch.zeros(\\n                    n_new, self.fourier_K, 2, 3, device=device\\n                )\\n            new_data_map[\\"fourier_pos_coeffs\\"] = new_fourier_coeffs\\n\\n        for attr in self._per_gauss_attrs():\\n            old_p = getattr(self, attr)\\n            new_chunk = new_data_map[attr].to(device)\\n            new_data = torch.cat([old_p.data, new_chunk], dim=0).contiguous()\\n            new_p = nn.Parameter(new_data)\\n            state = self._swap_param_in_optimizer(optimizer, old_p, new_p)\\n            if state is not None:\\n                n_new = new_chunk.shape[0]\\n                if \\"exp_avg\\" in state:\\n                    ea = state[\\"exp_avg\\"]\\n                    pad = torch.zeros(\\n                        (n_new,) + ea.shape[1:], device=ea.device, dtype=ea.dtype\\n                    )\\n                    state[\\"exp_avg\\"] = torch.cat([ea, pad], dim=0).contiguous()\\n                if \\"exp_avg_sq\\" in state:\\n                    es = state[\\"exp_avg_sq\\"]\\n                    pad = torch.zeros(\\n                        (n_new,) + es.shape[1:], device=es.device, dtype=es.dtype\\n                    )\\n                    state[\\"exp_avg_sq\\"] = torch.cat([es, pad], dim=0).contiguous()\\n                optimizer.state[new_p] = state\\n            setattr(self, attr, new_p)\\n\\n        # Buffer\'lar\\n        n_new = new_means.shape[0]\\n        if hasattr(self, \\"is_static\\"):\\n            new_static = torch.ones(n_new, dtype=torch.bool, device=device)\\n            self.is_static = torch.cat([self.is_static, new_static], dim=0).contiguous()\\n        if hasattr(self, \\"is_background\\"):\\n            new_bg = torch.zeros(n_new, dtype=torch.bool, device=device)\\n            self.is_background = torch.cat([self.is_background, new_bg], dim=0).contiguous()\\n\\n    @torch.no_grad()\\n    def prune_gaussians_keep_optimizer(\\n        self, optimizer, min_opacity: float = 0.005, max_scale: float = 0.1\\n    ) -> int:\\n        \\"\\"\\"Optimizer-aware prune. Bkz. _apply_mask_keep_optimizer.\\"\\"\\"\\n        keep = (\\n            (self.get_opacities.squeeze(-1) > min_opacity) &\\n            (self.get_scales.max(dim=-1).values < max_scale)\\n        )\\n        n_pruned = int((~keep).sum().item())\\n        if n_pruned > 0:\\n            self._apply_mask_keep_optimizer(keep, optimizer)\\n        return n_pruned\\n\\n    @torch.no_grad()\\n    def append_gaussians(\\n        self,\\n        new_means: torch.Tensor,\\n        new_scales: torch.Tensor,\\n        new_quats: torch.Tensor,\\n        new_opacities: torch.Tensor,\\n        new_sh_dc: torch.Tensor,\\n        new_sh_rest: torch.Tensor,\\n        new_fourier_coeffs: torch.Tensor | None = None,\\n    ) -> None:\\n        \\"\\"\\"Yeni Gaussian\'ları (clone/split sonucu) listenin sonuna ekle.\\"\\"\\"\\n        device = self.means.device\\n        cat = lambda a, b: torch.cat([a, b.to(device)], dim=0).contiguous()\\n        self.means     = nn.Parameter(cat(self.means, new_means))\\n        self.scales    = nn.Parameter(cat(self.scales, new_scales))\\n        self.quats     = nn.Parameter(cat(self.quats, new_quats))\\n        self.opacities = nn.Parameter(cat(self.opacities, new_opacities))\\n        self.sh_dc     = nn.Parameter(cat(self.sh_dc, new_sh_dc))\\n        self.sh_rest   = nn.Parameter(cat(self.sh_rest, new_sh_rest))\\n        # v3.6: fourier trajectory senkronu\\n        if self.fourier_pos_coeffs is not None:\\n            n_new = new_means.shape[0]\\n            if new_fourier_coeffs is None:\\n                # Yeni gaussian için trajectory zero-init (motion yok, öğrensin)\\n                new_fourier_coeffs = torch.zeros(\\n                    n_new, self.fourier_K, 2, 3, device=device\\n                )\\n            self.fourier_pos_coeffs = nn.Parameter(\\n                cat(self.fourier_pos_coeffs, new_fourier_coeffs)\\n            )\\n        # Phase 2.1 — is_static senkronu (split/clone yeni Gaussian\'lar parent\'tan inherit)\\n        if hasattr(self, \\"is_static\\"):\\n            n_new = new_means.shape[0]\\n            # Default: yeni Gaussian static (caller motion mask\'ten promote etmeli)\\n            new_static = torch.ones(n_new, dtype=torch.bool, device=device)\\n            self.is_static = torch.cat([self.is_static, new_static], dim=0).contiguous()\\n        # Phase 2.4 — is_background senkronu (yeni gauss default foreground)\\n        if hasattr(self, \\"is_background\\"):\\n            n_new = new_means.shape[0]\\n            new_bg = torch.zeros(n_new, dtype=torch.bool, device=device)\\n            self.is_background = torch.cat([self.is_background, new_bg], dim=0).contiguous()\\n\\n    # ------------------------------------------------------------------\\n    # Edit / refit helpers (Task C2 — object deletion)\\n    # ------------------------------------------------------------------\\n\\n    @torch.no_grad()\\n    def filter_in_place(self, keep: torch.Tensor) -> None:\\n        \\"\\"\\"Drop Gaussians where *keep* is False. Modifies all parameter tensors.\\n\\n        Wraps the existing ``_apply_mask`` helper which already handles the\\n        full attribute list (``means``, ``scales``, ``quats``, ``opacities``,\\n        ``sh_dc``, ``sh_rest``, ``fourier_pos_coeffs``, ``is_static``,\\n        ``is_background``).\\n\\n        After this call the optimizer state for these params is stale — the\\n        caller is responsible for rebuilding the optimizer.  The edit-refit\\n        flow always rebuilds via ``Trainer4DGS.__init__``, so this is fine.\\n        \\"\\"\\"\\n        assert keep.dtype == torch.bool, \\"keep must be a bool tensor\\"\\n        assert keep.shape[0] == self.num_points, (\\n            f\\"keep length {keep.shape[0]} != num_points {self.num_points}\\"\\n        )\\n        if (~keep).any():\\n            self._apply_mask(keep)\\n\\n    def set_freeze_mask(self, freeze_mask: torch.Tensor) -> None:\\n        \\"\\"\\"Mark Gaussians as frozen (no gradient updates this iter).\\n\\n        Used by edit-refit to restrict gradient updates to the affected zone.\\n        Stored as a plain attribute (not register_buffer) since GaussianModel\\n        is an nn.Module but the mask is ephemeral and should not be\\n        checkpointed.  Trainer4DGS reads ``gs._freeze_mask`` just before\\n        ``optimizer.step()`` and zeros the gradients for frozen Gaussians.\\n        \\"\\"\\"\\n        assert freeze_mask.dtype == torch.bool, \\"freeze_mask must be a bool tensor\\"\\n        assert freeze_mask.shape[0] == self.num_points, (\\n            f\\"freeze_mask length {freeze_mask.shape[0]} != num_points {self.num_points}\\"\\n        )\\n        self._freeze_mask: torch.Tensor = freeze_mask\\n\\n    # ------------------------------------------------------------------\\n    # I/O\\n    # ------------------------------------------------------------------\\n    def state_for_save(self) -> dict:\\n        \\"\\"\\"Checkpoint için (CPU) sözlük.\\"\\"\\"\\n        return {k: v.detach().cpu() for k, v in self.state_dict().items()} | {\\n            \\"sh_degree\\": self.sh_degree,\\n            \\"num_points\\": self.num_points,\\n            \\"fourier_K\\": self.fourier_K,\\n        }\\n\\n    @classmethod\\n    def from_checkpoint(cls, ckpt: dict) -> \\"GaussianModel\\":\\n        \\"\\"\\"Boş bir model oluştur ve ağırlıkları yükle.\\"\\"\\"\\n        N = int(ckpt[\\"num_points\\"])\\n        sh_deg = int(ckpt.get(\\"sh_degree\\", 3))\\n        fourier_K = int(ckpt.get(\\"fourier_K\\", 0))  # Eski ckpt\'lerde yok → 0\\n        # Dummy noktalar — sonra state_dict ile üzerine yazılacak\\n        dummy = torch.zeros(N, 3)\\n        m = cls(dummy, sh_degree=sh_deg, fourier_K=fourier_K)\\n        # state_dict yalnızca tensor key\'leri içersin\\n        sd = {k: v for k, v in ckpt.items() if isinstance(v, torch.Tensor)}\\n        # Backward compat: eski ckpt\'lerde fourier_pos_coeffs yok\\n        m.load_state_dict(sd, strict=False)\\n        return m\\n\\n\\n# ---------------------------------------------------------------------------\\n# Hızlı sağlık testi (CPU\'da çalışır)\\n# ---------------------------------------------------------------------------\\nif __name__ == \\"__main__\\":\\n    pts = torch.randn(1000, 3)\\n    rgb = torch.rand(1000, 3)\\n    m = GaussianModel(pts, init_colors=rgb, sh_degree=3)\\n    print(f\\"Means:     {m.means.shape}\\")\\n    print(f\\"Scales:    {m.get_scales.shape}, range=[{m.get_scales.min():.4f}, {m.get_scales.max():.4f}]\\")\\n    print(f\\"Opacities: {m.get_opacities.shape}, mean={m.get_opacities.mean():.4f}\\")\\n    print(f\\"SH dc:     {m.sh_dc.shape}\\")\\n    print(f\\"SH rest:   {m.sh_rest.shape}  (K = (deg+1)²-1 = {m.num_sh_rest})\\")\\n    # Prune test\\n    n_pruned = m.prune_gaussians(min_opacity=0.5)\\n    print(f\\"Prune (min_opacity=0.5) → {n_pruned} silindi, kalan: {m.num_points}\\")\\n", "app/backend/model/deformation.py": "\\"\\"\\"Faz 4b — HexPlane + MLP deformation field.\\n\\nStatik Gaussian\'lar için zaman-bağımlı (Δpos, Δquat, Δscale) sapması öğrenir.\\nHexPlane (4DGaussians paper) 4D uzayı 6 düzleme ayırarak O(n²) bellek kullanır.\\n\\nv2 — genişletilmiş mimari:\\n  - Varsayılan mlp_width 256 → 512, mlp_depth 2 → 4 hidden layer\\n  - HexPlane resolution 64 → 96, feat_dim 32 → 48\\n  - Zaman t için Fourier positional encoding (num_time_freqs varsayılan 6)\\n    Bu MLP\'nin zaman boyutunu daha iyi ayırt etmesini sağlar —\\n    linear t ile MLP genelde zamansız davranır.\\n\\"\\"\\"\\nfrom __future__ import annotations\\nimport math\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\n\\nclass HexPlane(nn.Module):\\n    \\"\\"\\"6 düzlem: XY, XZ, YZ, XT, YT, ZT — feature dim\'i concat.\\"\\"\\"\\n\\n    def __init__(self, resolution: int = 96, feat_dim: int = 48):\\n        super().__init__()\\n        self.resolution = resolution\\n        self.feat_dim = feat_dim\\n        self.planes = nn.ParameterList([\\n            nn.Parameter(torch.randn(1, feat_dim, resolution, resolution) * 0.01)\\n            for _ in range(6)\\n        ])\\n\\n    @staticmethod\\n    def _sample_plane(plane: torch.Tensor, c1: torch.Tensor, c2: torch.Tensor) -> torch.Tensor:\\n        grid = torch.stack([c1, c2], dim=-1).view(1, 1, -1, 2)\\n        feat = F.grid_sample(plane, grid, mode=\\"bilinear\\",\\n                             align_corners=True, padding_mode=\\"border\\")\\n        return feat.squeeze(0).squeeze(1).T   # (N, feat_dim)\\n\\n    def forward(self, x, y, z, t):\\n        feats = [\\n            self._sample_plane(self.planes[0], x, y),  # XY\\n            self._sample_plane(self.planes[1], x, z),  # XZ\\n            self._sample_plane(self.planes[2], y, z),  # YZ\\n            self._sample_plane(self.planes[3], x, t),  # XT\\n            self._sample_plane(self.planes[4], y, t),  # YT\\n            self._sample_plane(self.planes[5], z, t),  # ZT\\n        ]\\n        return torch.cat(feats, dim=-1)\\n\\n\\nclass MultiResHexPlane(nn.Module):\\n    \\"\\"\\"Phase 2.5 — Multi-resolution HexPlane stack (4DGaussians-style).\\n\\n    Multiple HexPlane scale\'lerinde sample edip feature\'lari concat eder.\\n    Coarse plane düşük frekansli motion\'i ogrenir, fine plane yüksek detay.\\n\\n    Args:\\n        resolutions: list of int, orn [12, 24, 48, 96]\\n        feat_dim:    her resolution icin feat dim (toplam = sum * 6 plane)\\n    \\"\\"\\"\\n\\n    def __init__(self, resolutions: list[int] = (24, 48, 96), feat_dim: int = 24):\\n        super().__init__()\\n        self.resolutions = list(resolutions)\\n        self.feat_dim = feat_dim\\n        self.planes_list = nn.ModuleList([\\n            HexPlane(resolution=r, feat_dim=feat_dim) for r in self.resolutions\\n        ])\\n\\n    @property\\n    def total_feat_dim(self) -> int:\\n        # Her HexPlane.forward 6 plane\'i concat -> feat_dim * 6\\n        # Multi-res: ayrica scale sayisi kadar\\n        return self.feat_dim * 6 * len(self.resolutions)\\n\\n    def forward(self, x, y, z, t):\\n        feats = [hp(x, y, z, t) for hp in self.planes_list]\\n        return torch.cat(feats, dim=-1)\\n\\n\\ndef fourier_encode_scalar(value: float, num_freqs: int, device) -> torch.Tensor:\\n    \\"\\"\\"\\n    [t, sin(πt), cos(πt), sin(2πt), cos(2πt), ..., sin(2^(L-1)·πt), cos(...)].\\n    Size: 1 + 2*num_freqs.\\n    \\"\\"\\"\\n    base = torch.tensor([value], device=device)\\n    encs = [base]\\n    for i in range(num_freqs):\\n        freq = (2.0 ** i) * math.pi\\n        encs.append(torch.sin(freq * base))\\n        encs.append(torch.cos(freq * base))\\n    return torch.cat(encs, dim=0)\\n\\n\\ndef decode_fourier_trajectory(\\n    coeffs: torch.Tensor,   # (N, K, 2, 3) — sin/cos × xyz\\n    t: float,               # ∈ [0, 1]\\n) -> torch.Tensor:\\n    \\"\\"\\"\\n    Per-gaussian Fourier trajectory decode.\\n    4DGS paper (Yang et al 2024) yaklaşımı: her gaussian kendi trajectory\'sini öğrenir.\\n\\n    Δpos(i, t) = Σ_{k=1}^{K} [A_{ik} · sin(2πkt) + B_{ik} · cos(2πkt)]\\n\\n    Ama DC koşulu için: t=0\'da Δpos=0 olsun istiyoruz (tüm B_k = 0 başta, öğrenilir).\\n    Yani B term\'leri t=0\'da toplamı 0 olmayabilir ama bu OK, sadece motion relative.\\n\\n    Args:\\n        coeffs: (N, K, 2, 3) tensor. son axes: [sin_coef, cos_coef] × [x, y, z]\\n        t: zaman değeri [0, 1]\\n\\n    Returns:\\n        (N, 3) Δpos tensor\\n    \\"\\"\\"\\n    if coeffs is None or coeffs.shape[1] == 0:\\n        return torch.zeros(coeffs.shape[0] if coeffs is not None else 0, 3,\\n                           device=coeffs.device if coeffs is not None else \\"cpu\\")\\n\\n    N, K, _, _ = coeffs.shape\\n    device = coeffs.device\\n    # Frekanslar: 2π·1, 2π·2, ..., 2π·K\\n    freqs = torch.arange(1, K + 1, device=device, dtype=coeffs.dtype)  # (K,)\\n    phases = 2.0 * math.pi * freqs * t                                 # (K,)\\n    sin_vals = torch.sin(phases)                                       # (K,)\\n    cos_vals = torch.cos(phases)                                       # (K,)\\n\\n    # coeffs[:, :, 0, :] → A (sin coef), shape (N, K, 3)\\n    # coeffs[:, :, 1, :] → B (cos coef), shape (N, K, 3)\\n    # Çıkış: Σ_k A_k·sin(2πkt) + B_k·cos(2πkt)  → (N, 3)\\n    delta = (\\n        (coeffs[:, :, 0, :] * sin_vals[None, :, None]).sum(dim=1) +\\n        (coeffs[:, :, 1, :] * cos_vals[None, :, None]).sum(dim=1)\\n    )\\n    return delta\\n\\n\\nclass DeformationField(nn.Module):\\n    \\"\\"\\"HexPlane + MLP → (Δpos, Δquat, Δscale). Fourier-encoded t.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        resolution: int = 96,\\n        feat_dim: int = 48,\\n        mlp_width: int = 512,\\n        mlp_depth: int = 4,         # hidden layer sayısı (önceki: implicit 2)\\n        num_time_freqs: int = 6,    # Fourier frekans sayısı (0 = kapalı)\\n        # Phase 2.5 — Multi-resolution HexPlane (default off, single-res)\\n        multires_resolutions: list | None = None,\\n        multires_feat_dim: int | None = None,\\n    ):\\n        super().__init__()\\n        self.num_time_freqs = num_time_freqs\\n\\n        # Phase 2.5 — Multi-res toggle\\n        if multires_resolutions:\\n            mr_feat = multires_feat_dim if multires_feat_dim else feat_dim\\n            self.hexplane = MultiResHexPlane(\\n                resolutions=list(multires_resolutions), feat_dim=mr_feat,\\n            )\\n            input_hexplane_dim = self.hexplane.total_feat_dim\\n            print(f\\"[deformation] MultiResHexPlane: resolutions={multires_resolutions} \\"\\n                  f\\"feat_dim={mr_feat} total={input_hexplane_dim}\\")\\n        else:\\n            self.hexplane = HexPlane(resolution, feat_dim)\\n            input_hexplane_dim = feat_dim * 6\\n\\n        # Girdi: HexPlane features + Fourier time (1 + 2*L)\\n        t_dim = 1 + 2 * num_time_freqs\\n        input_dim = input_hexplane_dim + t_dim\\n\\n        # MLP: input → width → width → ... (mlp_depth katman) → 10\\n        layers = []\\n        prev = input_dim\\n        for _ in range(mlp_depth):\\n            layers.append(nn.Linear(prev, mlp_width))\\n            layers.append(nn.SiLU())\\n            prev = mlp_width\\n        layers.append(nn.Linear(prev, 10))      # Δpos(3) + Δquat(4) + Δscale(3)\\n        self.mlp = nn.Sequential(*layers)\\n\\n        # Son katmanı sıfır ile başlat → eğitim başında identity\\n        nn.init.zeros_(self.mlp[-1].weight)\\n        nn.init.zeros_(self.mlp[-1].bias)\\n\\n    @staticmethod\\n    def _normalize_means(means: torch.Tensor, scene_extent: float) -> torch.Tensor:\\n        return torch.clamp(means / max(scene_extent, 1e-6), -1.0, 1.0)\\n\\n    def forward(\\n        self,\\n        means: torch.Tensor,        # (N, 3)\\n        t: float,                   # ∈ [0, 1]\\n        scene_extent: float = 1.0,\\n        already_normalized: bool = False,\\n    ):\\n        if not already_normalized:\\n            means = self._normalize_means(means, scene_extent)\\n\\n        x, y, z = means[:, 0], means[:, 1], means[:, 2]\\n        t_norm = t * 2.0 - 1.0  # → [-1, 1]\\n        t_vec = torch.full_like(x, t_norm)\\n\\n        # HexPlane feat (N, feat_dim*6)\\n        hex_feat = self.hexplane(x, y, z, t_vec)\\n\\n        # Fourier time encoding — aynı değer her gaussian için (N kopyası)\\n        t_enc = fourier_encode_scalar(t_norm, self.num_time_freqs, means.device)\\n        t_enc_broadcast = t_enc.unsqueeze(0).expand(means.shape[0], -1)\\n\\n        feat = torch.cat([hex_feat, t_enc_broadcast], dim=-1)\\n        delta = self.mlp(feat)\\n\\n        return delta[:, :3], delta[:, 3:7], delta[:, 7:10]\\n\\n\\n# ---------------------------------------------------------------------------\\n# Hızlı sağlık testi\\n# ---------------------------------------------------------------------------\\nif __name__ == \\"__main__\\":\\n    field = DeformationField()\\n    means = torch.randn(500, 3)\\n    dpos, dquat, dscale = field(means, t=0.5, scene_extent=2.0)\\n    n_params = sum(p.numel() for p in field.parameters())\\n    print(f\\"Default config params: {n_params:,}\\")\\n    print(f\\"Δpos max abs (zero-init): {dpos.abs().max():.6f}\\")\\n\\n    field_small = DeformationField(resolution=64, feat_dim=32, mlp_width=256, mlp_depth=2)\\n    n_small = sum(p.numel() for p in field_small.parameters())\\n    print(f\\"Previous default config params: {n_small:,}\\")\\n", "app/backend/model/renderer.py": "\\"\\"\\"Faz 5 — gsplat rasterizer wrapper.\\n\\ngsplat 1.x API\'si. with_depth=True ise tek call\'la RGB+ED (expected depth)\\nrender eder — gsplat internal olarak RGB ve depth\'i ayni geometric pass\'te\\nhesaplar; backgrounds tensor\'unu da otomatik genisletir (rendering.py:614-623).\\nIki ayri call yapan eski yol depth ON\'unu render maliyetini neredeyse iki\\nkatina cikariyordu.\\n\\"\\"\\"\\nfrom __future__ import annotations\\nimport torch\\nimport numpy as np\\n\\n\\ndef render_view(\\n    means: torch.Tensor,\\n    quats: torch.Tensor,\\n    scales: torch.Tensor,\\n    opacities: torch.Tensor,\\n    colors: torch.Tensor,\\n    K: torch.Tensor,\\n    w2c: torch.Tensor,\\n    width: int,\\n    height: int,\\n    sh_degree: int = 3,\\n    bg_color: tuple[float, float, float] = (0.0, 0.0, 0.0),\\n    with_depth: bool = False,\\n) -> tuple[torch.Tensor, torch.Tensor, dict]:\\n    \\"\\"\\"\\n    Returns:\\n        rgb:    (H, W, 3) ya da (H, W, 4) float [0, 1]  (with_depth=True ise last chan depth)\\n        alpha:  (H, W, 1)\\n        info:   gsplat\'tan dönen metadata\\n    \\"\\"\\"\\n    try:\\n        from gsplat import rasterization\\n    except ImportError as e:\\n        raise ImportError(\\n            \\"gsplat kurulu değil. pip install gsplat\\"\\n        ) from e\\n\\n    device = means.device\\n    if opacities.dim() == 2:\\n        opacities = opacities.squeeze(-1)\\n\\n    viewmat = w2c.unsqueeze(0)                              # (1, 4, 4) — assume already on device\\n    Ks = K.unsqueeze(0)                                     # (1, 3, 3) — assume already on device\\n    bg = torch.tensor(bg_color, device=device).unsqueeze(0)  # (1, 3)\\n\\n    mode = \\"RGB+ED\\" if with_depth else \\"RGB\\"\\n\\n    out, alpha, info = rasterization(\\n        means=means,\\n        quats=quats,\\n        scales=scales,\\n        opacities=opacities,\\n        colors=colors,\\n        viewmats=viewmat,\\n        Ks=Ks,\\n        width=width,\\n        height=height,\\n        sh_degree=sh_degree if colors.dim() == 3 else None,\\n        backgrounds=bg,                # gsplat extends with 0 for depth chan internally\\n        render_mode=mode,\\n        packed=False,\\n        rasterize_mode=\\"antialiased\\",  # gsplat 1.5+ Mip-Splatting anti-alias\\n    )\\n    # out: (1, H, W, 3) for RGB, (1, H, W, 4) for RGB+ED. alpha: (1, H, W, 1)\\n    return out[0], alpha[0], info\\n\\n\\ndef render_image_uint8(*args, **kwargs) -> np.ndarray:\\n    \\"\\"\\"Yardımcı: render_view → (H, W, 3) uint8.\\"\\"\\"\\n    rgb, _, _ = render_view(*args, **kwargs)\\n    # with_depth=True ise RGB only al\\n    if rgb.shape[-1] == 4:\\n        rgb = rgb[..., :3]\\n    return (rgb.clamp(0, 1).detach().cpu().numpy() * 255).astype(np.uint8)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Sağlık testi\\n# ---------------------------------------------------------------------------\\nif __name__ == \\"__main__\\":\\n    if not torch.cuda.is_available():\\n        print(\\"⚠ CUDA yok\\")\\n        raise SystemExit(0)\\n\\n    device = \\"cuda\\"\\n    N = 1000\\n    means = torch.randn(N, 3, device=device) * 0.5\\n    means[:, 2] += 5.0\\n    quats = torch.zeros(N, 4, device=device); quats[:, 0] = 1.0\\n    scales = torch.full((N, 3), 0.05, device=device)\\n    opac = torch.full((N,), 0.8, device=device)\\n    colors = torch.rand(N, 3, device=device)\\n\\n    K = torch.tensor([[400., 0, 320.], [0, 400., 240.], [0, 0, 1.]], device=device)\\n    w2c = torch.eye(4, device=device)\\n\\n    rgb, alpha, info = render_view(means, quats, scales, opac, colors,\\n                                   K, w2c, width=640, height=480, sh_degree=0)\\n    print(f\\"RGB only: {rgb.shape}\\")\\n\\n    rgbd, alpha, info = render_view(means, quats, scales, opac, colors,\\n                                    K, w2c, width=640, height=480, sh_degree=0,\\n                                    with_depth=True)\\n    print(f\\"RGB+D:    {rgbd.shape}  (last chan depth)\\")\\n", "app/backend/model/density_control.py": "\\"\\"\\"Faz 5b — Adaptive Density Control (ADC).\\n\\nOrijinal 3DGS makalesindeki yaklaşıma yakın:\\n  - prune: opasite çok düşük ya da scale çok büyük → sil\\n  - clone:  küçük + yüksek pozisyon gradyanı → tıpkısını ekle\\n  - split:  büyük + yüksek pozisyon gradyanı → ikiye böl, scale\'i küçült\\n\\"\\"\\"\\nfrom __future__ import annotations\\nimport torch\\nimport torch.nn.functional as F\\nfrom .gaussian_model import GaussianModel\\n\\n\\nclass DensityController:\\n    \\"\\"\\"Pozisyon gradyanlarını biriktiren ve clone/split/prune kararı veren yardımcı.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        grad_threshold: float = 2e-4,\\n        scale_split_threshold: float = 0.01,\\n        min_opacity: float = 0.005,\\n        max_scale: float = 0.1,\\n        max_gaussians: int = 0,   # 0 = sınırsız; >0 ise N >= cap iken split kapalı\\n    ):\\n        self.grad_threshold = grad_threshold\\n        self.scale_split_threshold = scale_split_threshold\\n        self.min_opacity = min_opacity\\n        self.max_scale = max_scale\\n        self.max_gaussians = int(max_gaussians)\\n\\n        self._grad_accum: torch.Tensor | None = None\\n        self._n_obs: torch.Tensor | None = None\\n\\n    def reset(self, num_points: int, device: torch.device) -> None:\\n        self._grad_accum = torch.zeros(num_points, device=device)\\n        self._n_obs = torch.zeros(num_points, device=device)\\n\\n    def accumulate(self, gs: GaussianModel) -> None:\\n        \\"\\"\\"Bir backward\'dan sonra çağır — pozisyon gradyanını biriktir.\\n\\n        v5.0 FIX (multi-view bug): n_obs\'u SADECE gradient ALAN gaussian\'lar\\n        icin artir. Eskiden her iter +1 ekliyordu, ama multi-view\'da her iter\\n        sadece 1 kamera goruluyor → ~%95 gaussian o iter gradient ALMIYOR.\\n        Bu yuzden avg_grad = sum_grad / total_iter → 20x kucuk → clone hic\\n        tetiklenmiyor → N surekli azaliyor. Single-view\'da grad_norm her\\n        gaussian icin > 0 (hepsi her iter goruluyor) → davranis ayni kalir.\\n        \\"\\"\\"\\n        if gs.means.grad is None:\\n            return\\n        if self._grad_accum is None or len(self._grad_accum) != gs.num_points:\\n            self.reset(gs.num_points, gs.means.device)\\n\\n        grad_norm = gs.means.grad.norm(dim=-1)\\n        self._grad_accum += grad_norm\\n        # Visibility-aware obs count: gradient gelmediyse n_obs artmaz.\\n        self._n_obs += (grad_norm > 1e-12).float()\\n\\n    @torch.no_grad()\\n    def step(\\n        self,\\n        gs: GaussianModel,\\n        optimizer=None,\\n        dynamic_densify_scale: float = 1.0,\\n    ) -> dict:\\n        \\"\\"\\"Clone + split + prune. İstatistikleri döner.\\n\\n        Args:\\n          gs: GaussianModel\\n          optimizer: torch.optim.Adam (or compatible). T4 perf fix — optimizer\\n            geçirilirse Adam exp_avg / exp_avg_sq state\'i parametre mutation\'ı\\n            ile lockstep gunceleneceği için her density step\'te momentum\\n            kaybolmaz. Geriye dönük uyum: None geçilirse legacy code path\\n            (her gauss param\'i fresh nn.Parameter, optimizer state baska yerde\\n            yeniden kurulmali — eski trainer._build_optimizer() davranisi).\\n          dynamic_densify_scale: 4D Quality v6.1 Madde 7 — dynamic flag\'i set\\n            olan gauss\'lar icin grad_threshold * scale (genelde 0.5 = 2x hassas).\\n            1.0 = no-op. gs.is_dynamic buffer\'i Phase 2.1 ile set edilir.\\n        \\"\\"\\"\\n        stats = {\\"cloned\\": 0, \\"split\\": 0, \\"pruned\\": 0, \\"before\\": gs.num_points}\\n\\n        if self._grad_accum is not None and self._n_obs is not None:\\n            obs = self._n_obs.clamp(min=1)\\n            avg_grad = self._grad_accum / obs\\n\\n            # v5.0: Multi-view\'da bir gaussian 1-2 kez gorunup buyuk gradient\\n            # alabilir → spurious clone\'u onlemek icin min_obs gate.\\n            min_obs = 3.0\\n            seen_enough = self._n_obs >= min_obs\\n\\n            # 4D Quality v6.1 — Madde 7: Adaptive densify dynamic regions\\n            # Dynamic gauss\'lar icin grad_threshold * dynamic_densify_scale.\\n            # 0.5 = %50 hassas (2x daha kolay densify) → motion-rich bolgelere\\n            # daha cok detail. Static gauss\'lar default threshold.\\n            grad_thr_per_gauss = torch.full_like(avg_grad, self.grad_threshold)\\n            if (dynamic_densify_scale != 1.0\\n                    and hasattr(gs, \\"is_dynamic\\")\\n                    and gs.is_dynamic is not None\\n                    and gs.is_dynamic.numel() == avg_grad.numel()):\\n                dyn_mask = gs.is_dynamic.to(avg_grad.device)\\n                grad_thr_per_gauss = torch.where(\\n                    dyn_mask,\\n                    torch.full_like(grad_thr_per_gauss,\\n                                    self.grad_threshold * dynamic_densify_scale),\\n                    grad_thr_per_gauss,\\n                )\\n\\n            high_grad = (avg_grad > grad_thr_per_gauss) & seen_enough\\n            scale_max = gs.get_scales.max(dim=-1).values\\n            small = scale_max <= self.scale_split_threshold\\n            large = scale_max > self.scale_split_threshold\\n\\n            clone_mask = high_grad & small\\n            split_mask = high_grad & large\\n\\n            # v3.7.2: Hard cap on N — eğer cap\'i aştıysak split/clone kapa,\\n            # sadece prune yapsın. Banana ultra\'da N=164k oldu, render çöktü.\\n            if self.max_gaussians > 0 and gs.num_points >= self.max_gaussians:\\n                clone_mask = torch.zeros_like(clone_mask)\\n                split_mask = torch.zeros_like(split_mask)\\n\\n            # Clone: aynı parametrelerle kopyala\\n            n_cloned = int(clone_mask.sum().item())\\n            if n_cloned > 0:\\n                clone_kwargs = dict(\\n                    new_means     = gs.means[clone_mask].clone(),\\n                    new_scales    = gs.scales[clone_mask].clone(),\\n                    new_quats     = gs.quats[clone_mask].clone(),\\n                    new_opacities = gs.opacities[clone_mask].clone(),\\n                    new_sh_dc     = gs.sh_dc[clone_mask].clone(),\\n                    new_sh_rest   = gs.sh_rest[clone_mask].clone(),\\n                )\\n                if optimizer is not None:\\n                    gs._append_keep_optimizer(optimizer, **clone_kwargs)\\n                else:\\n                    gs.append_gaussians(**clone_kwargs)\\n                stats[\\"cloned\\"] = n_cloned\\n\\n            # Split: aynı bölgede biraz kaymış 2 yeni Gaussian, scale küçült\\n            n_split = int(split_mask.sum().item())\\n            if n_split > 0:\\n                idx = split_mask.nonzero(as_tuple=False).squeeze(-1)\\n                # Her Gaussian\'dan 2 tane üret\\n                noise = torch.randn(n_split * 2, 3, device=gs.means.device)\\n                base_scale = gs.get_scales[idx].repeat(2, 1)\\n                offset = noise * base_scale * 0.5\\n\\n                new_means     = gs.means[idx].repeat(2, 1) + offset\\n                new_scales    = (gs.scales[idx].repeat(2, 1) - torch.log(torch.tensor(1.6, device=gs.means.device)))\\n                new_quats     = gs.quats[idx].repeat(2, 1)\\n                new_opacities = gs.opacities[idx].repeat(2, 1)\\n                new_sh_dc     = gs.sh_dc[idx].repeat(2, 1, 1)\\n                new_sh_rest   = gs.sh_rest[idx].repeat(2, 1, 1)\\n\\n                # Orijinalleri sil, sonra yeni iki tanesini ekle.\\n                # ÖNEMLİ: clone aşamasında tensor büyümüş olabilir (1529 → 1532),\\n                # bu yüzden keep mask\'ini mevcut tensor boyutunda oluşturuyoruz.\\n                # split_mask\'ten gelen idx değerleri orijinal prefix\'te kaldığı için\\n                # büyüyen tensor\'da da geçerli (clone\'lar sona eklenir, prefix bozulmaz).\\n                current_n = gs.num_points\\n                keep = torch.ones(current_n, dtype=torch.bool, device=gs.means.device)\\n                keep[idx] = False\\n                if optimizer is not None:\\n                    gs._apply_mask_keep_optimizer(keep, optimizer)\\n                    gs._append_keep_optimizer(\\n                        optimizer,\\n                        new_means, new_scales, new_quats,\\n                        new_opacities, new_sh_dc, new_sh_rest,\\n                    )\\n                else:\\n                    gs._apply_mask(keep)\\n                    gs.append_gaussians(\\n                        new_means, new_scales, new_quats,\\n                        new_opacities, new_sh_dc, new_sh_rest,\\n                    )\\n                stats[\\"split\\"] = n_split\\n\\n        # Prune\\n        if optimizer is not None:\\n            n_pruned = gs.prune_gaussians_keep_optimizer(\\n                optimizer, self.min_opacity, self.max_scale,\\n            )\\n        else:\\n            n_pruned = gs.prune_gaussians(self.min_opacity, self.max_scale)\\n        stats[\\"pruned\\"] = n_pruned\\n        stats[\\"after\\"] = gs.num_points\\n\\n        # Buffer\'ları sıfırla\\n        self.reset(gs.num_points, gs.means.device)\\n        return stats\\n", "app/backend/model/trainer.py": "\\"\\"\\"Faz 5 — 4DGS training loop.\\n\\nv3 — Stage 2 motion supervision tam:\\n  - L_recon: L1 + (1-SSIM), opsiyonel mask-weighted\\n  - L_depth: Metric3D/MiDaS scale-invariant L1\\n  - L_track: CoTracker 3D-anchored projection L1 (sparse motion supervision)\\n  - L_deform_reg / L_smoothness / L_rigidity (Stage 1 regularizers)\\n  - Regularizer\'lar için linear warmup (0 → full over first warmup_iters iter)\\n  - Diagnostics: her log_interval\'de Δpos mean/max, her loss bileşeni ayrı\\n\\"\\"\\"\\nfrom __future__ import annotations\\nimport math\\nimport time\\nimport concurrent.futures\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nimport numpy as np\\nfrom pathlib import Path\\nfrom typing import Any, Callable, Sequence\\n\\nfrom .gaussian_model import GaussianModel\\nfrom .deformation import DeformationField, decode_fourier_trajectory\\nfrom .renderer import render_view\\nfrom .density_control import DensityController\\n\\n\\nTrainProgressCallback = Callable[[int, int, float, float, int], None]\\n\\n\\n# ---------------------------------------------------------------------------\\n# Loss primitives\\n# ---------------------------------------------------------------------------\\ndef compute_recon_loss(\\n    rendered: torch.Tensor,\\n    gt: torch.Tensor,\\n    lambda_ssim: float = 0.2,\\n    pixel_weight: torch.Tensor | None = None,\\n) -> torch.Tensor:\\n    \\"\\"\\"L1 + (1-SSIM), opsiyonel pixel weighting.\\"\\"\\"\\n    diff = (rendered - gt).abs()\\n    if pixel_weight is not None:\\n        l1 = (diff * pixel_weight.unsqueeze(-1)).mean()\\n    else:\\n        l1 = diff.mean()\\n    try:\\n        from pytorch_msssim import ssim\\n        ssim_val = ssim(\\n            rendered.permute(2, 0, 1).unsqueeze(0),\\n            gt.permute(2, 0, 1).unsqueeze(0),\\n            data_range=1.0, size_average=True,\\n        )\\n        return (1.0 - lambda_ssim) * l1 + lambda_ssim * (1.0 - ssim_val)\\n    except ImportError:\\n        return l1\\n\\n\\ndef psnr(rendered: torch.Tensor, gt: torch.Tensor) -> float:\\n    mse = ((rendered - gt) ** 2).mean().item()\\n    return float(\\"inf\\") if mse < 1e-12 else -10.0 * math.log10(mse)\\n\\n\\n# ---------------------------------------------------------------------------\\n# 4D Quality v6.1 — Madde 11: Adaptive SH degree schedule\\n# ---------------------------------------------------------------------------\\ndef progressive_sh_degree(\\n    iter_idx: int, n_iters: int, max_degree: int = 3\\n) -> int:\\n    \\"\\"\\"Iter\'a göre SH degree dön: %25/%50/%75/%100 bandlarinda 0/1/2/3.\\n\\n    Erken iter\'lerde DC + 1st order yeterken, late iter\'lerde 3rd order detail\\n    kazandırır. Bu schedule ~0.3-0.5 dB PSNR artışı + daha temiz convergence.\\n    Standart 3DGS practice (INRIA reference impl).\\n    \\"\\"\\"\\n    if max_degree <= 0:\\n        return 0\\n    frac = iter_idx / max(n_iters, 1)\\n    if frac < 0.25:\\n        return 0\\n    elif frac < 0.50:\\n        return min(1, max_degree)\\n    elif frac < 0.75:\\n        return min(2, max_degree)\\n    return max_degree\\n\\n\\n# ---------------------------------------------------------------------------\\n# 4D Quality v6.1 — Madde 2-B: Mip-Splatting Python-side scale floor\\n# ---------------------------------------------------------------------------\\ndef apply_mip_scale_floor(\\n    scales: torch.Tensor, means: torch.Tensor, cam_centers: torch.Tensor,\\n    floor_frac: float = 0.001,\\n    chunk_size: int = 8192,\\n) -> torch.Tensor:\\n    \\"\\"\\"3D scale floor — her gauss\'a min scale = floor_frac × distance_to_nearest_cam.\\n\\n    Mip-Splatting paper\'inin 3D filter approximation\'i. Ekran-uzayinda\\n    Gaussian\'a min pixel-size garantisi vermez ama yakin gauss\'larin\\n    \\"noktalasmasini\\" engelliyor (ufak scale → tek piksel artifact\'i).\\n\\n    Memory-safe chunked impl: (N, M, 3) materialize etmek yerine N\'i\\n    chunk_size\'a bol → her chunk (chunk, M, 3) tensor (~ chunk×M×12 bytes).\\n    Default chunk=8192 + M=32 → ~12 MB peak, hizli + safe.\\n\\n    Args:\\n      scales: (N, 3) gauss scale (post-activation, units of world)\\n      means: (N, 3) gauss world position\\n      cam_centers: (M, 3) sahnedeki cam centers (genelde 32\'ye subsample edilmis)\\n      floor_frac: scale floor = floor_frac × min_dist (0.001-0.005 onerilen)\\n      chunk_size: gauss chunk boyutu (memory tradeoff)\\n    \\"\\"\\"\\n    if floor_frac <= 0 or cam_centers.numel() == 0:\\n        return scales\\n    N = means.shape[0]\\n    # Output buffer\\n    min_dist = torch.empty(N, device=means.device, dtype=means.dtype)\\n    for i in range(0, N, chunk_size):\\n        j = min(i + chunk_size, N)\\n        # (chunk, 1, 3) - (1, M, 3) -> (chunk, M, 3) -> (chunk, M) -> (chunk,)\\n        diff = means[i:j].unsqueeze(1) - cam_centers.unsqueeze(0)\\n        d = (diff * diff).sum(-1).sqrt()\\n        min_dist[i:j] = d.min(dim=1).values\\n    floor = (floor_frac * min_dist).unsqueeze(1).expand_as(scales)\\n    return torch.maximum(scales, floor)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Frame / depth / mask / tracks yardımcıları\\n# ---------------------------------------------------------------------------\\ndef load_frame_tensor(\\n    frame_path: Path,\\n    target_size: tuple[int, int] | None = None,\\n    dtype: str = \\"float32\\",\\n) -> torch.Tensor:\\n    \\"\\"\\"PNG → (H, W, 3) tensor.\\n\\n    dtype:\\n      - \\"float32\\" (default, legacy): returns float32 tensor in [0, 1] (CPU).\\n      - \\"uint8\\": returns uint8 tensor (CPU). Caller is responsible for\\n        converting to float on GPU via `.float() / 255.0` after `.to(device)`.\\n        ~3.5x memory savings vs float32 — used by preload_to_ram cache.\\n\\n    Corrupt-cache resilience: cv2.imread returns None for truncated/corrupt\\n    PNGs (from prior pipeline crashes). We raise FileNotFoundError so the\\n    upstream cache marker mismatch surfaces clearly. If you\'d rather skip\\n    silently, swap the raise for `return None` + adjust the type hint.\\n    \\"\\"\\"\\n    import cv2\\n    img = cv2.imread(str(frame_path))\\n    if img is None:\\n        # Could be missing OR corrupt. Delete if exists so next run regenerates.\\n        if Path(frame_path).exists():\\n            print(f\\"[frame] WARN: corrupt {Path(frame_path).name}; deleting\\")\\n            try:\\n                Path(frame_path).unlink()\\n            except Exception:\\n                pass\\n        raise FileNotFoundError(frame_path)\\n    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\\n    if target_size is not None:\\n        img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)\\n    if dtype == \\"uint8\\":\\n        # cv2 returns uint8 by default; keep as-is for memory-efficient cache.\\n        return torch.from_numpy(np.ascontiguousarray(img))  # uint8 (H, W, 3)\\n    # Legacy default: float32 [0, 1]\\n    return torch.from_numpy(img).float() / 255.0\\n\\n\\ndef _load_depth_for_frame(\\n    depth_dir: Path,\\n    frame_path: Path,\\n    target_size: tuple[int, int] | None = None,\\n    dtype: str = \\"float32\\",\\n) -> torch.Tensor | None:\\n    \\"\\"\\"Metric3D/MiDaS depth .npy → (H, W) tensor.\\n\\n    dtype:\\n      - \\"float32\\" (default, legacy): float32 CPU tensor.\\n      - \\"float16\\": float16 CPU tensor (~2x memory savings). Caller should\\n        cast to float32 on GPU via `.to(device).float()`.\\n\\n    Corrupt-cache resilience: if the .npy is truncated / unreadable\\n    (e.g., because a prior pipeline crash left a partial write),\\n    delete it + return None instead of crashing the run. Pipeline\\n    will see no depth for this frame and either regenerate it on\\n    the next foundation run, or skip depth supervision for it.\\n    \\"\\"\\"\\n    import cv2\\n    candidate = Path(depth_dir) / f\\"{Path(frame_path).stem}_depth.npy\\"\\n    if not candidate.exists():\\n        return None\\n    try:\\n        arr = np.load(candidate).astype(np.float32)\\n    except (EOFError, ValueError, OSError, Exception) as e:\\n        # Truncated / corrupt .npy from a prior crash — delete + skip.\\n        print(f\\"[depth] WARN: corrupt {candidate.name} ({type(e).__name__}: {e}); \\"\\n              f\\"deleting + skipping frame\\")\\n        try:\\n            candidate.unlink()\\n        except Exception:\\n            pass\\n        return None\\n    if target_size is not None:\\n        arr = cv2.resize(arr, target_size, interpolation=cv2.INTER_LINEAR)\\n    t = torch.from_numpy(arr)\\n    if dtype == \\"float16\\":\\n        t = t.to(torch.float16)\\n    return t\\n\\n\\ndef _load_mask_for_frame(\\n    mask_dir: Path,\\n    frame_path: Path,\\n    target_size: tuple[int, int] | None = None,\\n    dtype: str = \\"float32\\",\\n) -> torch.Tensor | None:\\n    \\"\\"\\"Dynamic mask .png → (H, W) tensor.\\n\\n    dtype:\\n      - \\"float32\\" (default, legacy): float32 in [0, 1].\\n      - \\"uint8\\": raw uint8 [0, 255]. Caller converts via `.float() / 255.0`\\n        on GPU. Used for the preload_to_ram cache (~4x memory savings).\\n    \\"\\"\\"\\n    import cv2\\n    import re\\n    stem = Path(frame_path).stem\\n    m = re.search(r\\"(\\\\d+)$\\", stem)\\n    if not m:\\n        return None\\n    idx0 = int(m.group(1))\\n    candidate = Path(mask_dir) / f\\"mask_{idx0 + 1:04d}.png\\"\\n    if not candidate.exists():\\n        return None\\n    img = cv2.imread(str(candidate), cv2.IMREAD_GRAYSCALE)\\n    if img is None:\\n        return None\\n    if target_size is not None:\\n        img = cv2.resize(img, target_size, interpolation=cv2.INTER_NEAREST)\\n    if dtype == \\"uint8\\":\\n        return torch.from_numpy(np.ascontiguousarray(img))  # uint8 (H, W)\\n    return torch.from_numpy(img.astype(\\"float32\\") / 255.0)\\n\\n\\ndef _unproject_pixel_to_world(\\n    uv: torch.Tensor,         # (N, 2) pixel coords\\n    depth_at_uv: torch.Tensor,# (N,)   depth values\\n    K: torch.Tensor,          # (3, 3) intrinsics at UV resolution\\n    w2c: torch.Tensor,        # (4, 4) world-to-camera\\n) -> torch.Tensor:\\n    \\"\\"\\"Pixel + depth → world 3D. Returns (N, 3).\\"\\"\\"\\n    # camera space: (K^-1 · [u, v, 1]) * depth\\n    K_inv = torch.linalg.inv(K)\\n    u, v = uv[:, 0], uv[:, 1]\\n    ones = torch.ones_like(u)\\n    pixels_h = torch.stack([u, v, ones], dim=0)  # (3, N)\\n    rays = K_inv @ pixels_h                       # (3, N), z=1 plane\\n    cam_pts = rays * depth_at_uv.unsqueeze(0)     # (3, N)\\n    # cam → world: (w2c)^-1 @ [X, Y, Z, 1]\\n    c2w = torch.linalg.inv(w2c)\\n    cam_pts_h = torch.cat([cam_pts, ones.unsqueeze(0)], dim=0)  # (4, N)\\n    world_h = c2w @ cam_pts_h                                     # (4, N)\\n    return world_h[:3].T                                          # (N, 3)\\n\\n\\ndef _project_world_to_pixel(\\n    world_pts: torch.Tensor,  # (N, 3)\\n    K: torch.Tensor,          # (3, 3)\\n    w2c: torch.Tensor,        # (4, 4)\\n) -> tuple[torch.Tensor, torch.Tensor]:\\n    \\"\\"\\"World → pixel. Returns (uv (N, 2), valid_mask (N,) — behind-cam filter).\\"\\"\\"\\n    ones = torch.ones(world_pts.shape[0], 1, device=world_pts.device)\\n    world_h = torch.cat([world_pts, ones], dim=-1)         # (N, 4)\\n    cam_pts = (w2c @ world_h.T).T                          # (N, 4)\\n    z = cam_pts[:, 2]\\n    valid = z > 1e-3\\n    cam_pts_safe = cam_pts[:, :3] / z.clamp(min=1e-3).unsqueeze(-1)  # (N, 3)\\n    img_h = (K @ cam_pts_safe.T).T                         # (N, 3)\\n    uv = img_h[:, :2]                                      # (N, 2)\\n    return uv, valid\\n\\n\\n# ---------------------------------------------------------------------------\\n# Trainer\\n# ---------------------------------------------------------------------------\\nclass Trainer4DGS:\\n    def __init__(\\n        self,\\n        gs: GaussianModel,\\n        deform: DeformationField,\\n        device: str = \\"cuda\\",\\n        scene_extent: float = 1.0,\\n        # LR\\n        lr_means: float = 1.6e-4,\\n        lr_scales: float = 5e-3,\\n        lr_quats: float = 1e-3,\\n        lr_opacities: float = 5e-2,\\n        lr_sh_dc: float = 2.5e-3,\\n        lr_sh_rest: float = 2.5e-3 / 20,\\n        lr_deform: float = 1e-3,\\n        # Density\\n        density_start_iter: int = 500,\\n        density_end_iter: int = 15_000,\\n        density_interval: int = 100,\\n        densify_grad_threshold: float = 2e-4,\\n        prune_min_opacity: float = 0.005,\\n        prune_max_scale: float = 0.1,\\n        # Loss\\n        lambda_ssim: float = 0.2,\\n        # Motion regularizers (Stage 1)\\n        lambda_deform_reg: float = 1e-3,\\n        lambda_smoothness: float = 1e-2,\\n        lambda_rigidity: float = 1e-2,\\n        rigidity_sample_k: int = 512,\\n        smoothness_dt: float = 0.02,\\n        # Foundation losses (Stage 2)\\n        lambda_depth: float = 0.0,\\n        lambda_mask_motion: float = 1.0,\\n        lambda_track: float = 0.0,\\n        track_sample_k: int = 256,\\n        # Scale regularizer (v3 — outlier blow-up önleme)\\n        lambda_scale: float = 1e-3,\\n        # v3.8: Anisotropy regularizer — streak/needle gaussian fix.\\n        # max_scale/min_scale ratio threshold üstünde quadratic ceza.\\n        # banana_demo\'da scene\'in etrafında uzun parlak çizgiler oluşturdu.\\n        lambda_aniso: float = 0.0,           # 0 = kapalı, 0.01-0.05 önerilen\\n        aniso_threshold: float = 5.0,        # ratio max/min < threshold serbest\\n        # v3.8: Total dpos clamp tightening (per-iter motion cap, fraction of scene)\\n        dpos_total_cap_frac: float = 0.2,    # v3.6.2 default, 0.05 daha sıkı\\n        # Opacity reset — INRIA 3DGS trick (her N iter low-opacity\'yi topluca reset)\\n        opacity_reset_interval: int = 3000,\\n        # v3.7.2: Hard cap on N (0 = sınırsız). Banana ultra\'da N=164k oldu, çöktü.\\n        max_gaussians: int = 0,\\n        # Warmup\\n        warmup_iters: int = 2000,\\n        # v3.6 / Yol C — Per-gaussian Fourier trajectory\\n        deform_pos_mode: str = \\"hybrid\\",     # \\"mlp\\" | \\"fourier\\" | \\"hybrid\\"\\n        lr_fourier: float = 5e-3,\\n        lambda_fourier_reg: float = 1e-4,\\n        # Phase 1.6 — LPIPS perceptual loss\\n        lambda_lpips: float = 0.0,\\n        lpips_net: str = \\"alex\\",\\n        lpips_warmup_iters: int = 1000,\\n        # Phase 1.7 — Multi-view cross-view consistency\\n        lambda_multiview_consistency: float = 0.0,\\n        # Phase 1.8 — RAFT optical flow loss\\n        lambda_flow: float = 0.0,\\n        flow_warmup_iters: int = 1000,\\n        # Phase 1.9 — Densify dynamics MV scale\\n        densify_mv_threshold_scale: float = 0.7,\\n        # Phase 2.2 — Multi-resolution training schedule [(iter, long_edge), ...]\\n        multires_schedule=None,\\n        # Phase 2.3 — Camera pose refinement\\n        lr_cam_K: float = 0.0,\\n        lr_cam_w2c: float = 0.0,\\n        cam_refine_start_iter: int = 5000,\\n    ):\\n        self.gs = gs.to(device)\\n        self.deform = deform.to(device)\\n        self.device = device\\n        self.scene_extent = scene_extent\\n        self.lambda_ssim = lambda_ssim\\n        self.lambda_deform_reg = lambda_deform_reg\\n        self.lambda_smoothness = lambda_smoothness\\n        self.lambda_rigidity = lambda_rigidity\\n        self.rigidity_sample_k = rigidity_sample_k\\n        self.smoothness_dt = smoothness_dt\\n        self.lambda_depth = lambda_depth\\n        self.lambda_mask_motion = lambda_mask_motion\\n        self.lambda_track = lambda_track\\n        self.track_sample_k = track_sample_k\\n        self.lambda_scale = lambda_scale\\n        # v3.8: Anisotropy + tightened clamp\\n        self.lambda_aniso = float(lambda_aniso)\\n        self.aniso_threshold = float(aniso_threshold)\\n        self.dpos_total_cap_frac = float(dpos_total_cap_frac)\\n        self.opacity_reset_interval = int(opacity_reset_interval)\\n        self.warmup_iters = max(1, warmup_iters)\\n        # v3.6 / Yol C\\n        self.deform_pos_mode = deform_pos_mode\\n        self.lr_fourier = lr_fourier\\n        self.lambda_fourier_reg = lambda_fourier_reg\\n        # Phase 1.6/1.7/1.8 — perceptual + cross-view + flow\\n        self.lambda_lpips = float(lambda_lpips)\\n        self.lpips_net = str(lpips_net)\\n        self.lpips_warmup_iters = int(lpips_warmup_iters)\\n        self.lambda_multiview_consistency = float(lambda_multiview_consistency)\\n        self.lambda_flow = float(lambda_flow)\\n        self.flow_warmup_iters = int(flow_warmup_iters)\\n        self._lpips_module = None  # lazy init in train()\\n        # Phase 1.9\\n        self.densify_mv_threshold_scale = float(densify_mv_threshold_scale)\\n        # Phase 2.2\\n        self.multires_schedule = list(multires_schedule) if multires_schedule else []\\n        # Phase 2.3\\n        self.lr_cam_K = float(lr_cam_K)\\n        self.lr_cam_w2c = float(lr_cam_w2c)\\n        self.cam_refine_start_iter = int(cam_refine_start_iter)\\n        # Phase 2.1 — Static/Dynamic split (default off; promote_dynamic ile aktiflesir)\\n        self.use_static_dynamic_split = False\\n        # Static 3DGS Faz 1 — static mode flag (4D features bypass)\\n        self.static_mode = False\\n        # 4D Quality v6.1 — runtime knobs (default off; train() siginatursinde override)\\n        self.sh_progressive_schedule = False\\n        self.lambda_accel = 0.0\\n        self.cam_grad_clip_norm = 0.0\\n        self.mip_scale_floor_frac = 0.0\\n        self.dynamic_densify_scale = 1.0\\n        # Phase 2.3 — Camera pose refine optimizer placeholder (None = inactive)\\n        self._cam_refine_optimizer = None\\n        if self.lr_cam_K > 0 or self.lr_cam_w2c > 0:\\n            print(f\\"[trainer] Camera pose refinement requested \\"\\n                  f\\"(lr_K={self.lr_cam_K}, lr_w2c={self.lr_cam_w2c}). \\"\\n                  f\\"Aktivasyon train()\'de scene cam\'larina baglanir.\\")\\n        if deform_pos_mode not in (\\"mlp\\", \\"fourier\\", \\"hybrid\\"):\\n            raise ValueError(f\\"deform_pos_mode geçersiz: {deform_pos_mode}\\")\\n        if deform_pos_mode != \\"mlp\\" and (\\n            self.gs.fourier_pos_coeffs is None or self.gs.fourier_K == 0\\n        ):\\n            raise ValueError(\\n                f\\"deform_pos_mode=\'{deform_pos_mode}\' ama GaussianModel fourier_K=0. \\"\\n                f\\"GaussianModel\'i fourier_K>0 ile başlat.\\"\\n            )\\n        print(f\\"[trainer] deform_pos_mode={deform_pos_mode} fourier_K={self.gs.fourier_K}\\")\\n\\n        self.density_start_iter = density_start_iter\\n        self.density_end_iter = density_end_iter\\n        self.density_interval = density_interval\\n\\n        # v3.2 FIX: prune_max_scale is now interpreted as FRACTION of scene_extent\\n        # (INRIA 3DGS original intent). Previous absolute-unit interpretation caused\\n        # mass prune on scenes with extent > 1 (cutlemon had extent=70, prune_max=0.1\\n        # → pruned %35 of initial points at first density step).\\n        effective_max_scale = prune_max_scale * max(scene_extent, 1e-6)\\n        print(f\\"[trainer] prune_max_scale effective: {prune_max_scale} × scene_extent({scene_extent:.2f}) = {effective_max_scale:.3f} units\\")\\n        self.density = DensityController(\\n            grad_threshold=densify_grad_threshold,\\n            min_opacity=prune_min_opacity,\\n            max_scale=effective_max_scale,\\n            max_gaussians=max_gaussians,\\n        )\\n        if max_gaussians > 0:\\n            print(f\\"[trainer] N hard cap: {max_gaussians:,} (cap dolunca split kapalı, prune devam)\\")\\n\\n        self.lr_specs = {\\n            \\"means\\":     lr_means,\\n            \\"scales\\":    lr_scales,\\n            \\"quats\\":     lr_quats,\\n            \\"opacities\\": lr_opacities,\\n            \\"sh_dc\\":     lr_sh_dc,\\n            \\"sh_rest\\":   lr_sh_rest,\\n        }\\n        self.lr_deform = lr_deform\\n        self._build_optimizer()\\n\\n    def _build_optimizer(self) -> None:\\n        groups = [\\n            {\\"params\\": [self.gs.means],     \\"lr\\": self.lr_specs[\\"means\\"]},\\n            {\\"params\\": [self.gs.scales],    \\"lr\\": self.lr_specs[\\"scales\\"]},\\n            {\\"params\\": [self.gs.quats],     \\"lr\\": self.lr_specs[\\"quats\\"]},\\n            {\\"params\\": [self.gs.opacities], \\"lr\\": self.lr_specs[\\"opacities\\"]},\\n            {\\"params\\": [self.gs.sh_dc],     \\"lr\\": self.lr_specs[\\"sh_dc\\"]},\\n            {\\"params\\": [self.gs.sh_rest],   \\"lr\\": self.lr_specs[\\"sh_rest\\"]},\\n            {\\"params\\": list(self.deform.parameters()), \\"lr\\": self.lr_deform},\\n        ]\\n        # v3.6: Fourier trajectory param group (eğer açıksa)\\n        if self.gs.fourier_pos_coeffs is not None:\\n            groups.append(\\n                {\\"params\\": [self.gs.fourier_pos_coeffs], \\"lr\\": self.lr_fourier}\\n            )\\n        self.optimizer = torch.optim.Adam(groups, eps=1e-15)\\n\\n    def _apply_deformation(self, t: float, return_raw: bool = False):\\n        # Static 3DGS Faz 1 — static_mode\'da hiç deformation yok, identity döner\\n        if getattr(self, \\"static_mode\\", False) or self.deform is None:\\n            id_out = (\\n                self.gs.means,\\n                F.normalize(self.gs.quats, dim=-1),\\n                self.gs.get_scales,\\n            )\\n            if return_raw:\\n                return id_out, (None, None, None)\\n            return id_out\\n        # MLP her zaman çağrılır (dquat, dscale için gerekli, dpos opsiyonel)\\n        dpos_mlp, dquat, dscale = self.deform(self.gs.means, t, self.scene_extent)\\n        # v3.5: dscale clamp — MLP\'nin scale delta\'sı ±2 ile sınırlı.\\n        # T6 perf fix: clamp\'i yeni isimde tut, RAW dscale\'i regularizer icin koru.\\n        dscale_clamped = dscale.clamp(min=-2.0, max=2.0)\\n\\n        # v3.6.2: dpos_mlp clamp — MLP serbest yazabildiği için 500+ unit\\n        # jump\'lara yol açıyordu (chickchicken_fourier_v2\'de Δpos max=513 görüldü).\\n        # Max motion per iter = scene_extent × 0.1 (yumuşak caydırıcı).\\n        # T6 perf fix: clamp\'i yeni isimde tut, RAW dpos_mlp\'i regularizer icin koru.\\n        mlp_cap = self.scene_extent * 0.1\\n        dpos_mlp_clamped = dpos_mlp.clamp(min=-mlp_cap, max=mlp_cap)\\n\\n        # v3.6 / Yol C: Pozisyon deformasyonunu pos_mode\'a göre seç\\n        if self.deform_pos_mode == \\"mlp\\":\\n            dpos = dpos_mlp_clamped\\n        elif self.deform_pos_mode == \\"fourier\\":\\n            dpos = decode_fourier_trajectory(self.gs.fourier_pos_coeffs, t)\\n        else:  # \\"hybrid\\"\\n            dpos_fourier = decode_fourier_trajectory(self.gs.fourier_pos_coeffs, t)\\n            dpos = dpos_mlp_clamped + dpos_fourier\\n\\n        # v3.6.2 → v3.8: FINAL dpos clamp — overall motion cap\\n        # Default scene_extent × 0.2; Ultra Clean preset 0.05 (4× daha sıkı).\\n        # banana_demo Ultra\'da Δpos max=13 (cap=21\'de, ama görsel streak\'leri için sıkıştırma faydalı).\\n        total_cap = self.scene_extent * self.dpos_total_cap_frac\\n        dpos = dpos.clamp(min=-total_cap, max=total_cap)\\n\\n        deformed_means  = self.gs.means + dpos\\n        deformed_quats  = F.normalize(self.gs.quats + dquat, dim=-1)\\n        deformed_scales = self.gs.get_scales * torch.exp(dscale_clamped)\\n        if return_raw:\\n            # T6: RAW MLP output (pre-clamp) — regularizer block reg=dpos.pow(2)\\n            # tipi loss\'larda kullanir; iki kez forward pass\'i onlemek icin paylasiyoruz.\\n            return (deformed_means, deformed_quats, deformed_scales), (dpos_mlp, dquat, dscale)\\n        return deformed_means, deformed_quats, deformed_scales\\n\\n    def _warmup_factor(self, it: int) -> float:\\n        \\"\\"\\"\\n        Linear warmup 0→1 over ilk warmup_iters iter.\\n        v3.2 fix: hardcoded 100 iter delay kaldırıldı — reg\'ler iter 1\'den\\n        itibaren yumuşak devreye giriyor. Önce 0-100 arası unregulated\\n        training yüzünden deformasyon MLP patlıyordu (Δpos 41, track 196).\\n\\n        v5.0 multi-view fix: static warmup phase sirasinda warmup=0 (deformation\\n        donuk). Static phase bittikten sonra warmup_iters icinde 0->1.\\n        \\"\\"\\"\\n        static_phase = getattr(self, \\"_static_phase_iters\\", 0)\\n        if it <= static_phase:\\n            return 0.0\\n        return min(1.0, max(0.0, (it - static_phase) / max(1, self.warmup_iters)))\\n\\n    @torch.no_grad()\\n    def _setup_static_dynamic_from_masks(\\n        self,\\n        masks_mv_dir: Path,\\n        train_cams: list,\\n        mv_cam_K: dict,\\n        mv_w2c: dict,\\n        threshold_frac: float = 0.10,\\n    ) -> None:\\n        \\"\\"\\"Phase 2.1 — Per-cam motion mask voting ile dynamic gauss seçimi.\\n\\n        Algoritma:\\n          - Her cam icin masks_multiview/cam{N}/mask_*.png yukle (binary motion)\\n          - Her gauss center\'i o cam\'a project (pinhole)\\n          - In-frame ise: pixel mask\'te ise +1 vote\\n          - Toplam vote / toplam view > threshold_frac -> DYNAMIC\\n\\n        Args:\\n            masks_mv_dir: paths[\'masks_mv\'] (cam{N}/mask_*.png yapisi)\\n            train_cams: train cam id list [\'cam01\', ..., \'cam20\']\\n            mv_cam_K, mv_w2c: per-cam intrinsic + extrinsic dict (raw resolution)\\n            threshold_frac: gauss en az %X view\'da motion -> dynamic\\n        \\"\\"\\"\\n        import cv2\\n        from pathlib import Path\\n\\n        masks_mv_dir = Path(masks_mv_dir)\\n        if not masks_mv_dir.exists():\\n            print(f\\"[trainer.static_dyn] masks_mv yok: {masks_mv_dir}, skip\\")\\n            return\\n        if not hasattr(self.gs, \\"promote_dynamic\\"):\\n            print(f\\"[trainer.static_dyn] gs.promote_dynamic API yok, skip\\")\\n            return\\n\\n        n_pts = self.gs.num_points\\n        device = self.gs.means.device\\n        dynamic_votes = torch.zeros(n_pts, dtype=torch.int32, device=device)\\n        n_total_views = 0\\n\\n        means = self.gs.means.detach()  # [N, 3]\\n        for cam_id in train_cams:\\n            cam_mask_dir = masks_mv_dir / cam_id\\n            if not cam_mask_dir.exists():\\n                continue\\n            mask_files = sorted(cam_mask_dir.glob(\\"mask_*.png\\"))\\n            if not mask_files:\\n                continue\\n            K = mv_cam_K[cam_id].to(device)\\n            w2c = mv_w2c[cam_id].to(device)\\n            R = w2c[:3, :3]\\n            t = w2c[:3, 3]\\n\\n            for mask_path in mask_files:\\n                m_np = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)\\n                if m_np is None:\\n                    continue\\n                H_m, W_m = m_np.shape\\n                mask_t = torch.from_numpy(m_np > 127).to(device)\\n\\n                # Project gauss centers to this cam\\n                pts_cam = (R @ means.T).T + t  # [N, 3]\\n                z = pts_cam[:, 2]\\n                valid_z = z > 0.1\\n                # Project (in raw resolution that K maps to)\\n                uv_h = (K @ pts_cam.T).T\\n                u = uv_h[:, 0] / uv_h[:, 2].clamp(min=0.1)\\n                v = uv_h[:, 1] / uv_h[:, 2].clamp(min=0.1)\\n                # Map to mask resolution if different (K and mask should match)\\n                in_frame = valid_z & (u >= 0) & (u < W_m) & (v >= 0) & (v < H_m)\\n                if not in_frame.any():\\n                    continue\\n\\n                ui = u[in_frame].long().clamp(0, W_m - 1)\\n                vi = v[in_frame].long().clamp(0, H_m - 1)\\n                pix_motion = mask_t[vi, ui]  # [N_in_frame] bool\\n                # Update votes for in-frame gauss\\n                idx_in = torch.where(in_frame)[0]\\n                dynamic_votes[idx_in[pix_motion]] += 1\\n                n_total_views += 1\\n\\n        if n_total_views == 0:\\n            print(f\\"[trainer.static_dyn] Hicbir mask okunamadi, skip\\")\\n            return\\n\\n        threshold = max(1, int(n_total_views * threshold_frac))\\n        dynamic_mask = dynamic_votes > threshold\\n        n_promoted = self.gs.promote_dynamic(dynamic_mask)\\n        self.use_static_dynamic_split = True\\n        print(f\\"[trainer.static_dyn] {n_promoted:,} gauss DYNAMIC \\"\\n              f\\"({100 * n_promoted / max(n_pts, 1):.1f}%) \\"\\n              f\\"(threshold {threshold}/{n_total_views} views)\\")\\n        print(f\\"  Static: {self.gs.num_static:,}, Dynamic: {self.gs.num_dynamic:,}\\")\\n\\n    @torch.no_grad()\\n    def _setup_static_dynamic_from_masks_sv(\\n        self,\\n        masks_dir: Path,\\n        frame_paths: list,\\n        cam_K: torch.Tensor,\\n        w2c_list: list,\\n        threshold_frac: float = 0.10,\\n    ) -> None:\\n        \\"\\"\\"Phase 2.1 — Single-view motion mask voting (tek cam, T frame).\\n\\n        Multi-view varyantının single-view portu. Her gauss tek cam\'a\\n        her frame\'de project edilir, mask motion piksellerinde ise vote.\\n        \\"\\"\\"\\n        import cv2\\n        masks_dir = Path(masks_dir)\\n        if not masks_dir.exists():\\n            print(f\\"[trainer.static_dyn] masks yok: {masks_dir}, skip\\")\\n            return\\n        if not hasattr(self.gs, \\"promote_dynamic\\"):\\n            return\\n\\n        n_pts = self.gs.num_points\\n        device = self.gs.means.device\\n        dynamic_votes = torch.zeros(n_pts, dtype=torch.int32, device=device)\\n        n_total = 0\\n\\n        means = self.gs.means.detach()\\n        K = cam_K.to(device)\\n\\n        # mask file naming: mask_0001.png (frame index 0-indexed → 1-indexed?)\\n        # dynamic_mask.py: mask_path = out / f\\"mask_{i+1:04d}.png\\" (1-indexed)\\n        mask_files = sorted(masks_dir.glob(\\"mask_*.png\\"))\\n        if not mask_files:\\n            print(f\\"[trainer.static_dyn] mask_*.png yok: {masks_dir}\\")\\n            return\\n\\n        T_avail = min(len(mask_files), len(frame_paths), len(w2c_list))\\n        for t in range(T_avail):\\n            m_np = cv2.imread(str(mask_files[t]), cv2.IMREAD_GRAYSCALE)\\n            if m_np is None:\\n                continue\\n            H_m, W_m = m_np.shape\\n            mask_t = torch.from_numpy(m_np > 127).to(device)\\n\\n            w2c = w2c_list[t].to(device)\\n            R = w2c[:3, :3]\\n            tvec = w2c[:3, 3]\\n            pts_cam = (R @ means.T).T + tvec\\n            z = pts_cam[:, 2]\\n            valid_z = z > 0.1\\n            uv_h = (K @ pts_cam.T).T\\n            u = uv_h[:, 0] / uv_h[:, 2].clamp(min=0.1)\\n            v = uv_h[:, 1] / uv_h[:, 2].clamp(min=0.1)\\n            in_frame = valid_z & (u >= 0) & (u < W_m) & (v >= 0) & (v < H_m)\\n            if not in_frame.any():\\n                continue\\n            ui = u[in_frame].long().clamp(0, W_m - 1)\\n            vi = v[in_frame].long().clamp(0, H_m - 1)\\n            pix_motion = mask_t[vi, ui]\\n            idx_in = torch.where(in_frame)[0]\\n            dynamic_votes[idx_in[pix_motion]] += 1\\n            n_total += 1\\n\\n        if n_total == 0:\\n            print(f\\"[trainer.static_dyn] Hicbir mask okunmadi, skip\\")\\n            return\\n\\n        threshold = max(1, int(n_total * threshold_frac))\\n        dynamic_mask = dynamic_votes > threshold\\n        n_promoted = self.gs.promote_dynamic(dynamic_mask)\\n        self.use_static_dynamic_split = True\\n        print(f\\"[trainer.static_dyn.sv] {n_promoted:,} gauss DYNAMIC \\"\\n              f\\"({100 * n_promoted / max(n_pts, 1):.1f}%) \\"\\n              f\\"(threshold {threshold}/{n_total} frames)\\")\\n        print(f\\"  Static: {self.gs.num_static:,}, Dynamic: {self.gs.num_dynamic:,}\\")\\n\\n    def _preload_all_to_ram(\\n        self,\\n        T: int,\\n        target_size: tuple[int, int],\\n        is_multiview: bool,\\n        # SV\\n        frame_paths: Sequence[Path] | None = None,\\n        depth_dir: Path | None = None,\\n        mask_dir: Path | None = None,\\n        frames_cached: list | None = None,\\n        depth_cached: list | None = None,\\n        mask_cached: list | None = None,\\n        # MV\\n        train_cams: list | None = None,\\n        mv_frame_paths: dict | None = None,\\n        depth_mv_dir: Path | None = None,\\n        masks_mv_dir: Path | None = None,  # Reserved; MV does not preload masks (consumed at startup only)\\n        frames_cached_mv: dict | None = None,\\n        depth_cached_mv: dict | None = None,\\n        max_workers: int = 16,\\n    ) -> None:\\n        \\"\\"\\"Eagerly load all frames/depth/masks into RAM at master resolution.\\n\\n        Uses ThreadPoolExecutor — most time is spent in cv2.imread/np.load (IO + decode),\\n        which releases the GIL. Each task writes to a pre-allocated cache slot.\\n\\n        Cache dtypes (memory-efficient):\\n          - frames: uint8 (3.5x savings vs float32)\\n          - depth:  fp16  (2x savings)\\n          - mask:   uint8 (4x savings)\\n        Caller (per-iter load site) converts to float32 on GPU after .to(device).\\n        \\"\\"\\"\\n        # Build the task list. Each task = (cache_slot_setter, callable that returns tensor).\\n        tasks: list[tuple[Callable[[torch.Tensor | None], None], Callable[[], torch.Tensor | None]]] = []\\n\\n        if is_multiview:\\n            assert train_cams is not None and mv_frame_paths is not None and frames_cached_mv is not None\\n            for cam_id in train_cams:\\n                paths_for_cam = mv_frame_paths[cam_id]\\n                cache_frames = frames_cached_mv[cam_id]\\n                cache_depth = depth_cached_mv[cam_id] if depth_cached_mv is not None else None\\n                for i in range(T):\\n                    fp = Path(paths_for_cam[i])\\n                    # RGB\\n                    def _set_rgb(t, cache=cache_frames, idx=i):\\n                        cache[idx] = t\\n                    def _load_rgb(p=fp, ts=target_size):\\n                        return load_frame_tensor(p, ts, dtype=\\"uint8\\")\\n                    tasks.append((_set_rgb, _load_rgb))\\n                    # Depth\\n                    if cache_depth is not None and depth_mv_dir is not None:\\n                        depth_subdir = Path(depth_mv_dir) / cam_id\\n                        def _set_d(t, cache=cache_depth, idx=i):\\n                            cache[idx] = t\\n                        def _load_d(d=depth_subdir, p=fp, ts=target_size):\\n                            return _load_depth_for_frame(d, p, ts, dtype=\\"float16\\")\\n                        tasks.append((_set_d, _load_d))\\n            # Note: MV path does NOT use per-frame mask cache at training time\\n            # (mask_cached_mv is not part of the existing trainer state). Masks\\n            # are consumed only by _setup_static_dynamic_from_masks at startup.\\n        else:\\n            assert frame_paths is not None and frames_cached is not None\\n            for i, fp in enumerate(frame_paths):\\n                fp = Path(fp)\\n                # RGB\\n                def _set_rgb(t, idx=i):\\n                    frames_cached[idx] = t\\n                def _load_rgb(p=fp, ts=target_size):\\n                    return load_frame_tensor(p, ts, dtype=\\"uint8\\")\\n                tasks.append((_set_rgb, _load_rgb))\\n                # Depth\\n                if depth_cached is not None and depth_dir is not None:\\n                    def _set_d(t, idx=i):\\n                        depth_cached[idx] = t\\n                    def _load_d(d=depth_dir, p=fp, ts=target_size):\\n                        return _load_depth_for_frame(d, p, ts, dtype=\\"float16\\")\\n                    tasks.append((_set_d, _load_d))\\n                # Mask\\n                if mask_cached is not None and mask_dir is not None:\\n                    def _set_m(t, idx=i):\\n                        mask_cached[idx] = t\\n                    def _load_m(d=mask_dir, p=fp, ts=target_size):\\n                        return _load_mask_for_frame(d, p, ts, dtype=\\"uint8\\")\\n                    tasks.append((_set_m, _load_m))\\n\\n        total = len(tasks)\\n        if total == 0:\\n            print(\\"[preload] hicbir task yok, skip\\")\\n            return\\n\\n        print(f\\"[preload] {total} task baslatiliyor \\"\\n              f\\"(workers={max_workers}, target_size={target_size}, \\"\\n              f\\"is_multiview={is_multiview}, T={T})\\")\\n\\n        t_start = time.time()\\n        progress_step = max(1, total // 10)\\n        done = 0\\n        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:\\n            futures = []\\n            for setter, loader in tasks:\\n                fut = ex.submit(loader)\\n                futures.append((fut, setter))\\n            for fut, setter in futures:\\n                try:\\n                    result = fut.result()\\n                    setter(result)\\n                except Exception as e:\\n                    # Don\'t kill preload on a single bad file; log + leave None.\\n                    print(f\\"[preload] task hatasi: {e}\\")\\n                done += 1\\n                if done % progress_step == 0 or done == total:\\n                    elapsed = time.time() - t_start\\n                    rate = done / max(elapsed, 1e-6)\\n                    eta = (total - done) / max(rate, 1e-6)\\n                    print(f\\"[preload] {done}/{total} loaded \\"\\n                          f\\"({elapsed:.1f}s, {rate:.0f} task/s, eta {eta:.1f}s)\\")\\n\\n        # Pin host memory so subsequent .to(device, non_blocking=True) is faster.\\n        # Optional best-effort — wrap in try; skip on failure (e.g. OOM).\\n        try:\\n            self._pin_cache_in_place(\\n                frames_cached_mv if is_multiview else frames_cached,\\n                depth_cached_mv if (is_multiview and depth_cached_mv is not None) else depth_cached,\\n                mask_cached if not is_multiview else None,\\n                is_multiview=is_multiview,\\n            )\\n        except Exception as _e:\\n            print(f\\"[preload] pin_memory skipped: {_e}\\")\\n\\n        elapsed_total = time.time() - t_start\\n        print(f\\"[preload] DONE — {total} tensors in {elapsed_total:.1f}s\\")\\n\\n    @staticmethod\\n    def _pin_cache_in_place(frames_obj, depth_obj, mask_obj, is_multiview: bool) -> None:\\n        \\"\\"\\"Pin host memory of every cached tensor in place. Best-effort.\\"\\"\\"\\n        def _pin_list(lst):\\n            if lst is None:\\n                return\\n            for i, t in enumerate(lst):\\n                if t is not None and not t.is_pinned():\\n                    try:\\n                        lst[i] = t.pin_memory()\\n                    except Exception:\\n                        pass\\n        if is_multiview:\\n            if frames_obj is not None:\\n                for c, lst in frames_obj.items():\\n                    _pin_list(lst)\\n            if depth_obj is not None:\\n                for c, lst in depth_obj.items():\\n                    _pin_list(lst)\\n        else:\\n            _pin_list(frames_obj)\\n            _pin_list(depth_obj)\\n            _pin_list(mask_obj)\\n\\n    def _prepare_tracks(\\n        self,\\n        tracks_path: Path,\\n        frame_paths: Sequence[Path],\\n        cams_K: torch.Tensor,\\n        cams_w2c: Sequence[torch.Tensor],\\n        depth_dir: Path | None,\\n        frame_original_size: tuple[int, int],  # (W_frame, H_frame)\\n    ) -> dict | None:\\n        \\"\\"\\"\\n        CoTracker tracks\'i yükle, frame 0\'daki pixel koordinatlarını\\n        MiDaS depth\'iyle 3D world anchor\'a lift et.\\n        Return: {\\n            \\"anchors_3d\\": (N_valid, 3),\\n            \\"tracks_2d\\":  (T_tr, N_valid, 2) — frame resolution\'a scale edilmiş,\\n            \\"visibility\\": (T_tr, N_valid),\\n            \\"anchor_indices_original\\": (N_valid,) CoTracker içi index,\\n        } ya da None.\\n        \\"\\"\\"\\n        if not tracks_path.exists():\\n            return None\\n        try:\\n            td = torch.load(tracks_path, map_location=\\"cpu\\", weights_only=True)\\n        except Exception:\\n            td = torch.load(tracks_path, map_location=\\"cpu\\")\\n        tracks_2d_v = td[\\"tracks\\"][0]       # (T, N, 2) video resolution\\n        visibility = td[\\"visibility\\"][0]    # (T, N) bool/float\\n        v_shape = td[\\"video_shape\\"]         # (1, T, 3, H_v, W_v)\\n        H_v, W_v = int(v_shape[-2]), int(v_shape[-1])\\n        T_tr, N_tr = tracks_2d_v.shape[0], tracks_2d_v.shape[1]\\n        W_f, H_f = frame_original_size\\n\\n        # video → frame resolution scaling\\n        sx = W_f / W_v\\n        sy = H_f / H_v\\n        tracks_2d_f = tracks_2d_v.clone().float()\\n        tracks_2d_f[..., 0] *= sx\\n        tracks_2d_f[..., 1] *= sy\\n\\n        # Lift frame 0 tracks to 3D\\n        if depth_dir is None:\\n            print(\\"[trainer.track] depth_dir yok — track loss kapalı\\")\\n            return None\\n        depth_0 = _load_depth_for_frame(depth_dir, frame_paths[0])\\n        if depth_0 is None:\\n            print(\\"[trainer.track] frame 0 depth bulunamadı — track loss kapalı\\")\\n            return None\\n\\n        tracks_f0 = tracks_2d_f[0]  # (N, 2)\\n        vis_f0 = visibility[0].bool() if visibility.dtype != torch.bool else visibility[0]\\n\\n        # Pixel-safe: clamp to [0, W-1] x [0, H-1]\\n        u0 = tracks_f0[:, 0].clamp(0, W_f - 1).long()\\n        v0 = tracks_f0[:, 1].clamp(0, H_f - 1).long()\\n        # Depth lookup\\n        depth_at_tracks = depth_0[v0, u0]\\n        # Valid: visible + positive depth\\n        valid = vis_f0 & (depth_at_tracks > 1e-3)\\n        if valid.sum().item() < 32:\\n            print(f\\"[trainer.track] sadece {valid.sum().item()} geçerli anchor — track loss kapalı\\")\\n            return None\\n\\n        tracks_f0_valid = tracks_f0[valid].to(self.device)      # (N\', 2)\\n        depths_valid = depth_at_tracks[valid].to(self.device)   # (N\',)\\n\\n        # K and w2c for frame 0 at FRAME resolution (not training resolution)\\n        # cams_K / cams_w2c come in at frame resolution\\n        K_0 = cams_K.to(self.device)\\n        w2c_0 = cams_w2c[0].to(self.device)\\n\\n        anchors_3d = _unproject_pixel_to_world(tracks_f0_valid, depths_valid, K_0, w2c_0)\\n\\n        # Keep per-frame tracks 2D + visibility for valid subset\\n        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)\\n        tracks_2d_valid = tracks_2d_f[:, valid_idx, :].to(self.device)    # (T, N\', 2)\\n        vis_valid = visibility[:, valid_idx].to(self.device)               # (T, N\')\\n        if vis_valid.dtype != torch.bool:\\n            vis_valid = vis_valid > 0.5\\n\\n        print(f\\"[trainer.track] {anchors_3d.shape[0]} 3D anchor hazır \\"\\n              f\\"({T_tr} frame boyunca izleniyor)\\")\\n        return {\\n            \\"anchors_3d\\": anchors_3d,\\n            \\"tracks_2d\\":  tracks_2d_valid,\\n            \\"visibility\\": vis_valid,\\n            \\"T_tr\\": T_tr,\\n        }\\n\\n    def train(\\n        self,\\n        frame_paths: Sequence[Path],\\n        cam_K: torch.Tensor,\\n        cam_w2c_per_frame: Sequence[torch.Tensor],\\n        n_iters: int = 30_000,\\n        image_size: tuple[int, int] = (640, 360),\\n        ckpt_dir: Path | None = None,\\n        ckpt_interval: int = 1000,\\n        log_interval: int = 50,\\n        progress_callback: TrainProgressCallback | None = None,\\n        depth_dir: Path | None = None,\\n        mask_dir: Path | None = None,\\n        tracks_path: Path | None = None,\\n        # Phase 1.8 single-view — RAFT optical flow path (forward_*.pt)\\n        flow_dir: Path | None = None,\\n        run_logger: Any = None,   # RunLogger instance, opsiyonel\\n        mv_frame_paths=None,\\n        mv_cam_K=None,\\n        mv_w2c=None,\\n        mv_test_camera=None,\\n        # Phase 1.4 — Per-cam MV depth supervision\\n        depth_mv_dir: Path | None = None,\\n        # Phase 1.5 — Per-cam MV dynamic mask (Phase 2.1 promote için)\\n        masks_mv_dir: Path | None = None,\\n        # Phase 1.8 — Per-cam MV RAFT optical flow\\n        flow_mv_dir: Path | None = None,\\n        # Phase 2.1 — Static/Dynamic auto-promote\\n        auto_static_dynamic: bool = False,\\n        static_dynamic_threshold: float = 0.10,\\n        # Static 3DGS Faz 1 — 4D features bypass\\n        static_mode: bool = False,\\n        # 4D Quality v6.1 — runtime knobs\\n        sh_progressive_schedule: bool = False,\\n        lambda_accel: float = 0.0,\\n        cam_grad_clip_norm: float = 0.0,\\n        mip_scale_floor_frac: float = 0.0,\\n        dynamic_densify_scale: float = 1.0,\\n        # Perf — RAM preload + uint8/fp16 cache + multires fix\\n        preload_to_ram: bool = False,\\n        # NVS hold-out — frame indices to EXCLUDE from training (single-view path).\\n        # Eval can then measure true novel-view PSNR on these. None = train on all.\\n        # Multi-view path uses mv_test_camera for held-out; this is SV-only.\\n        holdout_indices: Sequence[int] | None = None,\\n        # Cooperative cancel: callable returning True when training should stop.\\n        # Polled every iter; if True, the loop exits cleanly and the function\\n        # returns the partial history. Caller transitions the job to a\\n        # cancelled/failed state. Default = no-op (training runs to completion).\\n        cancel_check: Callable[[], bool] | None = None,\\n        # Edit refit: per-frame inpaint mask. When provided, depth supervision\\n        # is skipped (zeroed) for masked pixels (no GT depth in inpainted\\n        # regions).  Shape: (T, H, W) bool, on CPU — fetched per-iter.\\n        edit_mask_stack: torch.Tensor | None = None,\\n    ) -> dict:\\n        # Static mode — 4D dynamic features bypass (deformation, Fourier, motion regs)\\n        self.static_mode = bool(static_mode)\\n        if self.static_mode:\\n            print(f\\"\\\\n[trainer.STATIC] Static 3DGS modu — 4D dynamic features kapalı\\")\\n        # 4D Quality v6.1 — runtime knobs\\n        self.sh_progressive_schedule = bool(sh_progressive_schedule)\\n        self.lambda_accel = float(lambda_accel)\\n        self.cam_grad_clip_norm = float(cam_grad_clip_norm)\\n        self.mip_scale_floor_frac = float(mip_scale_floor_frac)\\n        self.dynamic_densify_scale = float(dynamic_densify_scale)\\n        if self.sh_progressive_schedule:\\n            print(f\\"[trainer.v6.1] SH progressive schedule aktif \\"\\n                  f\\"(0→{self.gs.sh_degree} degree over n_iters)\\")\\n        if self.lambda_accel > 0:\\n            print(f\\"[trainer.v6.1] 2nd-order accel reg aktif (λ={self.lambda_accel})\\")\\n        if self.mip_scale_floor_frac > 0:\\n            print(f\\"[trainer.v6.1] Mip-Splatting Python-side aktif \\"\\n                  f\\"(scale_floor_frac={self.mip_scale_floor_frac})\\")\\n        if self.cam_grad_clip_norm > 0 and (self.lr_cam_K > 0 or self.lr_cam_w2c > 0):\\n            print(f\\"[trainer.v6.1] Cam refine grad clip = {self.cam_grad_clip_norm}\\")\\n        # v5.0: Multi-view detect\\n        is_multiview = mv_frame_paths is not None\\n        # v5.0 ARCHITECTURAL FIX: Static warmup phase.\\n        # Multi-view\'da deformation MLP, statik geometri oturmadan aktive\\n        # olursa per-cam noise\'u \\"deformasyon\\" olarak ogrenip patliyor.\\n        # Standart 4DGaussians yaklasimi: ilk N iter sadece statik gaussian\\n        # + densify, sonra deformation devreye.\\n        #\\n        # v5.0.1 (uzun run optimizasyonu): static phase\'i CAP\'le. Geometry\\n        # genelde 5-8k iter\'de oturur. 50k iter run icin 25k static phase\\n        # gereksiz uzun, dynamic phase\'e zaman birak. min(n_iters//2, 8000).\\n        # Single-view\'da bu sorun yok, iter 1\'den deformation aktif kalabilir.\\n        # Static 3DGS modu: tüm training boyunca deformation kapalı, warmup gerekmez\\n        if self.static_mode:\\n            self._static_phase_iters = 0\\n        else:\\n            self._static_phase_iters = min(n_iters // 2, 8000) if is_multiview else 0\\n        if is_multiview and not self.static_mode:\\n            print(f\\"[trainer.mv] Static warmup phase: {self._static_phase_iters} iter \\"\\n                  f\\"(deformation tamamen donuk)\\")\\n            # Phase 1.9 — Multi-view densify tuning: threshold\'u scale ile carp.\\n            # MV\'de her cam ayri view; densify daha hassas olmali.\\n            mv_scale = float(getattr(self, \\"densify_mv_threshold_scale\\", 1.0))\\n            if 0.0 < mv_scale < 2.0 and abs(mv_scale - 1.0) > 1e-3:\\n                old_thr = self.density.grad_threshold\\n                self.density.grad_threshold = old_thr * mv_scale\\n                print(f\\"[trainer.mv] Densify grad_threshold {old_thr:.6f} → \\"\\n                      f\\"{self.density.grad_threshold:.6f} (scale={mv_scale}, MV daha hassas)\\")\\n\\n        if is_multiview:\\n            train_cams = sorted([c for c in mv_frame_paths.keys() if c != mv_test_camera])\\n            if not train_cams:\\n                raise ValueError(\\"mv_frame_paths bos veya test_camera disinda cam yok\\")\\n            cam_T_set = {len(mv_frame_paths[c]) for c in train_cams}\\n            if len(cam_T_set) != 1:\\n                raise ValueError(f\\"Cam\'lar arasinda farkli frame sayisi: {cam_T_set}\\")\\n            T = cam_T_set.pop()\\n            print(f\\"[trainer.mv] Multi-view: {len(train_cams)} train cam x {T} frame \\"\\n                  f\\"(test_cam={mv_test_camera})\\")\\n            ref_cam = train_cams[0]\\n            cam_K_orig = mv_cam_K[ref_cam].to(self.device)\\n            w2c_list = None\\n        else:\\n            T = len(frame_paths)\\n            if len(cam_w2c_per_frame) != T:\\n                raise ValueError(\\n                    f\\"frame_paths ve cam_w2c_per_frame uzunluklari farkli: {T} vs {len(cam_w2c_per_frame)}\\"\\n                )\\n            cam_K_orig = cam_K.to(self.device)\\n            w2c_list = [w.to(self.device) for w in cam_w2c_per_frame]\\n\\n        Ws, Hs = image_size\\n        if is_multiview:\\n            sample = load_frame_tensor(Path(mv_frame_paths[train_cams[0]][0]))\\n        else:\\n            sample = load_frame_tensor(Path(frame_paths[0]))\\n        H0, W0 = sample.shape[:2]             # original frame resolution\\n\\n        # Phase 2.2 — Multi-resolution schedule resolver\\n        # multires_schedule: [(start_iter, long_edge), ...]; bos = single resolution.\\n        def _resolve_long_edge_for_iter(it_check: int) -> int | None:\\n            sched = self.multires_schedule\\n            if not sched:\\n                return None\\n            current = None\\n            for (it_start, le) in sorted(sched, key=lambda x: x[0]):\\n                if it_check >= it_start:\\n                    current = le\\n            return current\\n\\n        def _resize_to_long_edge(le: int, base_w: int, base_h: int):\\n            ratio = le / max(base_w, base_h)\\n            return int(round(base_w * ratio)), int(round(base_h * ratio))\\n\\n        # Initial resolution: schedule var ise schedule[0] kullan, yoksa image_size\\n        initial_le = _resolve_long_edge_for_iter(0)\\n        if initial_le is not None:\\n            Ws, Hs = _resize_to_long_edge(initial_le, W0, H0)\\n            print(f\\"[trainer.multires] Initial schedule: long_edge={initial_le} -> ({Ws}x{Hs})\\")\\n\\n        # Perf — Master cache resolution.\\n        # OLD behavior: cache stored tensors at Ws,Hs. Multires resolution change\\n        # forced full disk re-read of all T frames per cam (5040x cam at 1080p MFS\\n        # = ~5min thrash per resolution step).\\n        # NEW: cache stores at LARGEST resolution that will appear in the schedule.\\n        # Per-iter, downsample on GPU via F.interpolate (bilinear+antialias for RGB,\\n        # bilinear for depth, nearest for mask). Resolution change is now free —\\n        # just toggle (Ws, Hs); cache untouched.\\n        if self.multires_schedule:\\n            max_le = max(le for (_, le) in self.multires_schedule)\\n            Wm, Hm = _resize_to_long_edge(max_le, W0, H0)\\n            # Master must be >= every scheduled resolution; if image_size > schedule\\n            # max (edge case), keep image_size as master.\\n            if Wm < image_size[0] or Hm < image_size[1]:\\n                Wm, Hm = image_size\\n        else:\\n            # No schedule — master == single training resolution.\\n            Wm, Hm = Ws, Hs\\n        print(f\\"[trainer.cache] Master cache resolution: ({Wm}x{Hm}) \\"\\n              f\\"(current train ({Ws}x{Hs}); preload_to_ram={preload_to_ram})\\")\\n\\n        sx, sy = Ws / W0, Hs / H0\\n        K_scaled = cam_K_orig.clone()         # training resolution (single-view fallback)\\n        K_scaled[0, 0] *= sx; K_scaled[0, 2] *= sx\\n        K_scaled[1, 1] *= sy; K_scaled[1, 2] *= sy\\n\\n        # v5.0: Per-cam K_scaled + w2c cache (multi-view)\\n        K_scaled_mv = {}\\n        w2c_mv = {}\\n        if is_multiview:\\n            for c in train_cams:\\n                K_c = mv_cam_K[c].to(self.device).clone()\\n                K_c[0, 0] *= sx; K_c[0, 2] *= sx\\n                K_c[1, 1] *= sy; K_c[1, 2] *= sy\\n                K_scaled_mv[c] = K_c\\n                w2c_mv[c] = mv_w2c[c].to(self.device)\\n\\n        # Phase 2.3 — Camera pose refine: K_scaled_mv ve w2c_mv\'yi nn.Parameter\'a wrap et\\n        cam_refine_active = (\\n            is_multiview and (self.lr_cam_K > 0 or self.lr_cam_w2c > 0)\\n        )\\n        if cam_refine_active:\\n            cam_refine_params: list[dict] = []\\n            for c in train_cams:\\n                if self.lr_cam_K > 0:\\n                    K_p = torch.nn.Parameter(K_scaled_mv[c].clone().detach())\\n                    K_scaled_mv[c] = K_p\\n                    cam_refine_params.append({\\"params\\": [K_p], \\"lr\\": self.lr_cam_K})\\n                if self.lr_cam_w2c > 0:\\n                    w_p = torch.nn.Parameter(w2c_mv[c].clone().detach())\\n                    w2c_mv[c] = w_p\\n                    cam_refine_params.append({\\"params\\": [w_p], \\"lr\\": self.lr_cam_w2c})\\n            if cam_refine_params:\\n                self._cam_refine_optimizer = torch.optim.Adam(cam_refine_params)\\n                print(f\\"[trainer.cam_refine] Aktif: {len(train_cams)} cam, \\"\\n                      f\\"{len(cam_refine_params)} param group, \\"\\n                      f\\"start_iter={self.cam_refine_start_iter}\\")\\n            print(f\\"[trainer.mv] K scaled per-cam, {len(train_cams)} entries\\")\\n\\n        # Phase 2.1 — Static/Dynamic auto-promote.\\n        # Mask voting ile ilk N iter\'den ONCE statik/dinamik secimini yap,\\n        # boylece training boyunca statik bolgeler deformation BYPASS eder.\\n        # Static 3DGS Faz 1 — static_mode\'da deformation komple kapali, dynamic\\n        # ayrimi anlamsiz; auto-promote\'u atla.\\n        if self.static_mode:\\n            auto_static_dynamic = False\\n        if (is_multiview and auto_static_dynamic and masks_mv_dir is not None\\n                and hasattr(self.gs, \\"promote_dynamic\\")):\\n            print(f\\"\\\\n[trainer.static_dyn] Phase 2.1 auto-promote (MV) — \\"\\n                  f\\"mask voting threshold={static_dynamic_threshold}\\")\\n            self._setup_static_dynamic_from_masks(\\n                masks_mv_dir=masks_mv_dir,\\n                train_cams=train_cams,\\n                mv_cam_K=mv_cam_K,\\n                mv_w2c=mv_w2c,\\n                threshold_frac=static_dynamic_threshold,\\n            )\\n        elif (not is_multiview and auto_static_dynamic and mask_dir is not None\\n              and hasattr(self.gs, \\"promote_dynamic\\")):\\n            print(f\\"\\\\n[trainer.static_dyn] Phase 2.1 auto-promote (SV) — \\"\\n                  f\\"mask voting threshold={static_dynamic_threshold}\\")\\n            self._setup_static_dynamic_from_masks_sv(\\n                masks_dir=mask_dir,\\n                frame_paths=frame_paths,\\n                cam_K=cam_K_orig,\\n                w2c_list=w2c_list,\\n                threshold_frac=static_dynamic_threshold,\\n            )\\n\\n        # Cache\'ler — stored at master resolution (Wm, Hm); per-iter downsampled on GPU.\\n        # uint8 RGB / fp16 depth / uint8 mask (memory-efficient). Caller converts\\n        # to float on GPU after .to(device) (free with non_blocking=True).\\n        frames_cached: list = [None] * T\\n        depth_cached:  list = [None] * T\\n        mask_cached:   list = [None] * T\\n        # v5.0: Multi-view per-cam frame cache\\n        frames_cached_mv = {c: [None] * T for c in train_cams} if is_multiview else None\\n        # Phase 1.4 — Multi-view per-cam depth cache\\n        depth_cached_mv = (\\n            {c: [None] * T for c in train_cams}\\n            if (is_multiview and depth_mv_dir is not None and self.lambda_depth > 0)\\n            else None\\n        )\\n        # Phase 1.8 — Multi-view per-cam flow cache (sparse, only if file exists)\\n        flow_cached_mv = (\\n            {c: [None] * T for c in train_cams}\\n            if (is_multiview and flow_mv_dir is not None and self.lambda_flow > 0)\\n            else None\\n        )\\n        # Phase 1.8 single-view — flow cache (single timeline, T frame)\\n        flow_cached_sv = (\\n            [None] * T\\n            if (not is_multiview and flow_dir is not None and self.lambda_flow > 0)\\n            else None\\n        )\\n\\n        use_depth = depth_dir is not None and self.lambda_depth > 0\\n        use_mask  = mask_dir is not None\\n        use_depth_mv = (\\n            is_multiview and depth_mv_dir is not None and self.lambda_depth > 0\\n        )\\n        use_flow_mv = (\\n            is_multiview and flow_mv_dir is not None and self.lambda_flow > 0\\n        )\\n        use_flow_sv = (\\n            not is_multiview and flow_dir is not None and self.lambda_flow > 0\\n        )\\n        if use_depth:\\n            print(f\\"[trainer] Depth loss aktif (lambda={self.lambda_depth})\\")\\n        if use_depth_mv:\\n            print(f\\"[trainer.mv] Phase 1.4 Depth-MV supervision: \\"\\n                  f\\"lambda={self.lambda_depth}, dir={depth_mv_dir}\\")\\n        if use_flow_mv:\\n            print(f\\"[trainer.mv] Phase 1.8 Flow-MV supervision: \\"\\n                  f\\"lambda={self.lambda_flow}, dir={flow_mv_dir}\\")\\n        if use_flow_sv:\\n            print(f\\"[trainer.sv] Phase 1.8 Flow-SV supervision: \\"\\n                  f\\"lambda={self.lambda_flow}, dir={flow_dir}\\")\\n        if use_mask:\\n            print(f\\"[trainer] Mask-weighted recon aktif (lambda={self.lambda_mask_motion})\\")\\n\\n        # Perf — Eager preload all frames/depth/masks into RAM via ThreadPoolExecutor.\\n        # Eliminates per-iter MFS network filesystem latency. uint8 RGB (~3.5x mem\\n        # savings vs float32) + fp16 depth + uint8 mask. Cast to float on GPU after\\n        # .to(device, non_blocking=True) — free with pinned host memory.\\n        if preload_to_ram:\\n            self._preload_all_to_ram(\\n                T=T,\\n                target_size=(Wm, Hm),\\n                is_multiview=is_multiview,\\n                # SV\\n                frame_paths=(frame_paths if not is_multiview else None),\\n                depth_dir=(depth_dir if (not is_multiview and use_depth) else None),\\n                mask_dir=(mask_dir if (not is_multiview and use_mask) else None),\\n                frames_cached=(frames_cached if not is_multiview else None),\\n                depth_cached=(depth_cached if (not is_multiview and use_depth) else None),\\n                mask_cached=(mask_cached if (not is_multiview and use_mask) else None),\\n                # MV\\n                train_cams=(train_cams if is_multiview else None),\\n                mv_frame_paths=(mv_frame_paths if is_multiview else None),\\n                depth_mv_dir=(depth_mv_dir if (is_multiview and use_depth_mv) else None),\\n                masks_mv_dir=(masks_mv_dir if (is_multiview and use_mask) else None),\\n                frames_cached_mv=(frames_cached_mv if is_multiview else None),\\n                depth_cached_mv=(depth_cached_mv if (is_multiview and use_depth_mv) else None),\\n            )\\n\\n        # Track data (CoTracker 3D anchors)\\n        track_data = None\\n        if tracks_path is not None and self.lambda_track > 0:\\n            try:\\n                track_data = self._prepare_tracks(\\n                    tracks_path, frame_paths, cam_K_orig, w2c_list,\\n                    depth_dir, (W0, H0),\\n                )\\n            except Exception as e:\\n                print(f\\"⚠ Track loss prep failed: {e}\\")\\n                track_data = None\\n        if track_data is not None:\\n            print(f\\"[trainer] Track loss aktif (lambda={self.lambda_track})\\")\\n\\n        history = {\\"loss\\": [], \\"psnr\\": [], \\"n_pts\\": []}\\n        t0 = time.time()\\n\\n        # v3.2: lr_deform için warmup — ilk warmup_iters iter\'de 0 → target\'a ramp up\\n        # Önce yüksek lr_deform MLP\'yi patlatıp Δpos=41 spike\'ları yaratıyordu.\\n        # v3.6: param group 6 = deform (MLP), group 7 (varsa) = fourier_pos_coeffs.\\n        # Her ikisini de warmup\'la ramp up.\\n        DEFORM_GROUP_IDX = 6   # deform MLP\\n        FOURIER_GROUP_IDX = 7  # fourier (yoksa len < 8)\\n        # Static 3DGS modu: deformation/fourier optimizer\'ları LR=0\\n        target_lr_deform = 0.0 if self.static_mode else self.lr_deform\\n        target_lr_fourier = 0.0 if self.static_mode else self.lr_fourier\\n        has_fourier_group = len(self.optimizer.param_groups) > FOURIER_GROUP_IDX\\n\\n        # 4D Quality v6.1 — Madde 2-B: Mip-Splatting cam_centers (training boyunca sabit)\\n        # Performans icin max 32 cam kullaniyoruz — her gauss\'in NEAREST cam\'i\\n        # tahmini icin yeterli, distance hesabi sabit-zaman.\\n        mip_cam_centers = None\\n        MIP_MAX_CAMS = 32\\n        if self.mip_scale_floor_frac > 0:\\n            try:\\n                cam_centers_list = []\\n                if is_multiview:\\n                    for c in train_cams:\\n                        w2c = mv_w2c[c]\\n                        cam_center = -w2c[:3, :3].T @ w2c[:3, 3]\\n                        cam_centers_list.append(cam_center)\\n                else:\\n                    for w2c in w2c_list:\\n                        cam_center = -w2c[:3, :3].T @ w2c[:3, 3]\\n                        cam_centers_list.append(cam_center)\\n                if cam_centers_list:\\n                    full_centers = torch.stack(cam_centers_list).to(self.device)\\n                    M = full_centers.shape[0]\\n                    if M > MIP_MAX_CAMS:\\n                        # Uniform stride subsample (ilk + son + arası eşit aralıklı)\\n                        stride = max(1, M // MIP_MAX_CAMS)\\n                        idx = torch.arange(0, M, stride, device=full_centers.device)[:MIP_MAX_CAMS]\\n                        mip_cam_centers = full_centers[idx]\\n                        print(f\\"[trainer.v6.1] Mip-Splatting cam_centers: {M} → {mip_cam_centers.shape[0]} (subsample, performans)\\")\\n                    else:\\n                        mip_cam_centers = full_centers\\n                        print(f\\"[trainer.v6.1] Mip-Splatting cam_centers ({mip_cam_centers.shape[0]} cam) hazirlandi\\")\\n            except Exception as _e:\\n                print(f\\"[trainer.v6.1] Mip cam_centers hata, devre disi: {_e}\\")\\n                self.mip_scale_floor_frac = 0.0\\n\\n        # SV training-index pool — excludes NVS hold-out frames so eval can\\n        # measure true novel views. MV uses mv_test_camera elsewhere; for MV\\n        # the pool is unused (train loop samples cam_id then idx from full T).\\n        if (not is_multiview) and holdout_indices:\\n            _holdout_set = {int(i) for i in holdout_indices if 0 <= int(i) < T}\\n            _train_idx_pool = torch.tensor(\\n                [i for i in range(T) if i not in _holdout_set],\\n                dtype=torch.long,\\n            )\\n            if _train_idx_pool.numel() == 0:\\n                raise ValueError(\\n                    f\\"holdout_indices excluded ALL {T} frames; nothing to train on\\"\\n                )\\n            print(f\\"[trainer.holdout] SV: {len(_holdout_set)}/{T} frames held out from training \\"\\n                  f\\"(pool size={_train_idx_pool.numel()})\\")\\n        else:\\n            _train_idx_pool = None  # sample uniformly from [0, T)\\n\\n        for it in range(1, n_iters + 1):\\n            if cancel_check is not None and cancel_check():\\n                print(f\\"[trainer] cancel requested at iter {it}, stopping cleanly\\")\\n                break\\n            if _train_idx_pool is not None:\\n                idx = int(_train_idx_pool[\\n                    torch.randint(0, _train_idx_pool.numel(), (1,)).item()\\n                ].item())\\n            else:\\n                idx = int(torch.randint(0, T, (1,)).item())\\n            t_norm = idx / max(T - 1, 1)\\n            warmup = self._warmup_factor(it)\\n\\n            # Update lr_deform + lr_fourier per iter (warmup schedule)\\n            self.optimizer.param_groups[DEFORM_GROUP_IDX][\\"lr\\"] = target_lr_deform * warmup\\n            if has_fourier_group:\\n                self.optimizer.param_groups[FOURIER_GROUP_IDX][\\"lr\\"] = target_lr_fourier * warmup\\n\\n            # 4D Quality v6.1 — Madde 11: Adaptive SH degree\\n            if self.sh_progressive_schedule:\\n                active_sh_degree = progressive_sh_degree(it, n_iters, self.gs.sh_degree)\\n            else:\\n                active_sh_degree = self.gs.sh_degree\\n\\n            # --- Phase 2.2: Multi-resolution transition ---\\n            # Perf fix: NO LONGER invalidate the cache on resolution change.\\n            # The cache holds master-resolution tensors (Wm, Hm); we just toggle\\n            # (Ws, Hs) and per-iter downsample on GPU below.\\n            new_le = _resolve_long_edge_for_iter(it)\\n            if new_le is not None:\\n                new_Ws, new_Hs = _resize_to_long_edge(new_le, W0, H0)\\n                if new_Ws != Ws or new_Hs != Hs:\\n                    print(f\\"[trainer.multires] Iter {it}: resolution {Ws}x{Hs} → {new_Ws}x{new_Hs} \\"\\n                          f\\"(long_edge={new_le}); cache PRESERVED at master ({Wm}x{Hm})\\")\\n                    Ws, Hs = new_Ws, new_Hs\\n                    sx, sy = Ws / W0, Hs / H0\\n                    K_scaled = cam_K_orig.clone()\\n                    K_scaled[0, 0] *= sx; K_scaled[0, 2] *= sx\\n                    K_scaled[1, 1] *= sy; K_scaled[1, 2] *= sy\\n                    if is_multiview:\\n                        for c in train_cams:\\n                            K_c = mv_cam_K[c].to(self.device).clone()\\n                            K_c[0, 0] *= sx; K_c[0, 2] *= sx\\n                            K_c[1, 1] *= sy; K_c[1, 2] *= sy\\n                            K_scaled_mv[c] = K_c\\n                    # Flow cache: flow tensors are pre-resized per-iter via F.interpolate\\n                    # already (see flow blocks below), so safe to keep. We invalidate\\n                    # only flow caches that were stored at the OLD training resolution\\n                    # (defensive — they\'re loaded lazily, just clear stale entries).\\n                    # Note: existing flow code does its own per-iter F.interpolate, so\\n                    # we can leave them alone — but to stay consistent with prior\\n                    # behavior (flow magnitudes computed at training resolution), we\\n                    # keep the lazy load-cached-at-load-time approach: just drop them.\\n                    if is_multiview and flow_cached_mv is not None:\\n                        for c in train_cams:\\n                            flow_cached_mv[c] = [None] * T\\n                    if not is_multiview and flow_cached_sv is not None:\\n                        flow_cached_sv = [None] * T\\n\\n            # GPU-side resize helpers — convert cached uint8/fp16 master tensor\\n            # to float32 at current training resolution (Ws, Hs).\\n            def _gt_rgb_from_cache(t_cpu: torch.Tensor) -> torch.Tensor:\\n                \\"\\"\\"Cached uint8 (H, W, 3) → device float32 (Hs, Ws, 3) in [0, 1].\\"\\"\\"\\n                t = t_cpu.to(self.device, non_blocking=True)\\n                if t.dtype == torch.uint8:\\n                    t = t.float() / 255.0\\n                else:\\n                    # Legacy float32 cache (preload_to_ram=False, dtype default)\\n                    t = t.float()\\n                # Resize if not at training resolution.\\n                # cache shape is (H_cache, W_cache, 3). F.interpolate wants NCHW.\\n                Hc, Wc = t.shape[:2]\\n                if Wc != Ws or Hc != Hs:\\n                    t_chw = t.permute(2, 0, 1).unsqueeze(0)  # (1, 3, Hc, Wc)\\n                    t_chw = F.interpolate(\\n                        t_chw, size=(Hs, Ws), mode=\\"bilinear\\",\\n                        align_corners=False, antialias=True,\\n                    )\\n                    t = t_chw.squeeze(0).permute(1, 2, 0).contiguous()\\n                return t\\n\\n            def _gt_depth_from_cache(t_cpu: torch.Tensor) -> torch.Tensor:\\n                \\"\\"\\"Cached fp16/fp32 (H, W) → device float32 (Hs, Ws).\\"\\"\\"\\n                t = t_cpu.to(self.device, non_blocking=True).float()\\n                Hc, Wc = t.shape[:2]\\n                if Wc != Ws or Hc != Hs:\\n                    t_chw = t.unsqueeze(0).unsqueeze(0)  # (1, 1, Hc, Wc)\\n                    t_chw = F.interpolate(\\n                        t_chw, size=(Hs, Ws), mode=\\"bilinear\\",\\n                        align_corners=False,\\n                    )\\n                    t = t_chw.squeeze(0).squeeze(0)\\n                return t\\n\\n            def _gt_mask_from_cache(t_cpu: torch.Tensor) -> torch.Tensor:\\n                \\"\\"\\"Cached uint8/float (H, W) → device float32 (Hs, Ws) in [0, 1].\\"\\"\\"\\n                t = t_cpu.to(self.device, non_blocking=True)\\n                if t.dtype == torch.uint8:\\n                    t = t.float() / 255.0\\n                else:\\n                    t = t.float()\\n                Hc, Wc = t.shape[:2]\\n                if Wc != Ws or Hc != Hs:\\n                    t_chw = t.unsqueeze(0).unsqueeze(0)\\n                    t_chw = F.interpolate(\\n                        t_chw, size=(Hs, Ws), mode=\\"nearest\\",\\n                    )\\n                    t = t_chw.squeeze(0).squeeze(0)\\n                return t\\n\\n            # --- Ground truth load ---\\n            if is_multiview:\\n                # v5.0: random cam selection\\n                cam_id = train_cams[int(torch.randint(0, len(train_cams), (1,)).item())]\\n                if frames_cached_mv[cam_id][idx] is None:\\n                    # Lazy disk read at master resolution (cache survives multires).\\n                    frames_cached_mv[cam_id][idx] = load_frame_tensor(\\n                        Path(mv_frame_paths[cam_id][idx]), (Wm, Hm),\\n                        dtype=(\\"uint8\\" if preload_to_ram else \\"float32\\"),\\n                    )\\n                gt = _gt_rgb_from_cache(frames_cached_mv[cam_id][idx])\\n                # Phase 1.4 — Per-cam depth load (MV path)\\n                gt_depth = None\\n                if use_depth_mv:\\n                    if depth_cached_mv[cam_id][idx] is None:\\n                        # frame_path\'in stem\'inden depth dosyası adı\\n                        frame_stem = Path(mv_frame_paths[cam_id][idx]).stem\\n                        depth_path = depth_mv_dir / cam_id / f\\"{frame_stem}_depth.npy\\"\\n                        if depth_path.exists():\\n                            depth_cached_mv[cam_id][idx] = _load_depth_for_frame(\\n                                depth_mv_dir / cam_id,\\n                                Path(mv_frame_paths[cam_id][idx]),\\n                                (Wm, Hm),\\n                                dtype=(\\"float16\\" if preload_to_ram else \\"float32\\"),\\n                            )\\n                    if depth_cached_mv[cam_id][idx] is not None:\\n                        gt_depth = _gt_depth_from_cache(depth_cached_mv[cam_id][idx])\\n                frame_mask = None\\n                K_active = K_scaled_mv[cam_id]\\n                w2c_active = w2c_mv[cam_id]\\n            else:\\n                if frames_cached[idx] is None:\\n                    frames_cached[idx] = load_frame_tensor(\\n                        Path(frame_paths[idx]), (Wm, Hm),\\n                        dtype=(\\"uint8\\" if preload_to_ram else \\"float32\\"),\\n                    )\\n                gt = _gt_rgb_from_cache(frames_cached[idx])\\n\\n                if use_depth and depth_cached[idx] is None:\\n                    depth_cached[idx] = _load_depth_for_frame(\\n                        depth_dir, frame_paths[idx], (Wm, Hm),\\n                        dtype=(\\"float16\\" if preload_to_ram else \\"float32\\"),\\n                    )\\n                gt_depth = (\\n                    _gt_depth_from_cache(depth_cached[idx])\\n                    if (use_depth and depth_cached[idx] is not None) else None\\n                )\\n\\n                if use_mask and mask_cached[idx] is None:\\n                    mask_cached[idx] = _load_mask_for_frame(\\n                        mask_dir, frame_paths[idx], (Wm, Hm),\\n                        dtype=(\\"uint8\\" if preload_to_ram else \\"float32\\"),\\n                    )\\n                frame_mask = (\\n                    _gt_mask_from_cache(mask_cached[idx])\\n                    if (use_mask and mask_cached[idx] is not None) else None\\n                )\\n\\n                K_active = K_scaled\\n                w2c_active = w2c_list[idx]\\n\\n            # --- Deformation + render ---\\n            # v5.0: Static phase\'de deformation tamamen bypass — MLP init\\n            # weights nonzero olabilir, sadece lr=0 yapmak yetersiz, forward\\n            # pass\'i da kapatmak gerek. Statik gaussian\'lar direkt render edilir.\\n            # T6 perf fix: raw_dpos/raw_dquat/raw_dscale = MLP\'nin pre-clamp output\'u,\\n            # regularizer block\'da yeniden hesaplanmasin diye saklayalim.\\n            raw_dpos = raw_dquat = raw_dscale = None\\n            if it <= getattr(self, \\"_static_phase_iters\\", 0):\\n                d_means = self.gs.means\\n                d_quats = F.normalize(self.gs.quats, dim=-1)\\n                d_scales = self.gs.get_scales\\n            else:\\n                (d_means, d_quats, d_scales), (raw_dpos, raw_dquat, raw_dscale) = \\\\\\n                    self._apply_deformation(t_norm, return_raw=True)\\n                # Phase 2.1 + 2.4 — Static + Background gaussian\'lari deformation\'dan bypass et.\\n                # is_static=True veya is_background=True olanlar undeformed kalir.\\n                bypass_mask = None\\n                if (self.use_static_dynamic_split\\n                        and hasattr(self.gs, \\"is_static\\")):\\n                    bypass_mask = self.gs.is_static.to(d_means.device)\\n                if hasattr(self.gs, \\"is_background\\") and self.gs.is_background.any():\\n                    bg = self.gs.is_background.to(d_means.device)\\n                    bypass_mask = bg if bypass_mask is None else (bypass_mask | bg)\\n                if bypass_mask is not None and bypass_mask.any():\\n                    means_undef = self.gs.means\\n                    quats_undef = F.normalize(self.gs.quats, dim=-1)\\n                    scales_undef = self.gs.get_scales\\n                    d_means = torch.where(bypass_mask.unsqueeze(-1), means_undef, d_means)\\n                    d_quats = torch.where(bypass_mask.unsqueeze(-1), quats_undef, d_quats)\\n                    d_scales = torch.where(bypass_mask.unsqueeze(-1), scales_undef, d_scales)\\n\\n            # 4D Quality v6.1 — Madde 2-B: Mip-Splatting 3D scale floor\\n            # Anti-aliasing approximation — yakin gauss\'larin tek-piksel\'e\\n            # cokmesini engeller, yumusak edge\'ler verir.\\n            if self.mip_scale_floor_frac > 0 and mip_cam_centers is not None:\\n                d_scales = apply_mip_scale_floor(\\n                    d_scales, d_means, mip_cam_centers,\\n                    floor_frac=self.mip_scale_floor_frac,\\n                )\\n\\n            render_out, _alpha, _info = render_view(\\n                means=d_means, quats=d_quats, scales=d_scales,\\n                opacities=self.gs.get_opacities, colors=self.gs.get_colors,\\n                K=K_active, w2c=w2c_active,\\n                width=Ws, height=Hs,\\n                sh_degree=active_sh_degree,\\n                with_depth=((use_depth and not is_multiview) or use_depth_mv),\\n            )\\n            # Phase 1.4 fix: with_depth MV path\'te de True olabilir, RGB extract et\\n            if use_depth or use_depth_mv:\\n                rgb = render_out[..., :3]\\n                rendered_depth = render_out[..., 3]\\n            else:\\n                rgb = render_out\\n                rendered_depth = None\\n\\n            # --- Reconstruction loss ---\\n            pixel_weight = None\\n            if frame_mask is not None and self.lambda_mask_motion > 0:\\n                pixel_weight = 1.0 + self.lambda_mask_motion * frame_mask\\n            loss_recon = compute_recon_loss(rgb, gt, self.lambda_ssim, pixel_weight=pixel_weight)\\n            loss = loss_recon\\n\\n            # Perf v2: defer .item() — store detached tensors during accumulation, sync once at log time.\\n            # Eskisi her iter ~12 cuda sync (her .item() bir host-device sync). Yeni: sadece log_interval\'de bir kere.\\n            will_log = (it % log_interval == 0)\\n            comp_t: dict = {\\"recon\\": loss_recon.detach()}\\n\\n            # --- Phase 1.6: LPIPS perceptual loss ---\\n            # rgb / gt: [H, W, 3] in [0, 1] -> [3, H, W] beklenir\\n            if self.lambda_lpips > 0:\\n                if self._lpips_module is None:\\n                    from .losses_perceptual import LPIPSLoss\\n                    self._lpips_module = LPIPSLoss(net=self.lpips_net)\\n                lpips_warmup = min(1.0, it / max(1, self.lpips_warmup_iters))\\n                pred_chw = rgb.permute(2, 0, 1).contiguous()\\n                gt_chw = gt.permute(2, 0, 1).contiguous()\\n                lpips_val = self._lpips_module(pred_chw, gt_chw)\\n                if torch.isfinite(lpips_val):\\n                    loss = loss + self.lambda_lpips * lpips_warmup * lpips_val\\n                    comp_t[\\"lpips\\"] = lpips_val.detach()\\n\\n            # --- Phase 1.7: Multi-view consistency loss ---\\n            # Aynı t\'de farkli bir cam\'da second render + recon. 2 cam supervision\\n            # ile single-cam overfit baski. ~%20 iter time art.\\n            # Maliyet: ekstra render. Sadece is_multiview ve lambda > 0.\\n            if (is_multiview and self.lambda_multiview_consistency > 0\\n                    and len(train_cams) >= 2):\\n                try:\\n                    # Farkli bir cam sec\\n                    other_cams = [c for c in train_cams if c != cam_id]\\n                    cam_id2 = other_cams[int(torch.randint(0, len(other_cams), (1,)).item())]\\n                    if frames_cached_mv[cam_id2][idx] is None:\\n                        frames_cached_mv[cam_id2][idx] = load_frame_tensor(\\n                            Path(mv_frame_paths[cam_id2][idx]), (Wm, Hm),\\n                            dtype=(\\"uint8\\" if preload_to_ram else \\"float32\\"),\\n                        )\\n                    gt2 = _gt_rgb_from_cache(frames_cached_mv[cam_id2][idx])\\n                    K2 = K_scaled_mv[cam_id2]\\n                    w2c2 = w2c_mv[cam_id2]\\n                    render_out2, _, _ = render_view(\\n                        means=d_means, quats=d_quats, scales=d_scales,\\n                        opacities=self.gs.get_opacities, colors=self.gs.get_colors,\\n                        K=K2, w2c=w2c2,\\n                        width=Ws, height=Hs,\\n                        sh_degree=active_sh_degree,\\n                        with_depth=False,\\n                    )\\n                    rgb2 = render_out2 if render_out2.shape[-1] == 3 else render_out2[..., :3]\\n                    mv_l1 = (rgb2 - gt2).abs().mean()\\n                    if torch.isfinite(mv_l1):\\n                        loss = loss + self.lambda_multiview_consistency * mv_l1\\n                        comp_t[\\"mv_consist\\"] = mv_l1.detach()\\n                except Exception as _e:\\n                    if it < 50:\\n                        print(f\\"  ⚠ mv_consistency failed iter {it}: {_e}\\")\\n\\n            # --- Phase 1.8: RAFT optical flow supervision (multi-view) ---\\n            # Yaklasim A (proxy): t ve t+1 frame\'lerini ayni cam\'da render et,\\n            # RGB diff magnitude\'i RAFT flow magnitude ile L1 ile esitle.\\n            # Gercek 2D motion vector degil — proxy ama motion supervision sinyali iyi.\\n            if (use_flow_mv and self.lambda_flow > 0\\n                    and idx < T - 1\\n                    and it > getattr(self, \\"_static_phase_iters\\", 0)):\\n                try:\\n                    flow_path = flow_mv_dir / cam_id / f\\"forward_{idx:04d}.pt\\"\\n                    if flow_path.exists():\\n                        if flow_cached_mv[cam_id][idx] is None:\\n                            flow_cached_mv[cam_id][idx] = torch.load(\\n                                flow_path, map_location=self.device\\n                            ).float()\\n                        gt_flow = flow_cached_mv[cam_id][idx]  # [2, H_raw, W_raw]\\n                        # Resize flow to training resolution\\n                        gt_flow_resized = F.interpolate(\\n                            gt_flow.unsqueeze(0), size=(Hs, Ws),\\n                            mode=\'bilinear\', align_corners=False,\\n                        ).squeeze(0)\\n                        gt_flow_mag = gt_flow_resized.norm(dim=0)  # [Hs, Ws]\\n                        # Render frame t+1 same cam (extra render)\\n                        t_next_norm = (idx + 1) / max(T - 1, 1)\\n                        d_means_n, d_quats_n, d_scales_n = self._apply_deformation(t_next_norm)\\n                        if (self.use_static_dynamic_split\\n                                and hasattr(self.gs, \\"is_static\\")):\\n                            bypass = self.gs.is_static.to(d_means_n.device)\\n                            if hasattr(self.gs, \\"is_background\\"):\\n                                bypass = bypass | self.gs.is_background.to(d_means_n.device)\\n                            if bypass.any():\\n                                m_und = self.gs.means\\n                                q_und = F.normalize(self.gs.quats, dim=-1)\\n                                s_und = self.gs.get_scales\\n                                d_means_n = torch.where(bypass.unsqueeze(-1), m_und, d_means_n)\\n                                d_quats_n = torch.where(bypass.unsqueeze(-1), q_und, d_quats_n)\\n                                d_scales_n = torch.where(bypass.unsqueeze(-1), s_und, d_scales_n)\\n                        ro_n, _, _ = render_view(\\n                            means=d_means_n, quats=d_quats_n, scales=d_scales_n,\\n                            opacities=self.gs.get_opacities, colors=self.gs.get_colors,\\n                            K=K_active, w2c=w2c_active,\\n                            width=Ws, height=Hs,\\n                            sh_degree=active_sh_degree,\\n                            with_depth=False,\\n                        )\\n                        rgb_n = ro_n if ro_n.shape[-1] == 3 else ro_n[..., :3]\\n                        rendered_diff_mag = (rgb_n - rgb).abs().mean(dim=-1)  # [Hs, Ws]\\n                        # Normalize gt_flow magnitude by max + threshold high-motion regions\\n                        flow_warmup = min(1.0, it / max(1, self.flow_warmup_iters))\\n                        # Loss: rendered diff magnitude high motion\'da yuksek olmali\\n                        gt_norm = gt_flow_mag / (gt_flow_mag.max() + 1e-6)\\n                        diff_norm = rendered_diff_mag / (rendered_diff_mag.max() + 1e-6)\\n                        flow_l1 = (diff_norm - gt_norm).abs().mean().clamp(max=1.0)\\n                        if torch.isfinite(flow_l1):\\n                            loss = loss + self.lambda_flow * flow_warmup * flow_l1\\n                            comp_t[\\"flow\\"] = flow_l1.detach()\\n                except Exception as _e:\\n                    if it < 50:\\n                        print(f\\"  ⚠ flow loss failed iter {it}: {_e}\\")\\n\\n            # --- Phase 1.8 SV: RAFT optical flow supervision (single-view) ---\\n            # MV varyantın single-view portu. flow_dir/forward_*.pt cache\'inden\\n            # gt flow yukle, t+1 frame\'i ayni cam\'da render et, RGB diff vs gt L1.\\n            if (use_flow_sv and self.lambda_flow > 0\\n                    and idx < T - 1\\n                    and it > getattr(self, \\"_static_phase_iters\\", 0)\\n                    and not is_multiview):\\n                try:\\n                    flow_path = flow_dir / f\\"forward_{idx:04d}.pt\\"\\n                    if flow_path.exists():\\n                        if flow_cached_sv[idx] is None:\\n                            flow_cached_sv[idx] = torch.load(\\n                                flow_path, map_location=self.device\\n                            ).float()\\n                        gt_flow = flow_cached_sv[idx]\\n                        gt_flow_resized = F.interpolate(\\n                            gt_flow.unsqueeze(0), size=(Hs, Ws),\\n                            mode=\'bilinear\', align_corners=False,\\n                        ).squeeze(0)\\n                        gt_flow_mag = gt_flow_resized.norm(dim=0)\\n                        # Render frame t+1 same cam pose (single-view next frame\'in pose\'u list\'te)\\n                        t_next_norm = (idx + 1) / max(T - 1, 1)\\n                        d_means_n, d_quats_n, d_scales_n = self._apply_deformation(t_next_norm)\\n                        if (self.use_static_dynamic_split\\n                                and hasattr(self.gs, \\"is_static\\")):\\n                            bypass = self.gs.is_static.to(d_means_n.device)\\n                            if hasattr(self.gs, \\"is_background\\"):\\n                                bypass = bypass | self.gs.is_background.to(d_means_n.device)\\n                            if bypass.any():\\n                                m_und = self.gs.means\\n                                q_und = F.normalize(self.gs.quats, dim=-1)\\n                                s_und = self.gs.get_scales\\n                                d_means_n = torch.where(bypass.unsqueeze(-1), m_und, d_means_n)\\n                                d_quats_n = torch.where(bypass.unsqueeze(-1), q_und, d_quats_n)\\n                                d_scales_n = torch.where(bypass.unsqueeze(-1), s_und, d_scales_n)\\n                        # Single-view next frame için K + w2c (cam_K_orig + w2c_list[idx+1])\\n                        ro_n, _, _ = render_view(\\n                            means=d_means_n, quats=d_quats_n, scales=d_scales_n,\\n                            opacities=self.gs.get_opacities, colors=self.gs.get_colors,\\n                            K=K_scaled, w2c=w2c_list[idx + 1],\\n                            width=Ws, height=Hs,\\n                            sh_degree=active_sh_degree,\\n                            with_depth=False,\\n                        )\\n                        rgb_n = ro_n if ro_n.shape[-1] == 3 else ro_n[..., :3]\\n                        rendered_diff_mag = (rgb_n - rgb).abs().mean(dim=-1)\\n                        flow_warmup = min(1.0, it / max(1, self.flow_warmup_iters))\\n                        gt_norm = gt_flow_mag / (gt_flow_mag.max() + 1e-6)\\n                        diff_norm = rendered_diff_mag / (rendered_diff_mag.max() + 1e-6)\\n                        flow_l1 = (diff_norm - gt_norm).abs().mean().clamp(max=1.0)\\n                        if torch.isfinite(flow_l1):\\n                            loss = loss + self.lambda_flow * flow_warmup * flow_l1\\n                            comp_t[\\"flow\\"] = flow_l1.detach()\\n                except Exception as _e:\\n                    if it < 50:\\n                        print(f\\"  ⚠ flow_sv loss failed iter {it}: {_e}\\")\\n\\n            # --- Depth consistency (log-space, warmup-gated, clamped) ---\\n            # Log-space L1 scale-invariant ve outlier\'a karşı dayanıklı.\\n            # Warmup gate: ilk 100 iter statik GS otursun, sonra depth devreye.\\n            # Clamp(2.0): tek frame outlier\'ı tüm run\'ı batırmasın.\\n            if (use_depth or use_depth_mv) and gt_depth is not None and rendered_depth is not None:\\n                valid = (gt_depth > 0.01) & (rendered_depth > 0.01)\\n                # Edit refit: inpainted pixels have no reliable GT depth —\\n                # exclude them from depth supervision.\\n                if edit_mask_stack is not None:\\n                    inpaint_m = edit_mask_stack[idx].to(self.device).bool()\\n                    # Resize mask to match depth map spatial dims if needed\\n                    if inpaint_m.shape != valid.shape:\\n                        inpaint_m = torch.nn.functional.interpolate(\\n                            inpaint_m.float().unsqueeze(0).unsqueeze(0),\\n                            size=valid.shape[-2:],\\n                            mode=\\"nearest\\",\\n                        ).squeeze(0).squeeze(0).bool()\\n                    valid = valid & (~inpaint_m)\\n                if valid.sum() > 100:\\n                    gt_v = gt_depth[valid].clamp(min=0.01, max=100.0)\\n                    r_v  = rendered_depth[valid].clamp(min=0.01, max=100.0)\\n                    log_gt = torch.log(gt_v)\\n                    log_r  = torch.log(r_v)\\n                    with torch.no_grad():\\n                        shift = log_r.median() - log_gt.median()\\n                    log_gt_aligned = log_gt + shift\\n                    depth_l1 = (log_r - log_gt_aligned).abs().mean().clamp(max=2.0)\\n                    loss = loss + self.lambda_depth * warmup * depth_l1\\n                    comp_t[\\"depth\\"] = depth_l1.detach()\\n\\n            # --- Track loss (CoTracker 3D-anchored projection) ---\\n            if track_data is not None and idx < track_data[\\"T_tr\\"]:\\n                anchors = track_data[\\"anchors_3d\\"]                  # (N, 3)\\n                gt_tracks = track_data[\\"tracks_2d\\"][idx]            # (N, 2) frame-res pixel\\n                vis = track_data[\\"visibility\\"][idx]                 # (N,) bool\\n\\n                K_scaled_frame = K_scaled  # training res ile kıyaslayacağız; tracks\'i scale edelim\\n                gt_tracks_scaled = gt_tracks.clone()\\n                gt_tracks_scaled[..., 0] *= sx\\n                gt_tracks_scaled[..., 1] *= sy\\n\\n                # Sample K tracks for speed\\n                # Perf v2: use sum without .item() — bool() in if catches it but only when track loss enabled.\\n                # Default config has lambda_track > 0, so this still costs one sync; acceptable.\\n                N_visible = int(vis.sum().item())\\n                if N_visible >= 16:\\n                    sample_size = min(self.track_sample_k, N_visible)\\n                    vis_idx = torch.nonzero(vis, as_tuple=False).squeeze(-1)\\n                    sel = vis_idx[torch.randperm(vis_idx.shape[0], device=self.device)[:sample_size]]\\n                    anchors_sel = anchors[sel]\\n                    gt_uv = gt_tracks_scaled[sel]\\n\\n                    # Apply deformation at t to anchors (static base + delta)\\n                    # v3.6.1: HEM MLP hem nearest-gaussian fourier trajectory.\\n                    # Bu sayede track loss gradient\'i hem MLP\'ye hem Fourier\'a akar.\\n                    # Önceden sadece MLP\'ye akıyordu → Fourier motion rastgele öğreniliyordu.\\n                    d_anchors_mlp, _, _ = self.deform(anchors_sel, t_norm, self.scene_extent)\\n                    if (self.deform_pos_mode != \\"mlp\\" and\\n                            self.gs.fourier_pos_coeffs is not None):\\n                        # KNN: her anchor için en yakın gaussian\\n                        with torch.no_grad():\\n                            dists = torch.cdist(anchors_sel, self.gs.means)  # (A, N)\\n                            nearest_idx = dists.argmin(dim=-1)                # (A,)\\n                        nearest_coeffs = self.gs.fourier_pos_coeffs[nearest_idx]  # (A, K, 2, 3)\\n                        d_anchors_fourier = decode_fourier_trajectory(nearest_coeffs, t_norm)\\n                        d_anchors_pos = d_anchors_mlp + d_anchors_fourier\\n                    else:\\n                        d_anchors_pos = d_anchors_mlp\\n                    deformed_anchors = anchors_sel + d_anchors_pos\\n\\n                    # Project to training image space\\n                    uv_pred, valid_proj = _project_world_to_pixel(\\n                        deformed_anchors, K_scaled_frame, w2c_list[idx],\\n                    )\\n                    if valid_proj.any():\\n                        diff = (uv_pred[valid_proj] - gt_uv[valid_proj]).abs().mean()\\n                        # Normalize by image diagonal so lambda ~ O(1)\\n                        diag = (Ws ** 2 + Hs ** 2) ** 0.5\\n                        track_l = diff / diag\\n                        loss = loss + self.lambda_track * warmup * track_l\\n                        comp_t[\\"track\\"] = track_l.detach()\\n\\n            # --- Motion regularizers (warmup\'la scale) ---\\n            if warmup > 0 and raw_dpos is not None and (\\n                self.lambda_deform_reg > 0\\n                or self.lambda_smoothness > 0\\n                or self.lambda_rigidity > 0\\n                or self.lambda_accel > 0\\n            ):\\n                # T6 perf fix: _apply_deformation\'dan gelen RAW MLP output\'u kullan,\\n                # ayni t_norm icin ikinci kez self.deform(...) cagrisi yapma.\\n                # Onceden bu blok her iter\'de duplicate forward yapiyordu.\\n                dpos, dquat, dscale = raw_dpos, raw_dquat, raw_dscale\\n                if self.lambda_deform_reg > 0:\\n                    reg = dpos.pow(2).mean() + dquat.pow(2).mean() + dscale.pow(2).mean()\\n                    loss = loss + self.lambda_deform_reg * warmup * reg\\n                    comp_t[\\"deform_reg\\"] = reg.detach()\\n                if self.lambda_smoothness > 0:\\n                    t2 = min(1.0, t_norm + self.smoothness_dt)\\n                    if t2 != t_norm:\\n                        dpos2, _, _ = self.deform(self.gs.means, t2, self.scene_extent)\\n                        smooth = (dpos - dpos2).pow(2).mean()\\n                        loss = loss + self.lambda_smoothness * warmup * smooth\\n                        comp_t[\\"smooth\\"] = smooth.detach()\\n                # 4D Quality v6.1 — Madde 6: 2nd-order acceleration regularizer\\n                # D(t-1) - 2D(t) + D(t+1) magnitude. Slow-motion render\'da titremeyi azalt.\\n                # Sample t-dt and t+dt; eger sinirlarda ise yumusakca skip.\\n                if self.lambda_accel > 0:\\n                    t_minus = max(0.0, t_norm - self.smoothness_dt)\\n                    t_plus  = min(1.0, t_norm + self.smoothness_dt)\\n                    if t_minus < t_norm < t_plus:\\n                        dpos_m, _, _ = self.deform(self.gs.means, t_minus, self.scene_extent)\\n                        dpos_p, _, _ = self.deform(self.gs.means, t_plus,  self.scene_extent)\\n                        accel = (dpos_m - 2.0 * dpos + dpos_p).pow(2).mean()\\n                        loss = loss + self.lambda_accel * warmup * accel\\n                        comp_t[\\"accel\\"] = accel.detach()\\n                N = self.gs.num_points\\n                K = min(self.rigidity_sample_k, N)\\n                if self.lambda_rigidity > 0 and K >= 8:\\n                    sample_idx = torch.randint(0, N, (K,), device=self.device)\\n                    base_pts = self.gs.means[sample_idx]\\n                    deformed_pts = d_means[sample_idx]\\n                    dist_base = torch.cdist(base_pts, base_pts)\\n                    dist_def  = torch.cdist(deformed_pts, deformed_pts)\\n                    rigid = (dist_base - dist_def).abs().mean()\\n                    loss = loss + self.lambda_rigidity * warmup * rigid\\n                    comp_t[\\"rigid\\"] = rigid.detach()\\n\\n            # --- Scale regularizer v3.1 — ASIMETRIK HINGE ---\\n            # v3\'te symmetric log_scale² formülü tüm scale\'leri 1\'e itip\\n            # homogenization yaratmıştı (arka plan gaussian\'ları küçüldü, detaylar büyüdü).\\n            # v3.1: sadece scene_extent\'in %5\'inden büyük scale\'leri cezalandır.\\n            # Küçük/orta gaussian\'lar serbest, sadece outlier bloat\'a müdahale.\\n            if self.lambda_scale > 0:\\n                # Threshold: log(scene_extent × 0.05). Örn scene_extent=48 → log(2.4)=0.875\\n                # log_scale > threshold olan kısım cezalandırılır, altı = 0 gradient.\\n                log_threshold = math.log(max(self.scene_extent * 0.05, 1e-3))\\n                excess = (self.gs.scales - log_threshold).clamp(min=0)\\n                scale_reg = excess.pow(2).mean()\\n                loss = loss + self.lambda_scale * scale_reg\\n                comp_t[\\"scale\\"] = scale_reg.detach()\\n\\n            # --- Anisotropy regularizer v3.8 — STREAK / NEEDLE GAUSSIAN FIX ---\\n            # banana_demo Ultra\'da scene\'in etrafında uzun parlak çizgiler oluştu.\\n            # Sebep: gaussian (0.05, 2, 0.05) gibi extreme aspect ratio (max/min=40).\\n            # Magnitude reg (yukarıdaki scale_reg) sadece büyük scale\'leri cezalandırır,\\n            # küçük-iğne\'ye dokunmaz. Aspect ratio reg eksikti.\\n            #\\n            # Quadratic hinge: ratio > threshold (default 5) ise (ratio - threshold)² ceza.\\n            # threshold = 5 normal 3DGS aspect ratio; 10+ kuyruklu yıldız.\\n            if self.lambda_aniso > 0 and self.gs.num_points > 0:\\n                # gs.scales is log-space → linear scales pozitif\\n                lin_scales = torch.exp(self.gs.scales)  # (N, 3)\\n                sc_max = lin_scales.max(dim=-1).values  # (N,)\\n                sc_min = lin_scales.min(dim=-1).values.clamp(min=1e-6)  # (N,)\\n                aniso_ratio = sc_max / sc_min  # (N,) — ≥ 1\\n                aniso_excess = (aniso_ratio - self.aniso_threshold).clamp(min=0)\\n                aniso_reg = aniso_excess.pow(2).mean()\\n                loss = loss + self.lambda_aniso * warmup * aniso_reg\\n                comp_t[\\"aniso\\"] = aniso_reg.detach()\\n\\n            # --- Fourier trajectory regularizer (v3.6 / Yol C) ---\\n            # v5.0 (Adim 2): L2 → L1 sparsity. Lasso effect — kucuk coefficient\'lari\\n            # TAM sifira ceker, sadece gercekten motion isteyen gaussian\'lar\\n            # nonzero kalir. Bu implicit static-dynamic separation:\\n            #   - fourier_pos_coeffs ≈ 0 → gaussian static (motion yok)\\n            #   - fourier_pos_coeffs > 0 → gaussian dynamic (motion var)\\n            # Multi-view\'da kritik: 100k+ gaussian × 8 K = 800k+ DOF, hepsi\\n            # birden hareket etmesin diye L1 baskisi gerekli.\\n            # Frekans agirligi k^2 → k (L1 zaten daha guclu, k^2 fazla agir).\\n            if (self.lambda_fourier_reg > 0 and\\n                    self.gs.fourier_pos_coeffs is not None and\\n                    self.deform_pos_mode != \\"mlp\\"):\\n                K = self.gs.fourier_K\\n                # Frekans ağırlığı: k (lineer). Yüksek k\'lar biraz daha cezalı.\\n                freq_weights = torch.arange(1, K + 1, device=self.gs.fourier_pos_coeffs.device,\\n                                            dtype=self.gs.fourier_pos_coeffs.dtype)\\n                # L1 magnitude: |coeff|\\n                mag = self.gs.fourier_pos_coeffs.abs()  # (N, K, 2, 3)\\n                # Sum over sin/cos + xyz, mean over N, weighted sum over K\\n                per_freq_l1 = mag.sum(dim=(0, 2, 3)) / max(self.gs.num_points, 1)  # (K,)\\n                fourier_reg = (per_freq_l1 * freq_weights).sum()\\n                # L1 magnitude fark farkli L2\'den. Initial (Adim 2) 5× idi —\\n                # cok agresif, dynamic motion\'i da bastirdi. v5.0.1: 2× multiplier\\n                # ile recon ve sparsity arasinda denge. Static gaussian\'lar\\n                # cogunlukla sifir, dynamic gaussian\'lar yeterli motion freedom.\\n                loss = loss + self.lambda_fourier_reg * 2.0 * fourier_reg\\n                comp_t[\\"fourier\\"] = fourier_reg.detach()\\n\\n            # --- Fourier SPATIAL smoothness (v3.6.2) ---\\n            # Komşu gaussian\'lar benzer trajectory\'ye sahip olsun. Motion diffuse et.\\n            # Aksi halde sadece track anchor\'larına yakın gaussian hareket ediyor,\\n            # geri kalan statik — chickchicken\'da %92 gaussian durgundu.\\n            # Her iter random K=128 gaussian sample + 1-NN üzerinden smoothness.\\n            #\\n            # v5.0 (Adim 2): Multi-view\'da BU ISTENMEZ. L1 sparsity ile statik\\n            # gaussian\'lar tam sifir kalsin; spatial smoothness motion bulasimi\\n            # yaratir, statik bolgeler dynamic\'e dogru kayar. Multi-view tespiti\\n            # icin: train()\'e is_multiview parametresi tasimak yerine, daha\\n            # genel bir flag kullanilsa daha iyi olur. Su an kapali tutuyoruz.\\n            if (False and  # v5.0: spatial smoothness disabled for multi-view static preservation\\n                    self.lambda_fourier_reg > 0 and\\n                    self.gs.fourier_pos_coeffs is not None and\\n                    self.deform_pos_mode != \\"mlp\\" and\\n                    self.gs.num_points > 16):\\n                Ns = min(128, self.gs.num_points)\\n                sample_idx = torch.randint(0, self.gs.num_points, (Ns,), device=self.device)\\n                sampled_means = self.gs.means[sample_idx].detach()  # (Ns, 3)\\n                # Her sample için en yakın komşu (kendisi hariç)\\n                with torch.no_grad():\\n                    dists = torch.cdist(sampled_means, self.gs.means)  # (Ns, N)\\n                    # En yakın ilk 2\'yi al (0 = kendisi eğer sample gaussian ise)\\n                    _, nearest_k = torch.topk(dists, k=2, largest=False, dim=-1)\\n                    # Ikinci en yakını (ilki kendisi olabilir)\\n                    neighbor_idx = nearest_k[:, 1]  # (Ns,)\\n                sampled_coeffs = self.gs.fourier_pos_coeffs[sample_idx]      # (Ns, K, 2, 3)\\n                neighbor_coeffs = self.gs.fourier_pos_coeffs[neighbor_idx]   # (Ns, K, 2, 3)\\n                spatial_smooth = (sampled_coeffs - neighbor_coeffs).pow(2).mean()\\n                # Weight: reg\'in %50\'si kadar — motion\'u öldürmeden smooth\\n                loss = loss + self.lambda_fourier_reg * 0.5 * spatial_smooth\\n                # comp\'ta ayrı tutmayalım, fourier ile toplu logla\\n\\n            # --- NaN/Inf guard (pre-backward) ---\\n            # Loss patlıyorsa backward yapma, bu iter\'i atla. 30k iter\'de\\n            # tek patlayan iter bile means/scales/quats\'u NaN\'layıp modeli\\n            # geri dönülmez kılar — cutlemon_full\'de oldu.\\n            # v3.4: consecutive skip counter — 10 ardışık skip olursa abort\\n            #\\n            # Perf v2: bu check her iter SHARED-stream sync gerektiriyor (bool() on\\n            # cuda scalar). Kritik bir guard oldugu icin tutuluyor — bir iter atlamak\\n            # bile divergent eder, sonraki iter\'lerde model patlar. Sync maliyetinin\\n            # asagidaki grad/param NaN check\'lerini log_interval\'a kaydirip net iter\\n            # baslarinda 1 sync hedefliyoruz.\\n            if not hasattr(self, \\"_consecutive_skip\\"):\\n                self._consecutive_skip = 0\\n            if not torch.isfinite(loss):\\n                self._consecutive_skip += 1\\n                print(f\\"  ⚠ iter {it}: loss={loss.item()} non-finite → skip (#{self._consecutive_skip})\\")\\n                if run_logger is not None:\\n                    try:\\n                        run_logger.warn(\\n                            f\\"iter {it} skipped: loss non-finite\\",\\n                            consecutive=self._consecutive_skip,\\n                        )\\n                    except Exception:\\n                        pass\\n                if self._consecutive_skip >= 10:\\n                    if run_logger is not None:\\n                        try:\\n                            run_logger.log_event(\\"training:aborted\\", iter=it, reason=\\"10+ consecutive non-finite loss\\")\\n                        except Exception:\\n                            pass\\n                    raise RuntimeError(\\n                        f\\"Training aborted at iter {it}: 10+ consecutive iters with non-finite loss. \\"\\n                        f\\"Model state diverged (check scales/params). \\"\\n                        f\\"Try: lower lambda_track, raise lambda_smoothness, or reduce lr_deform.\\"\\n                    )\\n                self.optimizer.zero_grad(set_to_none=False)\\n                continue\\n            self._consecutive_skip = 0\\n\\n            self.optimizer.zero_grad(set_to_none=False)\\n            loss.backward()\\n\\n            # --- Gradient clipping (stability) ---\\n            # Tüm parametre grup\'larından global norm ile clip — divergence\'ı engeller.\\n            # 1.0 conservative, GS için 10.0 daha uygun (büyük LR + densification).\\n            torch.nn.utils.clip_grad_norm_(\\n                [p for g in self.optimizer.param_groups for p in g[\\"params\\"]],\\n                max_norm=10.0,\\n            )\\n\\n            # --- Post-backward NaN guard on params ---\\n            # Perf v2: per-iter sync (16+ params x .all()) ciddi maliyet. Divergence\\n            # nadir ve onceki pre-backward isfinite(loss) guard\'ı yakaliyor; bu yuzden\\n            # log_interval cadence yeterli (her ~50 iter bir kontrol). Ilk 200 iter\'de\\n            # daha sik (init asamasi divergence olasiligi yuksek).\\n            run_grad_check = will_log or it < 200\\n            grad_nan = False\\n            if run_grad_check:\\n                grad_nan = any(\\n                    p.grad is not None and not torch.isfinite(p.grad).all()\\n                    for g in self.optimizer.param_groups for p in g[\\"params\\"]\\n                )\\n            if grad_nan:\\n                print(f\\"  ⚠ iter {it}: non-finite gradient → skip step\\")\\n                self.optimizer.zero_grad(set_to_none=False)\\n                continue\\n\\n            self.density.accumulate(self.gs)\\n            # Edit refit: zero gradients on out-of-zone (frozen) Gaussians so\\n            # only Gaussians inside the affected zone are updated this step.\\n            _fm = getattr(self.gs, \\"_freeze_mask\\", None)\\n            if _fm is not None:\\n                _fm_dev = _fm.to(self.device)\\n                for _attr in (\\"means\\", \\"scales\\", \\"quats\\", \\"opacities\\", \\"sh_dc\\", \\"sh_rest\\"):\\n                    _p = getattr(self.gs, _attr, None)\\n                    if isinstance(_p, torch.nn.Parameter) and _p.grad is not None:\\n                        _p.grad[_fm_dev] = 0.0\\n                if self.gs.fourier_pos_coeffs is not None:\\n                    _p = self.gs.fourier_pos_coeffs\\n                    if isinstance(_p, torch.nn.Parameter) and _p.grad is not None:\\n                        _p.grad[_fm_dev] = 0.0\\n            self.optimizer.step()\\n            # Phase 2.3 — Camera pose refinement step (warmup sonrasi aktif)\\n            if (self._cam_refine_optimizer is not None\\n                    and it >= self.cam_refine_start_iter):\\n                # 4D Quality v6.1 — Madde 12: cam params icin ayri grad norm clip.\\n                # Cam parametreleri gauss\'larla aynı global clip\'e tabi tutmak yerine\\n                # kendi norm budget\'ini kullanir. Large-scale scene\'de drift\'i azalt.\\n                if self.cam_grad_clip_norm > 0:\\n                    cam_params = [p for g in self._cam_refine_optimizer.param_groups\\n                                  for p in g[\\"params\\"] if p.grad is not None]\\n                    if cam_params:\\n                        torch.nn.utils.clip_grad_norm_(\\n                            cam_params, max_norm=self.cam_grad_clip_norm,\\n                        )\\n                self._cam_refine_optimizer.step()\\n                self._cam_refine_optimizer.zero_grad(set_to_none=True)\\n\\n            # --- Post-step param NaN/Inf check — v3.4 expanded ---\\n            # means + scales + quats hepsi kontrol ediliyor çünkü scales inf\'e\\n            # gitmesi training\'i sessizce bozuyordu (chickchicken_v3_3\'te log_scale=70+,\\n            # exp()=inf, her sonraki iter loss non-finite → skipped → silent garbage).\\n            #\\n            # Perf v2: 3 ayri .all() reduction + bool() per iter ~3 cuda sync. Asagidaki\\n            # log_interval gating sadece her ~50 iter bir kontrol; arada NaN olusursa\\n            # bir sonraki iter\'in pre-backward isfinite(loss) guard\'ı yakalar ve skip\\n            # eder, training diverge etmez (clamp\'ler yardim eder).\\n            if run_grad_check:\\n                means_bad = not torch.isfinite(self.gs.means).all()\\n                scales_bad = not torch.isfinite(self.gs.scales).all()\\n                quats_bad = not torch.isfinite(self.gs.quats).all()\\n                if means_bad or scales_bad or quats_bad:\\n                    reason = []\\n                    if means_bad: reason.append(\\"means\\")\\n                    if scales_bad: reason.append(\\"scales\\")\\n                    if quats_bad: reason.append(\\"quats\\")\\n                    print(f\\"  ⚠⚠ iter {it}: {\'+\'.join(reason)} NaN/Inf AFTER step — training aborted\\")\\n                    if run_logger is not None:\\n                        try:\\n                            run_logger.log_event(\\"training:aborted\\", iter=it, reason=f\\"{\'+\'.join(reason)} non-finite post-step\\")\\n                        except Exception:\\n                            pass\\n                    raise RuntimeError(\\n                        f\\"Training diverged at iter {it}: {\'+\'.join(reason)} non-finite. \\"\\n                        f\\"Last good checkpoint: run scripts/export_from_ckpt.py --scene <scene>\\"\\n                    )\\n\\n            # v3.7.4: Hard clamp on log_scale — TIGHTER bound.\\n            # Önce scene_extent idi (gaussian sahnenin tamamı kadar olabiliyordu),\\n            # banana_high\'ta max_scale=108=scene_extent → streak/overlap.\\n            # Şimdi: scene_extent × 0.05 = sahnenin %5\'i max.\\n            with torch.no_grad():\\n                max_log_scale = math.log(max(self.scene_extent * 0.05, 1e-3))\\n                self.gs.scales.data.clamp_(max=max_log_scale)\\n\\n                # v3.6.1: Hard clamp on Fourier coefficients.\\n                # Her katsayı büyüklüğü scene_extent × 0.05 = max 5% motion contribution.\\n                # Bir gaussian maksimum motion = K × max_coeff = 8 × 0.05 × scene = 40% scene.\\n                # Reg çalışsa bile katsayılar kontrolsüz büyüyordu (0.12 → 6.38 in 500 iter)\\n                # — bu clamp hard limit.\\n                if self.gs.fourier_pos_coeffs is not None:\\n                    max_coeff = self.scene_extent * 0.03  # 3% of scene per coefficient\\n                    self.gs.fourier_pos_coeffs.data.clamp_(min=-max_coeff, max=max_coeff)\\n\\n            # --- Density control ---\\n            # T4 perf fix: optimizer\'i density.step\'e geciriyoruz; densitiy ops\\n            # (clone/split/prune) Adam state\'i (exp_avg, exp_avg_sq) parametre\\n            # mutation\'ı ile lockstep gunceller. Eskiden self._build_optimizer()\\n            # ile her density step optimizer\'i sifirdan kuruyorduk → 200+ kez\\n            # Adam momentum siliniyordu → gerek convergence yavasligi, gerekse\\n            # kalite kaybi. torch.cuda.empty_cache() de kaldirildi (allocator\\n            # thrash + sync stall sebebiyle stuck\'lara yol aciyordu).\\n            if (self.density_start_iter <= it < self.density_end_iter\\n                    and it % self.density_interval == 0):\\n                stats = self.density.step(\\n                    self.gs,\\n                    optimizer=self.optimizer,\\n                    dynamic_densify_scale=self.dynamic_densify_scale,\\n                )\\n                if it % log_interval == 0:\\n                    print(f\\"  ↳ density: clone={stats[\'cloned\']} split={stats[\'split\']} \\"\\n                          f\\"prune={stats[\'pruned\']} | {stats[\'before\']} → {stats[\'after\']}\\")\\n                if run_logger is not None:\\n                    try:\\n                        run_logger.log_event(\\n                            \\"density\\",\\n                            iter=it,\\n                            cloned=stats[\\"cloned\\"],\\n                            split=stats[\\"split\\"],\\n                            pruned=stats[\\"pruned\\"],\\n                            before=stats[\\"before\\"],\\n                            after=stats[\\"after\\"],\\n                        )\\n                    except Exception:\\n                        pass\\n\\n            # --- Opacity reset (INRIA 3DGS trick, v3) ---\\n            # Her N iter\'de tüm gaussian\'ların opacity\'sini low bir değere reset et\\n            # → recon loss bu opacity\'leri tekrar >0.005 eşiğine yükseltemeyenleri\\n            # bir sonraki density step\'te pruneyecek (outlier cleanup mekanizması).\\n            # Sadece density window içinde yap.\\n            if (self.opacity_reset_interval > 0\\n                    and self.density_start_iter <= it < self.density_end_iter\\n                    and it % self.opacity_reset_interval == 0\\n                    and it > self.density_start_iter):\\n                with torch.no_grad():\\n                    # Mevcut opacity\'lerin min(current, 0.01 logit ≈ sigmoid^-1(0.01))\\n                    # Logit(0.01) ≈ -4.595. Daha yüksek olanları buna indir.\\n                    reset_logit = float(np.log(0.01 / (1 - 0.01)))  # ≈ -4.595\\n                    new_op = torch.minimum(\\n                        self.gs.opacities,\\n                        torch.full_like(self.gs.opacities, reset_logit),\\n                    )\\n                    self.gs.opacities.data.copy_(new_op)\\n                # Optimizer state\'te opacity momentum sıfırla (yeni değer başlıyor)\\n                for g in self.optimizer.param_groups:\\n                    for p in g[\\"params\\"]:\\n                        if p is self.gs.opacities:\\n                            state = self.optimizer.state.get(p, None)\\n                            if state:\\n                                if \\"exp_avg\\" in state: state[\\"exp_avg\\"].zero_()\\n                                if \\"exp_avg_sq\\" in state: state[\\"exp_avg_sq\\"].zero_()\\n                print(f\\"  ↳ opacity reset @ iter {it}\\")\\n                if run_logger is not None:\\n                    try:\\n                        run_logger.log_event(\\"opacity_reset\\", iter=it)\\n                    except Exception:\\n                        pass\\n\\n            # --- Log ---\\n            if it % log_interval == 0:\\n                with torch.no_grad():\\n                    p = psnr(rgb, gt)\\n                    # v3.7.1 METRIC FIX: Önceden Δpos = raw MLP dpos (pre-clamp, Fourier\'sız).\\n                    # Artık _apply_deformation\'un gerçek dpos çıktısını ölçüyor\\n                    # (MLP clamp + Fourier + total clamp dahil). Bu, PLY export\'taki\\n                    # gerçek motion\'la tutarlı. Önceden metrics yanıltıcıydı.\\n                    d_means_dbg, _, _ = self._apply_deformation(t_norm)\\n                    dpos_applied = d_means_dbg - self.gs.means\\n                    dpos_mean = dpos_applied.abs().mean().item()\\n                    dpos_max = dpos_applied.abs().max().item()\\n                # Perf v2: comp_t\'deki tensor\'leri tek seferde sync edip float dict\'e cevir.\\n                # Eskiden her iter ~12 sync; simdi sadece log_interval\'de bir.\\n                comp = {k: float(v.item()) if torch.is_tensor(v) else float(v)\\n                        for k, v in comp_t.items()}\\n                # Default\'lar (downstream get(...,0.0) calismasi icin):\\n                for _k in (\\"recon\\", \\"depth\\", \\"track\\", \\"deform_reg\\", \\"smooth\\", \\"rigid\\",\\n                           \\"scale\\", \\"fourier\\", \\"accel\\", \\"lpips\\", \\"mv_consist\\", \\"flow\\",\\n                           \\"aniso\\"):\\n                    comp.setdefault(_k, 0.0)\\n                history[\\"loss\\"].append(loss.item())\\n                history[\\"psnr\\"].append(p)\\n                history[\\"n_pts\\"].append(self.gs.num_points)\\n                elapsed = time.time() - t0\\n                ips = it / max(elapsed, 1e-6)\\n\\n                comp_str = \\" \\".join(\\n                    f\\"{k}={v:.4f}\\" for k, v in comp.items()\\n                    if (k == \\"recon\\") or (v > 1e-5)\\n                )\\n                print(f\\"[{it:>6}/{n_iters}] loss={loss.item():.4f} [{comp_str}] \\"\\n                      f\\"psnr={p:.2f} N={self.gs.num_points:,} \\"\\n                      f\\"Δpos={dpos_mean:.4f}/{dpos_max:.4f} wu={warmup:.2f} | {ips:.1f} it/s\\")\\n\\n                if progress_callback is not None:\\n                    try:\\n                        progress_callback(it, n_iters, float(loss.item()),\\n                                          float(p), int(self.gs.num_points))\\n                    except Exception as _e:\\n                        print(f\\"  ⚠ progress_callback exception: {_e}\\")\\n\\n                # Run logger — metrics.jsonl\'a yaz\\n                if run_logger is not None:\\n                    try:\\n                        run_logger.log_metric(\\n                            iter=it, n_iters=n_iters,\\n                            loss=float(loss.item()),\\n                            psnr=float(p),\\n                            n_points=int(self.gs.num_points),\\n                            dpos_mean=float(dpos_mean),\\n                            dpos_max=float(dpos_max),\\n                            warmup=float(warmup),\\n                            it_per_sec=float(ips),\\n                            recon=float(comp.get(\\"recon\\", 0.0)),\\n                            depth=float(comp.get(\\"depth\\", 0.0)),\\n                            track=float(comp.get(\\"track\\", 0.0)),\\n                            deform_reg=float(comp.get(\\"deform_reg\\", 0.0)),\\n                            smooth=float(comp.get(\\"smooth\\", 0.0)),\\n                            rigid=float(comp.get(\\"rigid\\", 0.0)),\\n                            scale=float(comp.get(\\"scale\\", 0.0)),\\n                            fourier=float(comp.get(\\"fourier\\", 0.0)),\\n                        )\\n                    except Exception as _e:\\n                        print(f\\"  ⚠ run_logger exception: {_e}\\")\\n\\n            if ckpt_dir is not None and it % ckpt_interval == 0:\\n                self._save_checkpoint(ckpt_dir, it)\\n\\n        if ckpt_dir is not None:\\n            self._save_checkpoint(ckpt_dir, n_iters, final=True)\\n\\n        return history\\n\\n    def _save_checkpoint(self, ckpt_dir: Path, it: int, final: bool = False) -> None:\\n        ckpt_dir = Path(ckpt_dir); ckpt_dir.mkdir(parents=True, exist_ok=True)\\n        suffix = \\"final\\" if final else f\\"{it:06d}\\"\\n        p = ckpt_dir / f\\"ckpt_{suffix}.pt\\"\\n        torch.save({\\n            \\"iter\\": it,\\n            \\"gs\\":   self.gs.state_for_save(),\\n            \\"deform\\": self.deform.state_dict(),\\n            \\"scene_extent\\": self.scene_extent,\\n            \\"sh_degree\\": self.gs.sh_degree,\\n        }, p)\\n        print(f\\"  ✓ checkpoint → {p}\\")\\n", "app/backend/preprocess/parse_colmap.py": "\\"\\"\\"Parse COLMAP text models: cameras.txt, images.txt, and points3D.txt.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\n\\n_CAMERA_PARAM_COUNTS = {\\n    \\"PINHOLE\\": 4,\\n    \\"SIMPLE_PINHOLE\\": 3,\\n    \\"SIMPLE_RADIAL\\": 4,\\n    \\"OPENCV\\": 8,\\n}\\n_IMAGE_COUNT_RE = re.compile(\\n    r\\"^#\\\\s*Number of images:\\\\s*(\\\\d+)(?:\\\\s*,|$)\\",\\n)\\n\\n\\ndef quat_to_rotmat(qw: float, qx: float, qy: float, qz: float) -> np.ndarray:\\n    \\"\\"\\"Convert a COLMAP quaternion (w, x, y, z) to a 3x3 rotation matrix.\\"\\"\\"\\n    norm = np.sqrt(qw * qw + qx * qx + qy * qy + qz * qz)\\n    if norm < 1e-12:\\n        return np.eye(3)\\n    qw, qx, qy, qz = qw / norm, qx / norm, qy / norm, qz / norm\\n    return np.array(\\n        [\\n            [\\n                1 - 2 * (qy * qy + qz * qz),\\n                2 * (qx * qy - qw * qz),\\n                2 * (qx * qz + qw * qy),\\n            ],\\n            [\\n                2 * (qx * qy + qw * qz),\\n                1 - 2 * (qx * qx + qz * qz),\\n                2 * (qy * qz - qw * qx),\\n            ],\\n            [\\n                2 * (qx * qz - qw * qy),\\n                2 * (qy * qz + qw * qx),\\n                1 - 2 * (qx * qx + qy * qy),\\n            ],\\n        ],\\n        dtype=np.float64,\\n    )\\n\\n\\ndef _find_sparse_dir(colmap_dir: Path) -> Path:\\n    \\"\\"\\"Find the legacy sparse sub-model that has converted camera text.\\"\\"\\"\\n    sparse_root = colmap_dir / \\"sparse\\"\\n    if sparse_root.exists() and sparse_root.is_dir():\\n        candidates = [\\n            directory\\n            for directory in sparse_root.iterdir()\\n            if directory.is_dir() and (directory / \\"cameras.txt\\").exists()\\n        ]\\n        if candidates:\\n\\n            def _size(directory: Path) -> int:\\n                total = 0\\n                for name in (\\"cameras.bin\\", \\"images.bin\\", \\"points3D.bin\\"):\\n                    binary_file = directory / name\\n                    if binary_file.exists():\\n                        total += binary_file.stat().st_size\\n                return total\\n\\n            candidates.sort(key=_size, reverse=True)\\n            return candidates[0]\\n\\n    if (colmap_dir / \\"cameras.txt\\").exists():\\n        return colmap_dir\\n\\n    raise FileNotFoundError(\\n        f\\"cameras.txt not found under {colmap_dir}; tried sparse/* and the root\\"\\n    )\\n\\n\\ndef _parse_camera_row(line: str, line_number: int) -> tuple[int, dict[str, object]]:\\n    parts = line.split()\\n    if len(parts) < 4:\\n        raise ValueError(f\\"Malformed camera row at line {line_number}\\")\\n\\n    model = parts[1]\\n    if model not in _CAMERA_PARAM_COUNTS:\\n        raise NotImplementedError(f\\"Camera model unsupported: {model}\\")\\n\\n    expected_params = _CAMERA_PARAM_COUNTS[model]\\n    if len(parts) != 4 + expected_params:\\n        raise ValueError(\\n            f\\"Malformed camera row at line {line_number}: {model} expects \\"\\n            f\\"{expected_params} parameters\\"\\n        )\\n    try:\\n        camera_id = int(parts[0])\\n        width = int(parts[2])\\n        height = int(parts[3])\\n        params = tuple(float(value) for value in parts[4:])\\n    except (OverflowError, ValueError) as exc:\\n        raise ValueError(f\\"Malformed camera row at line {line_number}\\") from exc\\n\\n    if model == \\"PINHOLE\\":\\n        fx, fy, cx, cy = params\\n    elif model in {\\"SIMPLE_PINHOLE\\", \\"SIMPLE_RADIAL\\"}:\\n        focal, cx, cy = params[:3]\\n        fx = fy = focal\\n    else:\\n        fx, fy, cx, cy = params[:4]\\n\\n    intrinsics = np.array(\\n        [[fx, 0.0, cx], [0.0, fy, cy], [0.0, 0.0, 1.0]],\\n        dtype=np.float64,\\n    )\\n    return camera_id, {\\n        \\"K\\": intrinsics,\\n        \\"width\\": width,\\n        \\"height\\": height,\\n    }\\n\\n\\ndef _parse_cameras_text(model_dir: Path) -> dict[int, dict[str, object]]:\\n    cameras: dict[int, dict[str, object]] = {}\\n    with (model_dir / \\"cameras.txt\\").open(encoding=\\"utf-8\\") as handle:\\n        for line_number, raw_line in enumerate(handle, start=1):\\n            line = raw_line.strip()\\n            if not line or line.startswith(\\"#\\"):\\n                continue\\n            camera_id, camera = _parse_camera_row(line, line_number)\\n            if camera_id in cameras:\\n                raise ValueError(f\\"Duplicate camera ID: {camera_id}\\")\\n            cameras[camera_id] = camera\\n    if not cameras:\\n        raise RuntimeError(\\"cameras.txt contains no camera records\\")\\n    return cameras\\n\\n\\ndef _validate_points2d_row(row: str, line_number: int) -> None:\\n    if not row:\\n        return\\n    parts = row.split()\\n    if len(parts) % 3 != 0:\\n        raise ValueError(f\\"Malformed POINTS2D row at line {line_number}\\")\\n    try:\\n        for offset in range(0, len(parts), 3):\\n            float(parts[offset])\\n            float(parts[offset + 1])\\n            int(parts[offset + 2])\\n    except (OverflowError, ValueError) as exc:\\n        raise ValueError(f\\"Malformed POINTS2D row at line {line_number}\\") from exc\\n\\n\\ndef _image_record_lines(images_file: Path) -> list[tuple[int, str]]:\\n    records: list[tuple[int, str]] = []\\n    pending_header: tuple[int, str] | None = None\\n    declared_count: int | None = None\\n    with images_file.open(encoding=\\"utf-8\\") as handle:\\n        for line_number, raw_line in enumerate(handle, start=1):\\n            line = raw_line.strip()\\n            if line.startswith(\\"#\\"):\\n                count_match = _IMAGE_COUNT_RE.match(line)\\n                if count_match is not None:\\n                    parsed_count = int(count_match.group(1))\\n                    if declared_count is not None and parsed_count != declared_count:\\n                        raise ValueError(\\n                            \\"images.txt has conflicting declared image counts\\"\\n                        )\\n                    declared_count = parsed_count\\n                continue\\n            if pending_header is None:\\n                if not line:\\n                    continue\\n                pending_header = (line_number, line)\\n                continue\\n\\n            _validate_points2d_row(line, line_number)\\n            records.append(pending_header)\\n            pending_header = None\\n\\n    if pending_header is not None:\\n        line_number, _ = pending_header\\n        raise ValueError(\\n            f\\"Missing POINTS2D record after image header at line {line_number}\\"\\n        )\\n    if declared_count is None:\\n        raise ValueError(\\"images.txt is missing its declared image count\\")\\n    if len(records) != declared_count:\\n        raise ValueError(\\n            \\"POINTS2D/image count mismatch: \\"\\n            f\\"declared {declared_count}, parsed {len(records)}\\"\\n        )\\n    return records\\n\\n\\ndef _parse_image_header(\\n    header: str,\\n    line_number: int,\\n) -> tuple[int, tuple[float, ...], tuple[float, ...], int, str]:\\n    parts = header.split(maxsplit=9)\\n    if len(parts) != 10 or not parts[9]:\\n        raise ValueError(f\\"Malformed image header at line {line_number}\\")\\n    try:\\n        image_id = int(parts[0])\\n        quaternion = tuple(float(value) for value in parts[1:5])\\n        translation = tuple(float(value) for value in parts[5:8])\\n        camera_id = int(parts[8])\\n    except (OverflowError, ValueError) as exc:\\n        raise ValueError(f\\"Malformed image header at line {line_number}\\") from exc\\n\\n    quaternion_array = np.asarray(quaternion, dtype=np.float64)\\n    quaternion_norm = float(np.linalg.norm(quaternion_array))\\n    if not np.all(np.isfinite(quaternion_array)) or not np.isfinite(quaternion_norm):\\n        raise ValueError(f\\"Invalid quaternion at image line {line_number}\\")\\n    if quaternion_norm <= 1e-12:\\n        raise ValueError(f\\"Near-zero quaternion at image line {line_number}\\")\\n    return image_id, quaternion, translation, camera_id, parts[9]\\n\\n\\ndef _parse_camera_and_image_text(model_dir: Path) -> dict[str, dict]:\\n    cameras = _parse_cameras_text(model_dir)\\n    parsed_images: list[tuple[int, tuple[float, ...], tuple[float, ...], int, str]] = []\\n    image_ids: set[int] = set()\\n    image_names: set[str] = set()\\n    for line_number, header in _image_record_lines(model_dir / \\"images.txt\\"):\\n        parsed = _parse_image_header(header, line_number)\\n        image_id, _, _, camera_id, name = parsed\\n        if image_id in image_ids:\\n            raise ValueError(f\\"Duplicate image ID: {image_id}\\")\\n        if name in image_names:\\n            raise ValueError(f\\"Duplicate image name: {name}\\")\\n        if camera_id not in cameras:\\n            raise ValueError(\\n                f\\"Image {image_id} references missing camera ID {camera_id}\\"\\n            )\\n        image_ids.add(image_id)\\n        image_names.add(name)\\n        parsed_images.append(parsed)\\n\\n    images: dict[str, dict] = {}\\n    for image_id, quaternion, translation, camera_id, name in parsed_images:\\n        rotation = quat_to_rotmat(*quaternion)\\n        translation_array = np.asarray(translation, dtype=np.float64)\\n        world_to_camera = np.eye(4, dtype=np.float64)\\n        world_to_camera[:3, :3] = rotation\\n        world_to_camera[:3, 3] = translation_array\\n        camera = cameras[camera_id]\\n        images[name] = {\\n            \\"image_id\\": image_id,\\n            \\"K\\": camera[\\"K\\"],\\n            \\"R\\": rotation,\\n            \\"t\\": translation_array,\\n            \\"w2c\\": world_to_camera,\\n            \\"width\\": camera[\\"width\\"],\\n            \\"height\\": camera[\\"height\\"],\\n        }\\n\\n    if not images:\\n        raise RuntimeError(\\"images.txt contains no image records\\")\\n    return images\\n\\n\\ndef parse_cameras_from_model(model_dir: str | Path) -> dict[str, dict]:\\n    \\"\\"\\"Read cameras and registered images from one exact COLMAP text model.\\"\\"\\"\\n    model = Path(model_dir)\\n    cameras_file = model / \\"cameras.txt\\"\\n    images_file = model / \\"images.txt\\"\\n    if (\\n        cameras_file.is_symlink()\\n        or images_file.is_symlink()\\n        or not cameras_file.is_file()\\n        or not images_file.is_file()\\n    ):\\n        raise FileNotFoundError(f\\"Incomplete COLMAP text model: {model}\\")\\n    return _parse_camera_and_image_text(model)\\n\\n\\ndef parse_cameras(colmap_dir: str | Path) -> dict[str, dict]:\\n    \\"\\"\\"Read cameras using the legacy sparse-model discovery policy.\\"\\"\\"\\n    cameras = parse_cameras_from_model(_find_sparse_dir(Path(colmap_dir)))\\n    print(f\\"[ok] {len(cameras)} camera poses loaded\\")\\n    return cameras\\n\\n\\ndef _parse_points3d_text(\\n    points_file: Path,\\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\\n    xyz_rows: list[list[float]] = []\\n    rgb_rows: list[list[int]] = []\\n    track_lengths: list[int] = []\\n    reprojection_errors: list[float] = []\\n    point_ids: set[int] = set()\\n\\n    with points_file.open(encoding=\\"utf-8\\") as handle:\\n        for line_number, raw_line in enumerate(handle, start=1):\\n            line = raw_line.strip()\\n            if not line or line.startswith(\\"#\\"):\\n                continue\\n            parts = line.split()\\n            if len(parts) < 8:\\n                raise ValueError(f\\"Malformed point row at line {line_number}\\")\\n            if (len(parts) - 8) % 2 != 0:\\n                raise ValueError(f\\"Malformed track data at point line {line_number}\\")\\n            try:\\n                point_id = int(parts[0])\\n                xyz = [float(value) for value in parts[1:4]]\\n                rgb = [int(value) for value in parts[4:7]]\\n                reprojection_error = float(parts[7])\\n                for offset in range(8, len(parts), 2):\\n                    int(parts[offset])\\n                    int(parts[offset + 1])\\n            except (OverflowError, ValueError) as exc:\\n                raise ValueError(f\\"Malformed point row at line {line_number}\\") from exc\\n\\n            if point_id in point_ids:\\n                raise ValueError(f\\"Duplicate point ID: {point_id}\\")\\n            if any(channel < 0 or channel > 255 for channel in rgb):\\n                raise ValueError(f\\"Malformed point RGB at line {line_number}\\")\\n            point_ids.add(point_id)\\n            xyz_rows.append(xyz)\\n            rgb_rows.append(rgb)\\n            track_lengths.append((len(parts) - 8) // 2)\\n            reprojection_errors.append(reprojection_error)\\n\\n    if not xyz_rows:\\n        raise RuntimeError(f\\"{points_file.name} is empty\\")\\n    return (\\n        np.asarray(xyz_rows, dtype=np.float32),\\n        np.asarray(rgb_rows, dtype=np.uint8),\\n        np.asarray(track_lengths, dtype=np.int32),\\n        np.asarray(reprojection_errors, dtype=np.float32),\\n    )\\n\\n\\ndef load_points3d_from_model(\\n    model_dir: str | Path,\\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\\n    \\"\\"\\"Read points and confidence metrics from one exact COLMAP text model.\\"\\"\\"\\n    points_file = Path(model_dir) / \\"points3D.txt\\"\\n    if points_file.is_symlink() or not points_file.is_file():\\n        raise FileNotFoundError(f\\"points3D.txt not found: {points_file}\\")\\n    return _parse_points3d_text(points_file)\\n\\n\\ndef load_points3d(colmap_dir: str | Path) -> tuple[np.ndarray, np.ndarray]:\\n    \\"\\"\\"Load sparse points using the legacy sparse-model discovery policy.\\"\\"\\"\\n    xyz, rgb, _, _ = load_points3d_from_model(_find_sparse_dir(Path(colmap_dir)))\\n    print(f\\"[ok] {len(xyz)} sparse 3D points loaded\\")\\n    return xyz, rgb\\n\\n\\ndef load_points3d_with_confidence(\\n    colmap_dir: str | Path,\\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\\n    \\"\\"\\"Load sparse points plus track lengths and reprojection errors.\\"\\"\\"\\n    xyz, rgb, track_lengths, reprojection_errors = load_points3d_from_model(\\n        _find_sparse_dir(Path(colmap_dir))\\n    )\\n    print(\\n        f\\"[ok] {len(xyz)} sparse points + confidence loaded \\"\\n        f\\"(track median={np.median(track_lengths):.0f}, \\"\\n        f\\"max={track_lengths.max()}; \\"\\n        f\\"error median={np.median(reprojection_errors):.2f}, \\"\\n        f\\"max={reprojection_errors.max():.2f})\\"\\n    )\\n    return xyz, rgb, track_lengths, reprojection_errors\\n\\n\\ndef scene_extent(xyz: np.ndarray) -> float:\\n    \\"\\"\\"Return the radial extent of a point cloud around its centroid.\\"\\"\\"\\n    centroid = xyz.mean(axis=0, keepdims=True)\\n    return float(np.linalg.norm(xyz - centroid, axis=1).max())\\n\\n\\nif __name__ == \\"__main__\\":\\n    import argparse\\n\\n    parser = argparse.ArgumentParser(description=\\"Parse COLMAP text output\\")\\n    parser.add_argument(\\"colmap_dir\\", type=str)\\n    arguments = parser.parse_args()\\n\\n    parsed_cameras = parse_cameras(arguments.colmap_dir)\\n    parsed_xyz, parsed_rgb = load_points3d(arguments.colmap_dir)\\n    first_camera = next(iter(parsed_cameras.values()))\\n    print(f\\"\\\\nExample camera intrinsics:\\\\n{first_camera[\'K\']}\\")\\n    print(f\\"Scene radial extent: {scene_extent(parsed_xyz):.3f}\\")\\n", "app/backend/export/to_splat.py": "\\"\\"\\"Faz 6 — Eğitilmiş 4DGS modelini her timestamp için .ply olarak export et.\\n\\nÇıktı formatı, INRIA 3DGS .ply formatına uyumlu:\\n  vertex props:\\n    x, y, z, nx, ny, nz,\\n    f_dc_0..f_dc_2, f_rest_0..f_rest_(K*3-1),\\n    opacity, scale_0..scale_2, rot_0..rot_3\\n\\"\\"\\"\\nfrom __future__ import annotations\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom pathlib import Path\\nfrom typing import List\\n\\nfrom ..model.gaussian_model import GaussianModel\\nfrom ..model.deformation import DeformationField, decode_fourier_trajectory\\n\\n\\ndef write_ply(\\n    means: np.ndarray,        # (N, 3) float\\n    log_scales: np.ndarray,   # (N, 3) — log-space (3DGS .ply formatı log saklar)\\n    quats: np.ndarray,        # (N, 4) (w, x, y, z), normalize edilmiş\\n    opacities_logit: np.ndarray,  # (N,) — logit-space (3DGS formatı logit saklar)\\n    sh_dc: np.ndarray,        # (N, 3) — DC bileşeni\\n    sh_rest: np.ndarray,      # (N, K, 3) — yüksek frekans\\n    output_path: str | Path,\\n) -> Path:\\n    \\"\\"\\"3DGS-uyumlu .ply yaz.\\"\\"\\"\\n    from plyfile import PlyData, PlyElement\\n\\n    output_path = Path(output_path)\\n    output_path.parent.mkdir(parents=True, exist_ok=True)\\n\\n    N = means.shape[0]\\n    K = sh_rest.shape[1]\\n    sh_rest_flat = sh_rest.transpose(0, 2, 1).reshape(N, -1)  # (N, 3*K)\\n\\n    dtype_full = (\\n        [(p, \\"f4\\") for p in (\\"x\\", \\"y\\", \\"z\\", \\"nx\\", \\"ny\\", \\"nz\\")]\\n        + [(f\\"f_dc_{i}\\", \\"f4\\") for i in range(3)]\\n        + [(f\\"f_rest_{i}\\", \\"f4\\") for i in range(3 * K)]\\n        + [(\\"opacity\\", \\"f4\\")]\\n        + [(f\\"scale_{i}\\", \\"f4\\") for i in range(3)]\\n        + [(f\\"rot_{i}\\", \\"f4\\") for i in range(4)]\\n    )\\n    arr = np.empty(N, dtype=dtype_full)\\n    arr[\\"x\\"], arr[\\"y\\"], arr[\\"z\\"] = means[:, 0], means[:, 1], means[:, 2]\\n    arr[\\"nx\\"] = arr[\\"ny\\"] = arr[\\"nz\\"] = 0.0\\n    for i in range(3):\\n        arr[f\\"f_dc_{i}\\"] = sh_dc[:, i]\\n    for i in range(3 * K):\\n        arr[f\\"f_rest_{i}\\"] = sh_rest_flat[:, i]\\n    arr[\\"opacity\\"] = opacities_logit\\n    for i in range(3):\\n        arr[f\\"scale_{i}\\"] = log_scales[:, i]\\n    for i in range(4):\\n        arr[f\\"rot_{i}\\"] = quats[:, i]\\n\\n    PlyData([PlyElement.describe(arr, \\"vertex\\")]).write(str(output_path))\\n    return output_path\\n\\n\\n@torch.no_grad()\\ndef export_to_ply(\\n    gs: GaussianModel,\\n    deform: DeformationField | None,\\n    output_dir: str | Path,\\n    num_timestamps: int = 60,\\n    scene_extent: float = 1.0,\\n    device: str = \\"cuda\\",\\n) -> List[Path]:\\n    \\"\\"\\"\\n    Her zaman adımı için deformed Gaussian\'ları .ply olarak yaz.\\n\\n    Args:\\n        gs: Eğitilmiş GaussianModel\\n        deform: DeformationField — None ise sadece statik tek frame export edilir\\n        output_dir: Çıktı klasörü\\n        num_timestamps: Kaç zaman noktası (örn. 60 → 0.0, 1/59, 2/59, ..., 1.0)\\n        scene_extent: Deformation field\'ın means\'i normalize ettiği değer\\n\\n    Returns:\\n        Yazılan .ply dosyalarının listesi.\\n    \\"\\"\\"\\n    output_dir = Path(output_dir)\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n\\n    gs = gs.to(device).eval()\\n    if deform is not None:\\n        deform = deform.to(device).eval()\\n\\n    timestamps = [0.0] if deform is None else [\\n        t / max(num_timestamps - 1, 1) for t in range(num_timestamps)\\n    ]\\n\\n    written: List[Path] = []\\n    for t_idx, t in enumerate(timestamps):\\n        if deform is not None:\\n            dpos_mlp, dquat, dscale = deform(gs.means, t, scene_extent)\\n            # v3.5 safety: dscale clamp (trainer\'da zaten clamp\'lanıyor ama\\n            # eski checkpoint\'lerde uygulanmamış olabilir, export burada da yapsın)\\n            dscale = dscale.clamp(min=-2.0, max=2.0)\\n            # v3.6.2: dpos_mlp clamp\\n            mlp_cap = scene_extent * 0.1\\n            dpos_mlp = dpos_mlp.clamp(min=-mlp_cap, max=mlp_cap)\\n\\n            # v3.6 / Yol C: Fourier trajectory ekle (varsa)\\n            if getattr(gs, \\"fourier_pos_coeffs\\", None) is not None:\\n                dpos_fourier = decode_fourier_trajectory(gs.fourier_pos_coeffs, t)\\n                dpos = dpos_mlp + dpos_fourier\\n            else:\\n                dpos = dpos_mlp\\n            # v3.6.2: total dpos clamp\\n            total_cap = scene_extent * 0.2\\n            dpos = dpos.clamp(min=-total_cap, max=total_cap)\\n\\n            d_means       = (gs.means + dpos)\\n            d_log_scales  = (gs.scales + dscale)         # log-space toplam\\n            d_quats       = F.normalize(gs.quats + dquat, dim=-1)\\n            # v3.7.4: TIGHTER scale clamp — scene_extent × 0.05 (önceden tamamı idi).\\n            # Outlier streak/overlap önleme.\\n            max_log = torch.log(torch.tensor(max(scene_extent * 0.05, 1e-3), device=d_log_scales.device))\\n            d_log_scales = d_log_scales.clamp(max=float(max_log))\\n        else:\\n            d_means       = gs.means\\n            d_log_scales  = gs.scales\\n            d_quats       = F.normalize(gs.quats, dim=-1)\\n\\n        out_path = output_dir / f\\"frame_{t_idx:04d}.ply\\"\\n        write_ply(\\n            means          = d_means.detach().cpu().numpy(),\\n            log_scales     = d_log_scales.detach().cpu().numpy(),\\n            quats          = d_quats.detach().cpu().numpy(),\\n            opacities_logit= gs.opacities.detach().cpu().numpy().squeeze(-1),\\n            sh_dc          = gs.sh_dc.detach().cpu().numpy().squeeze(1),\\n            sh_rest        = gs.sh_rest.detach().cpu().numpy(),\\n            output_path    = out_path,\\n        )\\n        written.append(out_path)\\n    print(f\\"✓ {len(written)} frame export edildi → {output_dir}\\")\\n    return written\\n\\n\\ndef load_checkpoint_and_export(\\n    ckpt_path: str | Path,\\n    output_dir: str | Path,\\n    num_timestamps: int = 60,\\n    device: str = \\"cuda\\",\\n) -> List[Path]:\\n    \\"\\"\\"Checkpoint dosyasından modeli yükle ve export et.\\"\\"\\"\\n    ckpt = torch.load(ckpt_path, map_location=device)\\n    gs = GaussianModel.from_checkpoint(ckpt[\\"gs\\"])\\n    deform = None\\n    if \\"deform\\" in ckpt:\\n        # Deformation field varsayılan boyutlarla rebuild edilir;\\n        # checkpoint state_dict ile gerçek boyutlar yüklenir\\n        from ..model.deformation import DeformationField\\n        # Resolution & feat_dim\'i state_dict\'ten çıkar\\n        plane_keys = [k for k in ckpt[\\"deform\\"] if k.startswith(\\"hexplane.planes.\\")]\\n        if plane_keys:\\n            sample_key = sorted(plane_keys)[0]\\n            shape = ckpt[\\"deform\\"][sample_key].shape  # (1, feat_dim, R, R)\\n            feat_dim, R = shape[1], shape[2]\\n        else:\\n            feat_dim, R = 32, 64\\n        # MLP genişliği\\n        mlp_w = ckpt[\\"deform\\"].get(\\"mlp.0.weight\\", torch.zeros(256, 1)).shape[0]\\n        deform = DeformationField(resolution=R, feat_dim=feat_dim, mlp_width=mlp_w)\\n        deform.load_state_dict(ckpt[\\"deform\\"])\\n\\n    return export_to_ply(\\n        gs, deform, output_dir,\\n        num_timestamps=num_timestamps,\\n        scene_extent=float(ckpt.get(\\"scene_extent\\", 1.0)),\\n        device=device,\\n    )\\n\\n\\nif __name__ == \\"__main__\\":\\n    import argparse\\n    p = argparse.ArgumentParser(description=\\"Checkpoint → .ply export\\")\\n    p.add_argument(\\"ckpt\\", type=str)\\n    p.add_argument(\\"output_dir\\", type=str)\\n    p.add_argument(\\"--num-timestamps\\", type=int, default=60)\\n    p.add_argument(\\"--device\\", default=\\"cuda\\")\\n    args = p.parse_args()\\n\\n    load_checkpoint_and_export(args.ckpt, args.output_dir,\\n                               args.num_timestamps, args.device)\\n", "app/backend/__init__.py": "", "app/backend/model/__init__.py": "", "app/backend/preprocess/__init__.py": "", "app/backend/export/__init__.py": "", "lib/splat_restorer/__init__.py": "\\"\\"\\"Splat render restorer pilot (Difix3D-style). See README.md.\\"\\"\\"\\n", "lib/splat_restorer/baseline.py": "\\"\\"\\"Cheap baselines: identity, unsharp mask, and a small residual U-Net.\\n\\nIf the diffusion restorer cannot beat the small U-Net trained on the same\\npairs, its pretrained image prior is not earning its cost.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport math\\nimport random\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\nfrom .data import PairDataset, collect_items, split_holdout\\nfrom .workspace import Workspace, atomic_copy, atomic_write_json\\n\\n\\ndef unsharp(img: torch.Tensor, amount: float = 0.6, sigma: float = 1.2) -> torch.Tensor:\\n    \\"\\"\\"img: (B, 3, H, W) in [0, 1].\\"\\"\\"\\n    k = int(2 * math.ceil(3 * sigma) + 1)\\n    x = torch.arange(k, device=img.device, dtype=img.dtype) - k // 2\\n    g = torch.exp(-(x**2) / (2 * sigma**2))\\n    g = g / g.sum()\\n    blur = F.conv2d(F.pad(img, (k // 2,) * 4, mode=\\"reflect\\"), g.view(1, 1, 1, k).repeat(3, 1, 1, 1), groups=3)\\n    blur = F.conv2d(blur, g.view(1, 1, k, 1).repeat(3, 1, 1, 1), groups=3)\\n    return (img + amount * (img - blur)).clamp(0, 1)\\n\\n\\nclass _Block(nn.Module):\\n    def __init__(self, c_in, c_out):\\n        super().__init__()\\n        self.net = nn.Sequential(\\n            nn.Conv2d(c_in, c_out, 3, padding=1), nn.GELU(), nn.Conv2d(c_out, c_out, 3, padding=1), nn.GELU()\\n        )\\n        self.skip = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()\\n\\n    def forward(self, x):\\n        return self.net(x) + self.skip(x)\\n\\n\\nclass SmallUNet(nn.Module):\\n    \\"\\"\\"~8M-parameter residual U-Net; optional reference image as extra input.\\"\\"\\"\\n\\n    def __init__(self, use_ref: bool = True, width: int = 48):\\n        super().__init__()\\n        c_in = 6 if use_ref else 3\\n        w = width\\n        self.use_ref = use_ref\\n        self.inc = _Block(c_in, w)\\n        self.d1 = _Block(w, 2 * w)\\n        self.d2 = _Block(2 * w, 4 * w)\\n        self.d3 = _Block(4 * w, 8 * w)\\n        self.u2 = _Block(8 * w + 4 * w, 4 * w)\\n        self.u1 = _Block(4 * w + 2 * w, 2 * w)\\n        self.u0 = _Block(2 * w + w, w)\\n        self.out = nn.Conv2d(w, 3, 3, padding=1)\\n        nn.init.zeros_(self.out.weight)\\n        nn.init.zeros_(self.out.bias)\\n\\n    def forward(self, deg, ref=None):\\n        x = torch.cat([deg, ref], 1) if self.use_ref else deg\\n        e0 = self.inc(x)\\n        e1 = self.d1(F.avg_pool2d(e0, 2))\\n        e2 = self.d2(F.avg_pool2d(e1, 2))\\n        e3 = self.d3(F.avg_pool2d(e2, 2))\\n        up = lambda t, ref_t: F.interpolate(t, size=ref_t.shape[-2:], mode=\\"bilinear\\", align_corners=False)\\n        d2 = self.u2(torch.cat([up(e3, e2), e2], 1))\\n        d1 = self.u1(torch.cat([up(d2, e1), e1], 1))\\n        d0 = self.u0(torch.cat([up(d1, e0), e0], 1))\\n        return (deg + self.out(d0)).clamp(0, 1)\\n\\n    @torch.no_grad()\\n    def restore(self, deg, ref=None):\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n            return self.forward(deg, ref).float().clamp(0, 1)\\n\\n\\n@dataclass\\nclass BaselineConfig:\\n    run_name: str\\n    scenes: list[str]\\n    use_ref: bool = True\\n    steps: int = 20_000\\n    batch_size: int = 16\\n    lr: float = 3e-4\\n    crop: int = 256\\n    lambda_lpips: float = 0.2\\n    num_workers: int = 8\\n    seed: int = 0\\n\\n\\ndef train_baseline(ws: Workspace, cfg: BaselineConfig, pairs_root: Path, device=\\"cuda\\", log=print) -> Path:\\n    import lpips\\n\\n    run_dir = ws.runs_dir / cfg.run_name\\n    run_dir.mkdir(parents=True, exist_ok=True)\\n    final_path = run_dir / \\"model_final.pt\\"\\n    if final_path.exists():\\n        log(f\\"[{cfg.run_name}] already finished -> {final_path}\\")\\n        return final_path\\n    atomic_write_json(run_dir / \\"config.json\\", asdict(cfg))\\n    random.seed(cfg.seed)\\n    torch.manual_seed(cfg.seed)\\n\\n    items = collect_items(pairs_root, cfg.scenes)\\n    train_items, _ = split_holdout(items, frac=0.02, seed=cfg.seed)\\n    loader = torch.utils.data.DataLoader(\\n        PairDataset(train_items, train=True, crop=cfg.crop), batch_size=cfg.batch_size, shuffle=True,\\n        num_workers=cfg.num_workers, drop_last=True, pin_memory=True, persistent_workers=cfg.num_workers > 0,\\n    )\\n    net = SmallUNet(use_ref=cfg.use_ref).to(device)\\n    opt = torch.optim.AdamW(net.parameters(), lr=cfg.lr, weight_decay=1e-4)\\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg.steps)\\n    net_lpips = lpips.LPIPS(net=\\"vgg\\", verbose=False).to(device).eval().requires_grad_(False)\\n    log(f\\"[{cfg.run_name}] {sum(p.numel() for p in net.parameters()) / 1e6:.1f}M params, {len(train_items)} pairs\\")\\n\\n    log_f = open(run_dir / \\"log.csv\\", \\"w\\", newline=\\"\\")\\n    writer = csv.writer(log_f)\\n    writer.writerow([\\"step\\", \\"loss\\", \\"sec_per_step\\"])\\n    step, t_last = 0, time.time()\\n    while step < cfg.steps:\\n        for batch in loader:\\n            deg, gt, ref = (batch[k].to(device, non_blocking=True) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n            with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n                out = net(deg, ref)\\n                loss = (out.float() - gt).abs().mean()\\n                loss = loss + cfg.lambda_lpips * net_lpips(out.float() * 2 - 1, gt * 2 - 1).float().mean()\\n            opt.zero_grad(set_to_none=True)\\n            loss.backward()\\n            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)\\n            opt.step()\\n            sched.step()\\n            step += 1\\n            if step % 200 == 0:\\n                dt = (time.time() - t_last) / 200\\n                t_last = time.time()\\n                writer.writerow([step, f\\"{loss.item():.5f}\\", f\\"{dt:.3f}\\"])\\n                log_f.flush()\\n                if step % 2000 == 0:\\n                    log(f\\"[{cfg.run_name}] step {step}/{cfg.steps} loss {loss.item():.4f}\\")\\n            if step >= cfg.steps:\\n                break\\n    log_f.close()\\n    tmp = ws.local / f\\"{cfg.run_name}_final.pt\\"\\n    torch.save({\\"state_dict\\": net.state_dict(), \\"use_ref\\": cfg.use_ref}, tmp)\\n    atomic_copy(tmp, final_path)\\n    atomic_write_json(run_dir / \\"DONE.json\\", {\\"steps\\": step})\\n    log(f\\"[{cfg.run_name}] finished -> {final_path}\\")\\n    return final_path\\n\\n\\ndef load_baseline(path: Path, device=\\"cuda\\") -> SmallUNet:\\n    sd = torch.load(path, map_location=\\"cpu\\", weights_only=False)\\n    net = SmallUNet(use_ref=sd[\\"use_ref\\"]).to(device)\\n    net.load_state_dict(sd[\\"state_dict\\"])\\n    return net.eval()\\n", "lib/splat_restorer/colmap_io.py": "\\"\\"\\"Minimal COLMAP sparse-model reader (binary and text).\\n\\nOnly what the pair generator needs: cameras, registered images (pose + name)\\nand the 3D points used to initialise the Gaussians.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport struct\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\n# model_id -> (name, number of params)\\nCAMERA_MODELS = {\\n    0: (\\"SIMPLE_PINHOLE\\", 3),\\n    1: (\\"PINHOLE\\", 4),\\n    2: (\\"SIMPLE_RADIAL\\", 4),\\n    3: (\\"RADIAL\\", 5),\\n    4: (\\"OPENCV\\", 8),\\n    5: (\\"OPENCV_FISHEYE\\", 8),\\n    6: (\\"FULL_OPENCV\\", 12),\\n    7: (\\"FOV\\", 5),\\n    8: (\\"SIMPLE_RADIAL_FISHEYE\\", 4),\\n    9: (\\"RADIAL_FISHEYE\\", 5),\\n    10: (\\"THIN_PRISM_FISHEYE\\", 12),\\n}\\n_NAME_TO_COUNT = {name: count for name, count in CAMERA_MODELS.values()}\\n\\n\\n@dataclass\\nclass Camera:\\n    id: int\\n    model: str\\n    width: int\\n    height: int\\n    params: np.ndarray\\n\\n    def intrinsics(self) -> tuple[float, float, float, float]:\\n        \\"\\"\\"Return (fx, fy, cx, cy); distortion terms are ignored.\\"\\"\\"\\n        p = self.params\\n        if self.model in (\\"SIMPLE_PINHOLE\\", \\"SIMPLE_RADIAL\\", \\"RADIAL\\", \\"SIMPLE_RADIAL_FISHEYE\\", \\"RADIAL_FISHEYE\\", \\"FOV\\"):\\n            return float(p[0]), float(p[0]), float(p[1]), float(p[2])\\n        return float(p[0]), float(p[1]), float(p[2]), float(p[3])\\n\\n    def has_distortion(self) -> bool:\\n        if self.model in (\\"SIMPLE_PINHOLE\\", \\"PINHOLE\\"):\\n            return False\\n        extra = self.params[3:] if self.model.startswith(\\"SIMPLE\\") or self.model in (\\"RADIAL\\", \\"FOV\\") else self.params[4:]\\n        return bool(np.any(np.abs(extra) > 1e-9))\\n\\n\\n@dataclass\\nclass Image:\\n    id: int\\n    qvec: np.ndarray  # w, x, y, z (world -> camera)\\n    tvec: np.ndarray\\n    camera_id: int\\n    name: str\\n\\n    def world_to_camera(self) -> np.ndarray:\\n        m = np.eye(4)\\n        m[:3, :3] = qvec_to_rotmat(self.qvec)\\n        m[:3, 3] = self.tvec\\n        return m\\n\\n\\n@dataclass\\nclass SparseModel:\\n    cameras: dict[int, Camera]\\n    images: dict[int, Image]\\n    points_xyz: np.ndarray  # (N, 3) float64\\n    points_rgb: np.ndarray  # (N, 3) uint8\\n\\n\\ndef qvec_to_rotmat(q: np.ndarray) -> np.ndarray:\\n    w, x, y, z = (float(v) for v in q)\\n    return np.array(\\n        [\\n            [1 - 2 * y * y - 2 * z * z, 2 * x * y - 2 * w * z, 2 * z * x + 2 * w * y],\\n            [2 * x * y + 2 * w * z, 1 - 2 * x * x - 2 * z * z, 2 * y * z - 2 * w * x],\\n            [2 * z * x - 2 * w * y, 2 * y * z + 2 * w * x, 1 - 2 * x * x - 2 * y * y],\\n        ]\\n    )\\n\\n\\ndef _read(f, fmt: str):\\n    size = struct.calcsize(\\"<\\" + fmt)\\n    data = f.read(size)\\n    if len(data) != size:\\n        raise ValueError(\\"Truncated COLMAP binary file\\")\\n    return struct.unpack(\\"<\\" + fmt, data)\\n\\n\\ndef _read_cameras_bin(path: Path) -> dict[int, Camera]:\\n    cams = {}\\n    with open(path, \\"rb\\") as f:\\n        (n,) = _read(f, \\"Q\\")\\n        for _ in range(n):\\n            cam_id, model_id, width, height = _read(f, \\"iiQQ\\")\\n            name, count = CAMERA_MODELS[model_id]\\n            params = np.array(_read(f, \\"d\\" * count))\\n            cams[cam_id] = Camera(cam_id, name, int(width), int(height), params)\\n    return cams\\n\\n\\ndef _read_images_bin(path: Path) -> dict[int, Image]:\\n    imgs = {}\\n    with open(path, \\"rb\\") as f:\\n        (n,) = _read(f, \\"Q\\")\\n        for _ in range(n):\\n            props = _read(f, \\"idddddddi\\")\\n            image_id = props[0]\\n            qvec = np.array(props[1:5])\\n            tvec = np.array(props[5:8])\\n            camera_id = props[8]\\n            name = b\\"\\"\\n            while True:\\n                c = f.read(1)\\n                if c == b\\"\\\\x00\\":\\n                    break\\n                if not c:\\n                    raise ValueError(\\"Truncated image name\\")\\n                name += c\\n            (n2d,) = _read(f, \\"Q\\")\\n            f.seek(24 * n2d, 1)  # x, y (double) + point3D id (int64)\\n            imgs[image_id] = Image(image_id, qvec, tvec, camera_id, name.decode(\\"utf-8\\"))\\n    return imgs\\n\\n\\ndef _read_points_bin(path: Path) -> tuple[np.ndarray, np.ndarray]:\\n    with open(path, \\"rb\\") as f:\\n        (n,) = _read(f, \\"Q\\")\\n        xyz = np.empty((n, 3), np.float64)\\n        rgb = np.empty((n, 3), np.uint8)\\n        for i in range(n):\\n            props = _read(f, \\"QdddBBBd\\")\\n            xyz[i] = props[1:4]\\n            rgb[i] = props[4:7]\\n            (track_len,) = _read(f, \\"Q\\")\\n            f.seek(8 * track_len, 1)\\n    return xyz, rgb\\n\\n\\ndef _read_cameras_txt(path: Path) -> dict[int, Camera]:\\n    cams = {}\\n    for line in path.read_text(encoding=\\"utf-8\\").splitlines():\\n        if not line.strip() or line.startswith(\\"#\\"):\\n            continue\\n        parts = line.split()\\n        cam_id, model, w, h = int(parts[0]), parts[1], int(parts[2]), int(parts[3])\\n        params = np.array([float(v) for v in parts[4 : 4 + _NAME_TO_COUNT[model]]])\\n        cams[cam_id] = Camera(cam_id, model, w, h, params)\\n    return cams\\n\\n\\ndef _read_images_txt(path: Path) -> dict[int, Image]:\\n    imgs = {}\\n    lines = [l for l in path.read_text(encoding=\\"utf-8\\").splitlines() if not l.startswith(\\"#\\")]\\n    # Pose lines alternate with 2D-point lines (which may be empty).\\n    for i in range(0, len(lines), 2):\\n        parts = lines[i].split()\\n        if len(parts) < 10:\\n            continue\\n        image_id = int(parts[0])\\n        qvec = np.array([float(v) for v in parts[1:5]])\\n        tvec = np.array([float(v) for v in parts[5:8]])\\n        imgs[image_id] = Image(image_id, qvec, tvec, int(parts[8]), \\" \\".join(parts[9:]))\\n    return imgs\\n\\n\\ndef _read_points_txt(path: Path) -> tuple[np.ndarray, np.ndarray]:\\n    xyz, rgb = [], []\\n    for line in path.read_text(encoding=\\"utf-8\\").splitlines():\\n        if not line.strip() or line.startswith(\\"#\\"):\\n            continue\\n        parts = line.split()\\n        xyz.append([float(v) for v in parts[1:4]])\\n        rgb.append([int(v) for v in parts[4:7]])\\n    return np.array(xyz, np.float64).reshape(-1, 3), np.array(rgb, np.uint8).reshape(-1, 3)\\n\\n\\ndef read_sparse_model(sparse_dir: Path) -> SparseModel:\\n    sparse_dir = Path(sparse_dir)\\n    if (sparse_dir / \\"cameras.bin\\").exists():\\n        cams = _read_cameras_bin(sparse_dir / \\"cameras.bin\\")\\n        imgs = _read_images_bin(sparse_dir / \\"images.bin\\")\\n        xyz, rgb = _read_points_bin(sparse_dir / \\"points3D.bin\\")\\n    elif (sparse_dir / \\"cameras.txt\\").exists():\\n        cams = _read_cameras_txt(sparse_dir / \\"cameras.txt\\")\\n        imgs = _read_images_txt(sparse_dir / \\"images.txt\\")\\n        xyz, rgb = _read_points_txt(sparse_dir / \\"points3D.txt\\")\\n    else:\\n        raise FileNotFoundError(f\\"No COLMAP model in {sparse_dir}\\")\\n    return SparseModel(cams, imgs, xyz, rgb)\\n", "lib/splat_restorer/data.py": "\\"\\"\\"Pair datasets read from extracted pair tars (see pairs.py).\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport random\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom PIL import Image\\n\\n\\ndef collect_items(pairs_root: Path, scenes: list[str], recipes: list[str] | None = None) -> list[dict]:\\n    items = []\\n    for s in scenes:\\n        index = json.loads((pairs_root / s / \\"index.json\\").read_text(encoding=\\"utf-8\\"))\\n        for rname, rec in index[\\"recipes\\"].items():\\n            if recipes and rname not in recipes:\\n                continue\\n            for it in rec[\\"items\\"]:\\n                items.append(\\n                    {\\n                        \\"scene\\": s,\\n                        \\"recipe\\": rname,\\n                        \\"frame\\": it[\\"frame\\"],\\n                        \\"deg\\": str(pairs_root / s / it[\\"deg\\"]),\\n                        \\"gt\\": str(pairs_root / s / it[\\"gt\\"]),\\n                        \\"ref\\": str(pairs_root / s / it[\\"ref_path\\"]),\\n                    }\\n                )\\n    return items\\n\\n\\ndef _load(path: str) -> torch.Tensor:\\n    arr = np.array(Image.open(path).convert(\\"RGB\\"), dtype=np.uint8)\\n    return torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0\\n\\n\\ndef eval_crop_box(h: int, w: int, multiple: int = 64) -> tuple[int, int, int, int]:\\n    \\"\\"\\"Centred crop whose sides are multiples of ``multiple`` (top, left, h, w).\\"\\"\\"\\n    ch, cw = h - h % multiple, w - w % multiple\\n    return (h - ch) // 2, (w - cw) // 2, ch, cw\\n\\n\\nclass PairDataset(torch.utils.data.Dataset):\\n    \\"\\"\\"Returns deg / gt / ref tensors in [0, 1], shape (3, H, W).\\n\\n    Training: one random ``crop`` x ``crop`` window (shared by all three images,\\n    the reference is a nearby view so the window roughly corresponds) plus a\\n    random horizontal flip. Evaluation: deterministic centred crop to multiples\\n    of 64 (540x960 -> 512x960).\\n    \\"\\"\\"\\n\\n    def __init__(self, items: list[dict], train: bool, crop: int = 512):\\n        self.items = items\\n        self.train = train\\n        self.crop = crop\\n\\n    def __len__(self):\\n        return len(self.items)\\n\\n    def __getitem__(self, idx):\\n        it = self.items[idx]\\n        deg, gt, ref = _load(it[\\"deg\\"]), _load(it[\\"gt\\"]), _load(it[\\"ref\\"])\\n        _, h, w = deg.shape\\n        if self.train:\\n            c = min(self.crop, h - h % 8, w - w % 8)\\n            top = random.randint(0, h - c)\\n            left = random.randint(0, w - c)\\n            sl = (slice(None), slice(top, top + c), slice(left, left + c))\\n            deg, gt = deg[sl], gt[sl]\\n            rh, rw = ref.shape[1:]\\n            rt, rl = min(top, rh - c), min(left, rw - c)\\n            ref = ref[:, rt : rt + c, rl : rl + c]\\n            if random.random() < 0.5:\\n                deg, gt, ref = deg.flip(-1), gt.flip(-1), ref.flip(-1)\\n        else:\\n            top, left, ch, cw = eval_crop_box(h, w)\\n            sl = (slice(None), slice(top, top + ch), slice(left, left + cw))\\n            deg, gt, ref = deg[sl], gt[sl], ref[sl]\\n        return {\\"deg\\": deg, \\"gt\\": gt, \\"ref\\": ref, \\"idx\\": idx}\\n\\n\\ndef split_holdout(items: list[dict], frac: float = 0.02, seed: int = 0) -> tuple[list[dict], list[dict]]:\\n    \\"\\"\\"Small validation slice for loss curves (same scenes; not the real test).\\"\\"\\"\\n    rng = random.Random(seed)\\n    order = list(range(len(items)))\\n    rng.shuffle(order)\\n    n_val = max(1, int(len(items) * frac))\\n    val = [items[i] for i in order[:n_val]]\\n    train = [items[i] for i in order[n_val:]]\\n    return train, val\\n", "lib/splat_restorer/distill.py": "\\"\\"\\"Distill-back test: does restoring novel views actually improve the *splat*?\\n\\nFor each test scene and recipe, starting from the same weak splat:\\n\\n* control: fine-tune on the real training photos only;\\n* <method>: fine-tune on the training photos plus restored renders of the\\n  pair-frame poses (pseudo ground truth, Difix3D-style).\\n\\nBoth are scored on the scene\'s eval frames, which no splat ever trained on.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image\\n\\nfrom .gs_trainer import TrainConfig, View, params_from_state, render, ssim, train_gaussians\\nfrom .splits import subsample\\nfrom .workspace import Workspace\\n\\n\\ndef _load_photo(path: Path, device) -> torch.Tensor:\\n    return torch.from_numpy(np.asarray(Image.open(path).convert(\\"RGB\\"), dtype=np.float32) / 255.0).to(device)\\n\\n\\ndef pad_to_multiple(x: torch.Tensor, m: int = 64) -> tuple[torch.Tensor, tuple[int, int]]:\\n    \\"\\"\\"x: (B, 3, H, W). Reflect-pad bottom/right to multiples of m.\\"\\"\\"\\n    h, w = x.shape[-2:]\\n    ph, pw = (-h) % m, (-w) % m\\n    return F.pad(x, (0, pw, 0, ph), mode=\\"reflect\\"), (h, w)\\n\\n\\ndef restore_full(fn, deg_hwc: torch.Tensor, ref_hwc: torch.Tensor) -> torch.Tensor:\\n    deg = deg_hwc.permute(2, 0, 1)[None]\\n    ref = ref_hwc.permute(2, 0, 1)[None]\\n    deg_p, (h, w) = pad_to_multiple(deg)\\n    ref_p, _ = pad_to_multiple(ref)\\n    out = fn(deg_p, ref_p).float()[..., :h, :w]\\n    return out[0].permute(1, 2, 0).clamp(0, 1)\\n\\n\\n@torch.no_grad()\\ndef _score(params, frames, photos, Ks, w2c, width, height, net_lpips) -> dict:\\n    ps, ss, ls = [], [], []\\n    for i in frames:\\n        img, _ = render(params, w2c[i], Ks[i], width, height, 3)\\n        gt = photos[i]\\n        ps.append(10 * math.log10(1 / max(F.mse_loss(img, gt).item(), 1e-12)))\\n        ss.append(ssim(img, gt).item())\\n        ls.append(net_lpips(img.permute(2, 0, 1)[None] * 2 - 1, gt.permute(2, 0, 1)[None] * 2 - 1).item())\\n    return {\\"psnr\\": float(np.mean(ps)), \\"ssim\\": float(np.mean(ss)), \\"lpips\\": float(np.mean(ls))}\\n\\n\\ndef distill_scene(\\n    ws: Workspace,\\n    pairs_root: Path,\\n    scene: str,\\n    recipe: str,\\n    methods: dict,\\n    finetune_steps: int = 1500,\\n    max_pseudo: int = 60,\\n    pseudo_share: float = 0.5,\\n    device=\\"cuda\\",\\n    net_lpips=None,\\n    log=print,\\n) -> dict:\\n    index = json.loads((pairs_root / scene / \\"index.json\\").read_text(encoding=\\"utf-8\\"))\\n    rec = index[\\"recipes\\"][recipe]\\n    width, height = index[\\"width\\"], index[\\"height\\"]\\n    Ks = torch.tensor(index[\\"cameras\\"][\\"K\\"], device=device)\\n    w2c = torch.tensor(index[\\"cameras\\"][\\"w2c\\"], device=device)\\n    root = pairs_root / scene\\n    eval_photos = {i: _load_photo(root / \\"eval\\" / f\\"{i:05d}.jpg\\", device) for i in rec[\\"eval\\"]}\\n    train_photos = {i: _load_photo(root / recipe / \\"ref\\" / f\\"{i:05d}.jpg\\", device) for i in rec[\\"train\\"]}\\n    start = torch.load(ws.splat_path(scene, recipe), map_location=\\"cpu\\", weights_only=False)\\n    ft_cfg = TrainConfig(steps=finetune_steps, densify=False, lr_means=1.6e-5, lr_means_final_frac=0.1, seed=0)\\n    score = lambda p: _score(p, rec[\\"eval\\"], eval_photos, Ks, w2c, width, height, net_lpips)\\n\\n    results = {\\"start\\": score(params_from_state(start, device))}\\n    train_views = [View(train_photos[i], w2c[i], Ks[i]) for i in rec[\\"train\\"]]\\n\\n    params = train_gaussians(train_views, width, height, ft_cfg, params=params_from_state(start, device))\\n    results[\\"control\\"] = score(params)\\n    del params\\n\\n    pseudo_frames = subsample(rec[\\"pair\\"], max_pseudo)\\n    from .splits import nearest_reference\\n\\n    w2c_np = np.array(index[\\"cameras\\"][\\"w2c\\"])\\n    centers = -np.einsum(\\"fij,fi->fj\\", w2c_np[:, :3, :3], w2c_np[:, :3, 3])\\n    forwards = w2c_np[:, 2, :3]\\n    base = params_from_state(start, device)\\n    with torch.no_grad():\\n        renders = {i: render(base, w2c[i], Ks[i], width, height, 3)[0] for i in pseudo_frames}\\n    del base\\n\\n    for name, loader in methods.items():\\n        fn = loader()\\n        with torch.no_grad():\\n            pseudo = {}\\n            for i in pseudo_frames:\\n                ref = train_photos[nearest_reference(i, rec[\\"train\\"], centers, forwards)]\\n                pseudo[i] = restore_full(fn, renders[i], ref)\\n        del fn\\n        torch.cuda.empty_cache()\\n        # Pseudo views together get ``pseudo_share`` of the samples.\\n        w_pseudo = pseudo_share / (1 - pseudo_share) * len(train_views) / max(len(pseudo), 1)\\n        views = train_views + [View(pseudo[i], w2c[i], Ks[i], weight=w_pseudo) for i in pseudo_frames]\\n        params = train_gaussians(views, width, height, ft_cfg, params=params_from_state(start, device))\\n        results[name] = score(params)\\n        del params, pseudo\\n        torch.cuda.empty_cache()\\n    log(f\\"  {scene[:12]} {recipe}: \\" + \\", \\".join(f\\"{k} {v[\'psnr\']:.2f}/{v[\'lpips\']:.3f}\\" for k, v in results.items()))\\n    return results\\n\\n\\ndef run_distill(\\n    ws: Workspace, pairs_root: Path, test_scenes: list[str], recipes: list[str], methods: dict, tag: str,\\n    finetune_steps: int = 1500, device=\\"cuda\\", log=print,\\n) -> Path:\\n    import lpips\\n\\n    net_lpips = lpips.LPIPS(net=\\"alex\\", verbose=False).to(device).eval()\\n    out_dir = ws.results_dir / tag\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    partial = out_dir / \\"per_scene.json\\"\\n    all_res = json.loads(partial.read_text(encoding=\\"utf-8\\")) if partial.exists() else {}\\n    for scene in test_scenes:\\n        for recipe in recipes:\\n            key = f\\"{scene}/{recipe}\\"\\n            if key in all_res:\\n                continue\\n            if not ws.splat_path(scene, recipe).exists():\\n                log(f\\"  skip {key}: no saved splat\\")\\n                continue\\n            all_res[key] = distill_scene(\\n                ws, pairs_root, scene, recipe, methods, finetune_steps=finetune_steps, device=device,\\n                net_lpips=net_lpips, log=log,\\n            )\\n            partial.write_text(json.dumps(all_res, indent=2), encoding=\\"utf-8\\")\\n\\n    lines = []\\n    for recipe in recipes:\\n        keys = [k for k in all_res if k.endswith(\\"/\\" + recipe)]\\n        if not keys:\\n            continue\\n        names = list(all_res[keys[0]])\\n        lines += [f\\"\\\\n### {recipe} ({len(keys)} scenes)\\\\n\\", \\"| condition | PSNR ↑ | Δ vs control | SSIM ↑ | LPIPS ↓ | Δ vs control | wins vs control |\\", \\"|---|---|---|---|---|---|---|\\"]\\n        ctrl = {m: np.mean([all_res[k][\\"control\\"][m] for k in keys]) for m in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n        for n in names:\\n            mean = {m: np.mean([all_res[k][n][m] for k in keys]) for m in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n            wins = sum(all_res[k][n][\\"psnr\\"] > all_res[k][\\"control\\"][\\"psnr\\"] for k in keys)\\n            lines.append(\\n                f\\"| {n} | {mean[\'psnr\']:.2f} | {mean[\'psnr\'] - ctrl[\'psnr\']:+.2f} | {mean[\'ssim\']:.4f} | \\"\\n                f\\"{mean[\'lpips\']:.4f} | {mean[\'lpips\'] - ctrl[\'lpips\']:+.4f} | {wins}/{len(keys)} |\\"\\n            )\\n    md = \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return out_dir\\n", "lib/splat_restorer/dl3dv.py": "\\"\\"\\"DL3DV-Benchmark access: scene listing, deterministic split, per-scene download.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport io\\nimport random\\nimport shutil\\nfrom pathlib import Path\\n\\nREPO = \\"DL3DV/DL3DV-Benchmark\\"\\nSCENE_SUBDIR = \\"gaussian_splat\\"  # undistorted images + COLMAP model\\nIMAGES = \\"images_4\\"  # 960x540\\n\\n\\ndef list_scenes() -> list[str]:\\n    from huggingface_hub import HfApi\\n\\n    entries = HfApi().list_repo_tree(REPO, repo_type=\\"dataset\\", recursive=False)\\n    return sorted(e.path for e in entries if len(e.path) == 64 and all(c in \\"0123456789abcdef\\" for c in e.path))\\n\\n\\ndef scene_labels() -> dict[str, str]:\\n    \\"\\"\\"scene hash -> coarse label (e.g. indoor/outdoor) from benchmark-meta.csv, if available.\\"\\"\\"\\n    from huggingface_hub import hf_hub_download\\n\\n    try:\\n        path = hf_hub_download(REPO, \\"benchmark-meta.csv\\", repo_type=\\"dataset\\")\\n    except Exception:\\n        return {}\\n    text = Path(path).read_text(encoding=\\"utf-8\\", errors=\\"replace\\")\\n    rows = list(csv.DictReader(io.StringIO(text)))\\n    if not rows:\\n        return {}\\n    hash_key = next((k for k in rows[0] if \\"hash\\" in k.lower()), None)\\n    # \\"environment\\" is bounded / unbounded (roughly indoor-ish vs. open scenes).\\n    label_key = next((k for k in rows[0] if k.lower() in (\\"environment\\", \\"env\\", \\"label\\")), None)\\n    if hash_key is None:\\n        return {}\\n    return {r[hash_key].strip(): (r.get(label_key) or \\"\\").strip() if label_key else \\"\\" for r in rows}\\n\\n\\ndef make_split(scenes: list[str], n_test: int, seed: int = 1234, labels: dict[str, str] | None = None) -> dict:\\n    \\"\\"\\"Shuffle once; n_test scenes become test, the rest an ordered train pool.\\n\\n    Test scenes are drawn round-robin over ``labels`` (e.g. bounded/unbounded)\\n    so both kinds are represented. Train runs with N scenes always take the\\n    first N of the pool, so a 10-scene run is a subset of the 25-scene run\\n    (clean scaling curve).\\n    \\"\\"\\"\\n    rng = random.Random(seed)\\n    order = list(scenes)\\n    rng.shuffle(order)\\n    labels = labels or {}\\n    groups: dict[str, list[str]] = {}\\n    for s in order:\\n        groups.setdefault(labels.get(s, \\"\\"), []).append(s)\\n    test: list[str] = []\\n    keys = sorted(groups)\\n    while len(test) < min(n_test, len(order)):\\n        for k in keys:\\n            if groups[k] and len(test) < n_test:\\n                test.append(groups[k].pop(0))\\n    test_set = set(test)\\n    return {\\n        \\"seed\\": seed,\\n        \\"test\\": test,\\n        \\"train_pool\\": [s for s in order if s not in test_set],\\n        \\"labels\\": {s: labels.get(s, \\"\\") for s in order},\\n    }\\n\\n\\ndef download_scene(scene: str, dest: Path) -> Path:\\n    \\"\\"\\"Download images_4 + sparse model of one scene. Returns the scene dir.\\"\\"\\"\\n    from huggingface_hub import snapshot_download\\n\\n    dest.mkdir(parents=True, exist_ok=True)\\n    snapshot_download(\\n        REPO,\\n        repo_type=\\"dataset\\",\\n        local_dir=str(dest),\\n        allow_patterns=[f\\"{scene}/{SCENE_SUBDIR}/sparse/0/*\\", f\\"{scene}/{SCENE_SUBDIR}/{IMAGES}/*\\"],\\n        max_workers=16,\\n    )\\n    scene_dir = dest / scene / SCENE_SUBDIR\\n    if not (scene_dir / IMAGES).is_dir():\\n        raise FileNotFoundError(f\\"Download of {scene} produced no {IMAGES} folder\\")\\n    return scene_dir\\n\\n\\ndef remove_download(dest: Path) -> None:\\n    shutil.rmtree(dest, ignore_errors=True)\\n", "lib/splat_restorer/evaluate.py": "\\"\\"\\"Image-level evaluation on held-out *test scenes* (never seen in training).\\n\\nMetrics: PSNR, SSIM, LPIPS-Alex (training used LPIPS-VGG, so the evaluation\\nmetric is not the one the models optimised). A method only \\"works\\" if it\\nimproves LPIPS *without* lowering PSNR much: diffusion models can hallucinate\\nplausible texture that looks sharper yet is wrong.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport json\\nimport math\\nimport random\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\n\\nfrom .data import PairDataset, collect_items\\nfrom .gs_trainer import ssim\\nfrom .workspace import Workspace\\n\\n\\ndef _metrics(out: torch.Tensor, gt: torch.Tensor, net_lpips) -> dict:\\n    mse = F.mse_loss(out, gt).item()\\n    return {\\n        \\"psnr\\": 10 * math.log10(1 / max(mse, 1e-12)),\\n        \\"ssim\\": ssim(out[0].permute(1, 2, 0), gt[0].permute(1, 2, 0)).item(),\\n        \\"lpips\\": net_lpips(out * 2 - 1, gt * 2 - 1).mean().item(),\\n    }\\n\\n\\ndef build_methods(ws: Workspace, restorer_runs: list[str], baseline_runs: list[str], device=\\"cuda\\") -> dict:\\n    \\"\\"\\"name -> zero-arg loader returning callable(deg, ref) -> out (all [0, 1], Bx3xHxW).\\"\\"\\"\\n    from .baseline import load_baseline, unsharp\\n\\n    methods = {\\n        \\"input\\": lambda: (lambda d, r: d),\\n        \\"unsharp\\": lambda: (lambda d, r: unsharp(d)),\\n    }\\n    for run in baseline_runs:\\n        path = ws.runs_dir / run / \\"model_final.pt\\"\\n        if path.exists():\\n            methods[run] = (lambda p=path: (lambda net: (lambda d, r: net.restore(d, r if net.use_ref else None)))(load_baseline(p, device)))\\n    for run in restorer_runs:\\n        path = ws.runs_dir / run / \\"model_final.pt\\"\\n        if path.exists():\\n            def loader(p=path):\\n                from .train_restorer import load_restorer\\n\\n                model, use_ref = load_restorer(p, device)\\n                return lambda d, r: model.restore(d, r if use_ref else None)\\n\\n            methods[run] = loader\\n    return methods\\n\\n\\n@torch.no_grad()\\ndef evaluate(\\n    ws: Workspace,\\n    pairs_root: Path,\\n    test_scenes: list[str],\\n    methods: dict,\\n    tag: str,\\n    max_items_per_scene: int = 40,\\n    n_sheets: int = 8,\\n    device=\\"cuda\\",\\n    log=print,\\n) -> Path:\\n    import lpips\\n\\n    net_lpips = lpips.LPIPS(net=\\"alex\\", verbose=False).to(device).eval()\\n    items = []\\n    for s in test_scenes:\\n        its = collect_items(pairs_root, [s])\\n        random.Random(0).shuffle(its)\\n        items += its[:max_items_per_scene]\\n    ds = PairDataset(items, train=False)\\n    log(f\\"[eval {tag}] {len(items)} pairs from {len(test_scenes)} test scenes, methods: {list(methods)}\\")\\n\\n    out_dir = ws.results_dir / tag\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    sheet_idx = sorted(random.Random(1).sample(range(len(ds)), min(n_sheets, len(ds))))\\n    sheet_cols: dict[int, list] = defaultdict(list)\\n    rows = []\\n    for name, loader in methods.items():\\n        fn = loader()\\n        for i in range(len(ds)):\\n            s = ds[i]\\n            deg, gt, ref = (s[k][None].to(device) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n            out = fn(deg, ref).float().clamp(0, 1)\\n            m = _metrics(out, gt, net_lpips)\\n            it = items[i]\\n            rows.append({\\"method\\": name, \\"scene\\": it[\\"scene\\"], \\"recipe\\": it[\\"recipe\\"], \\"frame\\": it[\\"frame\\"], **m})\\n            if i in sheet_idx:\\n                if not sheet_cols[i]:\\n                    sheet_cols[i].append((\\"ref\\", ref[0].cpu()))\\n                sheet_cols[i].append((name, out[0].cpu()))\\n                if name == list(methods)[-1]:\\n                    sheet_cols[i].append((\\"gt\\", gt[0].cpu()))\\n        del fn\\n        torch.cuda.empty_cache()\\n        log(f\\"[eval {tag}] {name} done\\")\\n\\n    with open(out_dir / \\"per_item.csv\\", \\"w\\", newline=\\"\\") as f:\\n        w = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        w.writeheader()\\n        w.writerows(rows)\\n\\n    summary = _summarise(rows)\\n    (out_dir / \\"summary.json\\").write_text(json.dumps(summary, indent=2), encoding=\\"utf-8\\")\\n    (out_dir / \\"summary.md\\").write_text(_markdown(summary), encoding=\\"utf-8\\")\\n    _contact_sheets(out_dir, sheet_cols)\\n    log(_markdown(summary))\\n    return out_dir\\n\\n\\ndef _summarise(rows: list[dict]) -> dict:\\n    groups = defaultdict(list)\\n    for r in rows:\\n        groups[(r[\\"method\\"], r[\\"recipe\\"])].append(r)\\n        groups[(r[\\"method\\"], \\"all\\")].append(r)\\n    out = {}\\n    for (method, recipe), rs in groups.items():\\n        out.setdefault(method, {})[recipe] = {\\n            k: float(np.mean([r[k] for r in rs])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")\\n        } | {\\"n\\": len(rs)}\\n    return out\\n\\n\\ndef _markdown(summary: dict) -> str:\\n    recipes = sorted({rec for m in summary.values() for rec in m}, key=lambda r: (r != \\"all\\", r))\\n    base = summary.get(\\"input\\", {})\\n    lines = []\\n    for rec in recipes:\\n        lines.append(f\\"\\\\n### {rec}\\\\n\\")\\n        lines.append(\\"| method | PSNR ↑ | ΔPSNR | SSIM ↑ | LPIPS ↓ | ΔLPIPS |\\")\\n        lines.append(\\"|---|---|---|---|---|---|\\")\\n        for method, by_rec in summary.items():\\n            if rec not in by_rec:\\n                continue\\n            m = by_rec[rec]\\n            b = base.get(rec, m)\\n            lines.append(\\n                f\\"| {method} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - b[\'psnr\']:+.2f} | {m[\'ssim\']:.4f} | \\"\\n                f\\"{m[\'lpips\']:.4f} | {m[\'lpips\'] - b[\'lpips\']:+.4f} |\\"\\n            )\\n    return \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n\\n\\ndef _contact_sheets(out_dir: Path, sheet_cols: dict) -> None:\\n    from PIL import Image, ImageDraw\\n\\n    for i, cols in sheet_cols.items():\\n        tiles = []\\n        for name, t in cols:\\n            arr = (t.permute(1, 2, 0).numpy() * 255 + 0.5).astype(np.uint8)\\n            img = Image.fromarray(arr)\\n            ImageDraw.Draw(img).text((8, 8), name, fill=(255, 255, 0))\\n            tiles.append(img)\\n        w, h = tiles[0].size\\n        cols_n = 3\\n        rows_n = math.ceil(len(tiles) / cols_n)\\n        sheet = Image.new(\\"RGB\\", (w * cols_n, h * rows_n))\\n        for k, tile in enumerate(tiles):\\n            sheet.paste(tile, ((k % cols_n) * w, (k // cols_n) * h))\\n        sheet.save(out_dir / f\\"sheet_{i:04d}.jpg\\", quality=88)\\n", "lib/splat_restorer/gs_trainer.py": "\\"\\"\\"Small, self-contained 3DGS trainer on top of gsplat 1.5.x.\\n\\nIt is intentionally plain: the pair generator only needs *plausible* splat\\nartefacts, and the distill-back test needs a fair, repeatable fine-tune.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\n\\nSH_C0 = 0.28209479177387814\\n\\n\\n@dataclass\\nclass View:\\n    image: torch.Tensor  # (H, W, 3) float in [0, 1] or uint8; any device\\n    w2c: torch.Tensor  # (4, 4)\\n    K: torch.Tensor  # (3, 3)\\n    weight: float = 1.0\\n\\n\\n@dataclass\\nclass TrainConfig:\\n    steps: int = 4000\\n    sh_degree: int = 3\\n    sh_increase_every: int = 1000\\n    ssim_lambda: float = 0.2\\n    densify: bool = True\\n    refine_start: int = 500\\n    refine_stop_frac: float = 0.75\\n    opacity_reset_every: int = 1_000_000  # disabled: short runs do not recover\\n    lr_means: float = 1.6e-4\\n    lr_means_final_frac: float = 0.01\\n    lr_scales: float = 5e-3\\n    lr_quats: float = 1e-3\\n    lr_opacities: float = 5e-2\\n    lr_sh0: float = 2.5e-3\\n    lr_shN: float = 2.5e-3 / 20\\n    init_opacity: float = 0.1\\n    seed: int = 0\\n    log_every: int = 0\\n    extra: dict = field(default_factory=dict)\\n\\n\\ndef _knn_mean_dist(xyz: np.ndarray, k: int = 3) -> np.ndarray:\\n    from scipy.spatial import cKDTree\\n\\n    tree = cKDTree(xyz)\\n    d, _ = tree.query(xyz, k=k + 1)\\n    return np.sqrt((d[:, 1:] ** 2).mean(axis=1))\\n\\n\\ndef init_params(xyz: np.ndarray, rgb: np.ndarray, sh_degree: int, init_opacity: float, device) -> torch.nn.ParameterDict:\\n    n = len(xyz)\\n    dist = np.clip(_knn_mean_dist(xyz), 1e-7, None)\\n    scales = np.log(np.repeat(dist[:, None], 3, axis=1))\\n    quats = np.zeros((n, 4), np.float32)\\n    quats[:, 0] = 1.0\\n    opac = np.full(n, math.log(init_opacity / (1 - init_opacity)), np.float32)\\n    sh0 = ((rgb - 0.5) / SH_C0)[:, None, :]\\n    k = (sh_degree + 1) ** 2 - 1\\n    shN = np.zeros((n, k, 3), np.float32)\\n    t = lambda a: torch.nn.Parameter(torch.as_tensor(a, dtype=torch.float32, device=device))\\n    return torch.nn.ParameterDict(\\n        {\\n            \\"means\\": t(xyz),\\n            \\"scales\\": t(scales),\\n            \\"quats\\": t(quats),\\n            \\"opacities\\": t(opac),\\n            \\"sh0\\": t(sh0),\\n            \\"shN\\": t(shN),\\n        }\\n    )\\n\\n\\ndef make_optimizers(params, cfg: TrainConfig, scene_scale: float) -> dict[str, torch.optim.Optimizer]:\\n    lrs = {\\n        \\"means\\": cfg.lr_means * scene_scale,\\n        \\"scales\\": cfg.lr_scales,\\n        \\"quats\\": cfg.lr_quats,\\n        \\"opacities\\": cfg.lr_opacities,\\n        \\"sh0\\": cfg.lr_sh0,\\n        \\"shN\\": cfg.lr_shN,\\n    }\\n    return {k: torch.optim.Adam([{\\"params\\": params[k], \\"lr\\": lr, \\"name\\": k}], eps=1e-15) for k, lr in lrs.items()}\\n\\n\\ndef render(params, w2c: torch.Tensor, K: torch.Tensor, width: int, height: int, sh_degree: int):\\n    \\"\\"\\"Render one view. Returns (image HxWx3 clamped to [0,1] in the graph, info).\\"\\"\\"\\n    from gsplat import rasterization\\n\\n    colors = torch.cat([params[\\"sh0\\"], params[\\"shN\\"]], dim=1)\\n    img, _alpha, info = rasterization(\\n        means=params[\\"means\\"],\\n        quats=params[\\"quats\\"],\\n        scales=torch.exp(params[\\"scales\\"]),\\n        opacities=torch.sigmoid(params[\\"opacities\\"]),\\n        colors=colors,\\n        viewmats=w2c[None],\\n        Ks=K[None],\\n        width=width,\\n        height=height,\\n        sh_degree=sh_degree,\\n        packed=False,\\n    )\\n    return img[0].clamp(0.0, 1.0), info\\n\\n\\ndef _gauss_window(size: int = 11, sigma: float = 1.5, device=None) -> torch.Tensor:\\n    x = torch.arange(size, dtype=torch.float32, device=device) - size // 2\\n    g = torch.exp(-(x**2) / (2 * sigma**2))\\n    g = g / g.sum()\\n    return (g[:, None] @ g[None, :])[None, None].repeat(3, 1, 1, 1)\\n\\n\\ndef ssim(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\\n    \\"\\"\\"SSIM of two (H, W, 3) images in [0, 1].\\"\\"\\"\\n    x = a.permute(2, 0, 1)[None]\\n    y = b.permute(2, 0, 1)[None]\\n    w = _gauss_window(device=a.device)\\n    pad = w.shape[-1] // 2\\n    mu_x = F.conv2d(x, w, padding=pad, groups=3)\\n    mu_y = F.conv2d(y, w, padding=pad, groups=3)\\n    sxx = F.conv2d(x * x, w, padding=pad, groups=3) - mu_x**2\\n    syy = F.conv2d(y * y, w, padding=pad, groups=3) - mu_y**2\\n    sxy = F.conv2d(x * y, w, padding=pad, groups=3) - mu_x * mu_y\\n    c1, c2 = 0.01**2, 0.03**2\\n    m = ((2 * mu_x * mu_y + c1) * (2 * sxy + c2)) / ((mu_x**2 + mu_y**2 + c1) * (sxx + syy + c2))\\n    return m.mean()\\n\\n\\ndef psnr(a: torch.Tensor, b: torch.Tensor) -> float:\\n    mse = F.mse_loss(a, b).item()\\n    return float(10 * math.log10(1.0 / max(mse, 1e-12)))\\n\\n\\ndef train_gaussians(\\n    views: list[View],\\n    width: int,\\n    height: int,\\n    cfg: TrainConfig,\\n    scene_scale: float = 1.1,\\n    params: torch.nn.ParameterDict | None = None,\\n    init_xyz: np.ndarray | None = None,\\n    init_rgb: np.ndarray | None = None,\\n    log=print,\\n) -> torch.nn.ParameterDict:\\n    \\"\\"\\"Optimise Gaussians on ``views``. Pass ``params`` to fine-tune existing ones.\\"\\"\\"\\n    from gsplat.strategy import DefaultStrategy\\n\\n    device = views[0].w2c.device\\n    gen = torch.Generator().manual_seed(cfg.seed)\\n    if params is None:\\n        assert init_xyz is not None and init_rgb is not None\\n        params = init_params(init_xyz, init_rgb, cfg.sh_degree, cfg.init_opacity, device)\\n    optimizers = make_optimizers(params, cfg, scene_scale)\\n    means_gamma = cfg.lr_means_final_frac ** (1.0 / max(cfg.steps, 1))\\n\\n    strategy = None\\n    if cfg.densify:\\n        strategy = DefaultStrategy(\\n            refine_start_iter=cfg.refine_start,\\n            refine_stop_iter=int(cfg.steps * cfg.refine_stop_frac),\\n            reset_every=cfg.opacity_reset_every,\\n            verbose=False,\\n        )\\n        strategy.check_sanity(params, optimizers)\\n        state = strategy.initialize_state(scene_scale=scene_scale)\\n\\n    weights = torch.tensor([v.weight for v in views], dtype=torch.float64)\\n    for step in range(cfg.steps):\\n        idx = int(torch.multinomial(weights, 1, generator=gen))\\n        v = views[idx]\\n        sh_deg = min(step // cfg.sh_increase_every, cfg.sh_degree) if cfg.densify else cfg.sh_degree\\n        img, info = render(params, v.w2c, v.K, width, height, sh_deg)\\n        if strategy is not None:\\n            strategy.step_pre_backward(params, optimizers, state, step, info)\\n        # Large photo sets may live on the CPU and/or as uint8 to save GPU memory.\\n        gt = v.image.to(img.device, non_blocking=True)\\n        if not gt.dtype.is_floating_point:\\n            gt = gt.float() / 255.0\\n        l1 = (img - gt).abs().mean()\\n        loss = (1 - cfg.ssim_lambda) * l1 + cfg.ssim_lambda * (1 - ssim(img, gt))\\n        loss.backward()\\n        if strategy is not None:\\n            strategy.step_post_backward(params, optimizers, state, step, info, packed=False)\\n        for opt in optimizers.values():\\n            opt.step()\\n            opt.zero_grad(set_to_none=True)\\n        for g in optimizers[\\"means\\"].param_groups:\\n            g[\\"lr\\"] *= means_gamma\\n        if cfg.log_every and (step + 1) % cfg.log_every == 0:\\n            log(f\\"    step {step + 1}/{cfg.steps} loss={loss.item():.4f} n={len(params[\'means\'])}\\")\\n    return params\\n\\n\\ndef params_to_cpu_state(params: torch.nn.ParameterDict) -> dict[str, torch.Tensor]:\\n    return {k: v.detach().cpu() for k, v in params.items()}\\n\\n\\ndef params_from_state(state: dict[str, torch.Tensor], device) -> torch.nn.ParameterDict:\\n    return torch.nn.ParameterDict({k: torch.nn.Parameter(v.to(device)) for k, v in state.items()})\\n", "lib/splat_restorer/LICENSE-DIFIX3D.txt": "NVIDIA License\\n\\n1. Definitions\\n\\n“Licensor” means any person or entity that distributes its Work.\\n“Work” means (a) the original work of authorship made available under this license, which may include software, documentation, or other files, and (b) any additions to or derivative works  thereof  that are made available under this license.\\nThe terms “reproduce,” “reproduction,” “derivative works,” and “distribution” have the meaning as provided under U.S. copyright law; provided, however, that for the purposes of this license, derivative works shall not include works that remain separable from, or merely link (or bind by name) to the interfaces of, the Work.\\nWorks are “made available” under this license by including in or with the Work either (a) a copyright notice referencing the applicability of this license to the Work, or (b) a copy of this license.\\n\\n2. License Grant\\n\\n2.1 Copyright Grant. Subject to the terms and conditions of this license, each Licensor grants to you a perpetual, worldwide, non-exclusive, royalty-free, copyright license to use, reproduce, prepare derivative works of, publicly display, publicly perform, sublicense and distribute its Work and any resulting derivative works in any form.\\n\\n3. Limitations\\n\\n3.1 Redistribution. You may reproduce or distribute the Work only if (a) you do so under this license, (b) you include a complete copy of this license with your distribution, and (c) you retain without modification any copyright, patent, trademark, or attribution notices that are present in the Work.\\n\\n3.2 Derivative Works. You may specify that additional or different terms apply to the use, reproduction, and distribution of your derivative works of the Work (“Your Terms”) only if (a) Your Terms provide that the use limitation in Section 3.3 applies to your derivative works, and (b) you identify the specific derivative works that are subject to Your Terms. Notwithstanding Your Terms, this license (including the redistribution requirements in Section 3.1) will continue to apply to the Work itself.\\n\\n3.3 Use Limitation. The Work and any derivative works thereof only may be used or intended for use non-commercially. Notwithstanding the foregoing, NVIDIA Corporation and its affiliates may use the Work and any derivative works commercially. As used herein, “non-commercially” means for research or evaluation purposes only.\\n\\n3.4 Patent Claims. If you bring or threaten to bring a patent claim against any Licensor (including any claim, cross-claim or counterclaim in a lawsuit) to enforce any patents that you allege are infringed by any Work, then your rights under this license from such Licensor (including the grant in Section 2.1) will terminate immediately.\\n\\n3.5 Trademarks. This license does not grant any rights to use any Licensor’s or its affiliates’ names, logos, or trademarks, except as necessary to reproduce the notices described in this license.\\n\\n3.6 Termination. If you violate any term of this license, then your rights under this license (including the grant in Section 2.1) will terminate immediately.\\n\\n4. Disclaimer of Warranty.\\n\\nTHE WORK IS PROVIDED “AS IS” WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, EITHER EXPRESS OR IMPLIED, INCLUDING WARRANTIES OR CONDITIONS OF \\nMERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE, TITLE OR NON-INFRINGEMENT. YOU BEAR THE RISK OF UNDERTAKING ANY ACTIVITIES UNDER THIS LICENSE. \\n\\n5. Limitation of Liability.\\n\\nEXCEPT AS PROHIBITED BY APPLICABLE LAW, IN NO EVENT AND UNDER NO LEGAL THEORY, WHETHER IN TORT (INCLUDING NEGLIGENCE), CONTRACT, OR OTHERWISE SHALL ANY LICENSOR BE LIABLE TO YOU FOR DAMAGES, INCLUDING ANY DIRECT, INDIRECT, SPECIAL, INCIDENTAL, OR CONSEQUENTIAL DAMAGES ARISING OUT OF OR RELATED TO THIS LICENSE, THE USE OR INABILITY TO USE THE WORK (INCLUDING BUT NOT LIMITED TO LOSS OF GOODWILL, BUSINESS INTERRUPTION, LOST PROFITS OR DATA, COMPUTER FAILURE OR MALFUNCTION, OR ANY OTHER DAMAGES OR LOSSES), EVEN IF THE LICENSOR HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.\\n\\nSTABILITY AI COMMUNITY LICENSE AGREEMENT\\n\\nLast Updated: July 5, 2024\\n\\nINTRODUCTION\\nThis Agreement applies to any individual person or entity (“You”, “Your” or “Licensee”) that uses or distributes any portion or element of the Stability AI Materials or Derivative Works thereof for any Research & Non-Commercial or Commercial purpose. Capitalized terms not otherwise defined herein are defined in Section V below.\\n\\nThis Agreement is intended to allow research, non-commercial, and limited commercial uses of the Models free of charge. In order to ensure that certain limited commercial uses of the Models continue to be allowed, this Agreement preserves free access to the Models for people or organizations generating annual revenue of less than US $1,000,000 (or local currency equivalent).\\n\\nBy clicking “I Accept” or by using or distributing or using any portion or element of the Stability Materials or Derivative Works, You agree that You have read, understood and are bound by the terms of this Agreement. If You are acting on behalf of a company, organization or other entity, then “You” includes you and that entity, and You agree that You: (i) are an authorized representative of such entity with the authority to bind such entity to this Agreement, and (ii) You agree to the terms of this Agreement on that entity’s behalf.\\n\\nRESEARCH & NON-COMMERCIAL USE LICENSE\\nSubject to the terms of this Agreement, Stability AI grants You a non-exclusive, worldwide, non-transferable, non-sublicensable, revocable and royalty-free limited license under Stability AI’s intellectual property or other rights owned by Stability AI embodied in the Stability AI Materials to use, reproduce, distribute, and create Derivative Works of, and make modifications to, the Stability AI Materials for any Research or Non-Commercial Purpose. “Research Purpose” means academic or scientific advancement, and in each case, is not primarily intended for commercial advantage or monetary compensation to You or others. “Non-Commercial Purpose” means any purpose other than a Research Purpose that is not primarily intended for commercial advantage or monetary compensation to You or others, such as personal use (i.e., hobbyist) or evaluation and testing.\\n\\nCOMMERCIAL USE LICENSE\\nSubject to the terms of this Agreement (including the remainder of this Section III), Stability AI grants You a non-exclusive, worldwide, non-transferable, non-sublicensable, revocable and royalty-free limited license under Stability AI’s intellectual property or other rights owned by Stability AI embodied in the Stability AI Materials to use, reproduce, distribute, and create Derivative Works of, and make modifications to, the Stability AI Materials for any Commercial Purpose. “Commercial Purpose” means any purpose other than a Research Purpose or Non-Commercial Purpose that is primarily intended for commercial advantage or monetary compensation to You or others, including but not limited to, (i) creating, modifying, or distributing Your product or service, including via a hosted service or application programming interface, and (ii) for Your business’s or organization’s internal operations. If You are using or distributing the Stability AI Materials for a Commercial Purpose, You must register with Stability AI at (https://stability.ai/community-license). If at any time You or Your Affiliate(s), either individually or in aggregate, generate more than USD $1,000,000 in annual revenue (or the equivalent thereof in Your local currency), regardless of whether that revenue is generated directly or indirectly from the Stability AI Materials or Derivative Works, any licenses granted to You under this Agreement shall terminate as of such date. You must request a license from Stability AI at (https://stability.ai/enterprise) , which Stability AI may grant to You in its sole discretion. If you receive Stability AI Materials, or any Derivative Works thereof, from a Licensee as part of an integrated end user product, then Section III of this Agreement will not apply to you.\\n\\nGENERAL TERMS\\nYour Research, Non-Commercial, and Commercial License(s) under this Agreement are subject to the following terms. a. Distribution & Attribution. If You distribute or make available the Stability AI Materials or a Derivative Work to a third party, or a product or service that uses any portion of them, You shall: (i) provide a copy of this Agreement to that third party, (ii) retain the following attribution notice within a \\"Notice\\" text file distributed as a part of such copies: \\"This Stability AI Model is licensed under the Stability AI Community License, Copyright © Stability AI Ltd. All Rights Reserved”, and (iii) prominently display “Powered by Stability AI” on a related website, user interface, blogpost, about page, or product documentation. If You create a Derivative Work, You may add your own attribution notice(s) to the “Notice” text file included with that Derivative Work, provided that You clearly indicate which attributions apply to the Stability AI Materials and state in the “Notice” text file that You changed the Stability AI Materials and how it was modified. b. Use Restrictions. Your use of the Stability AI Materials and Derivative Works, including any output or results of the Stability AI Materials or Derivative Works, must comply with applicable laws and regulations (including Trade Control Laws and equivalent regulations) and adhere to the Documentation and Stability AI’s AUP, which is hereby incorporated by reference. Furthermore, You will not use the Stability AI Materials or Derivative Works, or any output or results of the Stability AI Materials or Derivative Works, to create or improve any foundational generative AI model (excluding the Models or Derivative Works). c. Intellectual Property. (i) Trademark License. No trademark licenses are granted under this Agreement, and in connection with the Stability AI Materials or Derivative Works, You may not use any name or mark owned by or associated with Stability AI or any of its Affiliates, except as required under Section IV(a) herein. (ii) Ownership of Derivative Works. As between You and Stability AI, You are the owner of Derivative Works You create, subject to Stability AI’s ownership of the Stability AI Materials and any Derivative Works made by or for Stability AI. (iii) Ownership of Outputs. As between You and Stability AI, You own any outputs generated from the Models or Derivative Works to the extent permitted by applicable law. (iv) Disputes. If You or Your Affiliate(s) institute litigation or other proceedings against Stability AI (including a cross-claim or counterclaim in a lawsuit) alleging that the Stability AI Materials, Derivative Works or associated outputs or results, or any portion of any of the foregoing, constitutes infringement of intellectual property or other rights owned or licensable by You, then any licenses granted to You under this Agreement shall terminate as of the date such litigation or claim is filed or instituted. You will indemnify and hold harmless Stability AI from and against any claim by any third party arising out of or related to Your use or distribution of the Stability AI Materials or Derivative Works in violation of this Agreement. (v) Feedback. From time to time, You may provide Stability AI with verbal and/or written suggestions, comments or other feedback related to Stability AI’s existing or prospective technology, products or services (collectively, “Feedback”). You are not obligated to provide Stability AI with Feedback, but to the extent that You do, You hereby grant Stability AI a perpetual, irrevocable, royalty-free, fully-paid, sub-licensable, transferable, non-exclusive, worldwide right and license to exploit the Feedback in any manner without restriction. Your Feedback is provided “AS IS” and You make no warranties whatsoever about any Feedback. d. Disclaimer Of Warranty. UNLESS REQUIRED BY APPLICABLE LAW, THE STABILITY AI MATERIALS AND ANY OUTPUT AND RESULTS THEREFROM ARE PROVIDED ON AN \\"AS IS\\" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED, INCLUDING, WITHOUT LIMITATION, ANY WARRANTIES OF TITLE, NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE. YOU ARE SOLELY RESPONSIBLE FOR DETERMINING THE APPROPRIATENESS OR LAWFULNESS OF USING OR REDISTRIBUTING THE STABILITY AI MATERIALS, DERIVATIVE WORKS OR ANY OUTPUT OR RESULTS AND ASSUME ANY RISKS ASSOCIATED WITH YOUR USE OF THE STABILITY AI MATERIALS, DERIVATIVE WORKS AND ANY OUTPUT AND RESULTS. e. Limitation Of Liability. IN NO EVENT WILL STABILITY AI OR ITS AFFILIATES BE LIABLE UNDER ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, TORT, NEGLIGENCE, PRODUCTS LIABILITY, OR OTHERWISE, ARISING OUT OF THIS AGREEMENT, FOR ANY LOST PROFITS OR ANY DIRECT, INDIRECT, SPECIAL, CONSEQUENTIAL, INCIDENTAL, EXEMPLARY OR PUNITIVE DAMAGES, EVEN IF STABILITY AI OR ITS AFFILIATES HAVE BEEN ADVISED OF THE POSSIBILITY OF ANY OF THE FOREGOING. f. Term And Termination. The term of this Agreement will commence upon Your acceptance of this Agreement or access to the Stability AI Materials and will continue in full force and effect until terminated in accordance with the terms and conditions herein. Stability AI may terminate this Agreement if You are in breach of any term or condition of this Agreement. Upon termination of this Agreement, You shall delete and cease use of any Stability AI Materials or Derivative Works. Section IV(d), (e), and (g) shall survive the termination of this Agreement. g. Governing Law. This Agreement will be governed by and constructed in accordance with the laws of the United States and the State of California without regard to choice of law principles, and the UN Convention on Contracts for International Sale of Goods does not apply to this Agreement.\\n\\nDEFINITIONS\\n“Affiliate(s)” means any entity that directly or indirectly controls, is controlled by, or is under common control with the subject entity; for purposes of this definition, “control” means direct or indirect ownership or control of more than 50% of the voting interests of the subject entity.\\n\\n\\"Agreement\\" means this Stability AI Community License Agreement.\\n\\n“AUP” means the Stability AI Acceptable Use Policy available at (https://stability.ai/use-policy), as may be updated from time to time.\\n\\n\\"Derivative Work(s)” means (a) any derivative work of the Stability AI Materials as recognized by U.S. copyright laws and (b) any modifications to a Model, and any other model created which is based on or derived from the Model or the Model’s output, including “fine tune” and “low-rank adaptation” models derived from a Model or a Model’s output, but do not include the output of any Model.\\n\\n“Documentation” means any specifications, manuals, documentation, and other written information provided by Stability AI related to the Software or Models.\\n\\n“Model(s)\\" means, collectively, Stability AI’s proprietary models and algorithms, including machine-learning models, trained model weights and other elements of the foregoing listed on Stability’s Core Models Webpage available at (https://stability.ai/core-models), as may be updated from time to time.\\n\\n\\"Stability AI\\" or \\"we\\" means Stability AI Ltd. and its Affiliates.\\n\\n\\"Software\\" means Stability AI’s proprietary software made available under this Agreement now or in the future.\\n\\n“Stability AI Materials” means, collectively, Stability’s proprietary Models, Software and Documentation (and any portion or combination thereof) made available under this Agreement.\\n\\n“Trade Control Laws” means any applicable U.S. and non-U.S. export control and trade sanctions laws and regulations.", "lib/splat_restorer/own_splat.py": "\\"\\"\\"Test a restorer on one of our own trained splats.\\n\\nInputs: the COLMAP dataset the splat was trained on (``images/`` +\\n``sparse/<model>/``), the exported PLY and the list of held-out frames the\\ntrainer never saw. Two questions:\\n\\n1. ``evaluate_holdout``: on held-out photos, is the restored render closer to\\n   the real photo than the raw render (PSNR / SSIM / LPIPS)?\\n2. ``distill_back``: if the splat is fine-tuned with restored renders of new\\n   viewpoints (Difix3D-style pseudo views), do the held-out frames improve,\\n   compared with fine-tuning on the real photos alone?\\n\\n``prepare_dataset`` reproduces the importer the splat was trained with\\n(undistort to PINHOLE with alpha=0, resize to the long edge, frames in sorted\\nname order), so frame indices and held-out photos match the training run.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image, ImageDraw\\n\\nfrom .colmap_io import read_sparse_model\\nfrom .distill import restore_full\\nfrom .gs_trainer import TrainConfig, View, params_from_state, params_to_cpu_state, render, ssim, train_gaussians\\nfrom .splits import nearest_reference, subsample\\n\\n# ----------------------------------------------------------------------------- data\\n\\n\\n@dataclass\\nclass SplatData:\\n    frames_dir: Path\\n    frames: list[str]  # frame file names, index = frame id\\n    names: list[str]  # original image names\\n    width: int\\n    height: int\\n    K: np.ndarray  # (3, 3)\\n    w2c: np.ndarray  # (N, 4, 4)\\n\\n    @property\\n    def num_frames(self) -> int:\\n        return len(self.frames)\\n\\n    @property\\n    def centers(self) -> np.ndarray:\\n        R, t = self.w2c[:, :3, :3], self.w2c[:, :3, 3]\\n        return -np.einsum(\\"nji,nj->ni\\", R, t)\\n\\n    @property\\n    def forwards(self) -> np.ndarray:\\n        return self.w2c[:, 2, :3]\\n\\n    def photo_u8(self, i: int) -> torch.Tensor:\\n        return torch.from_numpy(np.asarray(Image.open(self.frames_dir / self.frames[i]).convert(\\"RGB\\")).copy())\\n\\n    def photo(self, i: int, device) -> torch.Tensor:\\n        return self.photo_u8(i).to(device).float() / 255.0\\n\\n\\ndef _k_and_dist(cam) -> tuple[np.ndarray, np.ndarray]:\\n    p = cam.params\\n    if cam.model == \\"SIMPLE_PINHOLE\\":\\n        fx = fy = p[0]; cx, cy = p[1:3]; dist = [0.0] * 4\\n    elif cam.model == \\"SIMPLE_RADIAL\\":\\n        fx = fy = p[0]; cx, cy = p[1:3]; dist = [p[3], 0.0, 0.0, 0.0]\\n    elif cam.model == \\"PINHOLE\\":\\n        fx, fy, cx, cy = p[:4]; dist = [0.0] * 4\\n    elif cam.model == \\"OPENCV\\":\\n        fx, fy, cx, cy = p[:4]; dist = list(p[4:8])\\n    else:\\n        raise ValueError(f\\"Unsupported camera model {cam.model} (fisheye/360 not handled)\\")\\n    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1.0]], dtype=np.float64)\\n    return K, np.array(dist, dtype=np.float64)\\n\\n\\ndef prepare_dataset(source: Path, out: Path, long_edge: int = 1920, model_name: str = \\"0\\",\\n                    workers: int = 8, log=print) -> SplatData:\\n    \\"\\"\\"Undistorted, resized frames + cameras, like the training run\'s importer. Resumable.\\"\\"\\"\\n    import cv2\\n\\n    source, out = Path(source), Path(out)\\n    meta_path = out / \\"cameras.json\\"\\n    if meta_path.exists():\\n        return load_prepared(out)\\n    model = read_sparse_model(source / \\"sparse\\" / model_name)\\n    if len(model.cameras) != 1:\\n        raise ValueError(\\"Only single-camera datasets are supported.\\")\\n    cam = next(iter(model.cameras.values()))\\n    K, dist = _k_and_dist(cam)\\n    w, h = cam.width, cam.height\\n    s = min(1.0, long_edge / max(w, h))\\n    ow, oh = max(1, round(w * s)), max(1, round(h * s))\\n    if np.any(dist):\\n        newK, _ = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 0, (ow, oh))\\n    else:\\n        newK = K.copy(); newK[0] *= ow / w; newK[1] *= oh / h\\n    mx, my = cv2.initUndistortRectifyMap(K, dist, None, newK, (ow, oh), cv2.CV_32FC1)\\n\\n    images = sorted(model.images.values(), key=lambda im: im.name)\\n    frames_dir = out / \\"frames\\"\\n    frames_dir.mkdir(parents=True, exist_ok=True)\\n\\n    def convert(item):\\n        i, im = item\\n        dest = frames_dir / f\\"frame_{i:06d}.png\\"\\n        if dest.exists():\\n            return\\n        img = cv2.imread(str(source / \\"images\\" / im.name))\\n        if img is None or img.shape[:2] != (h, w):\\n            raise ValueError(f\\"Bad image {im.name}\\")\\n        tmp = frames_dir / f\\"frame_{i:06d}.tmp.png\\"\\n        cv2.imwrite(str(tmp), cv2.remap(img, mx, my, cv2.INTER_LINEAR), [cv2.IMWRITE_PNG_COMPRESSION, 1])\\n        tmp.replace(dest)\\n\\n    log(f\\"[data] {len(images)} images {w}x{h} {cam.model} -> {ow}x{oh} PINHOLE\\")\\n    with ThreadPoolExecutor(workers) as pool:\\n        for n, _ in enumerate(pool.map(convert, enumerate(images)), 1):\\n            if n % 100 == 0 or n == len(images):\\n                log(f\\"[data] {n}/{len(images)}\\")\\n    meta = {\\n        \\"width\\": ow, \\"height\\": oh, \\"K\\": newK.tolist(),\\n        \\"names\\": [im.name for im in images],\\n        \\"frames\\": [f\\"frame_{i:06d}.png\\" for i in range(len(images))],\\n        \\"w2c\\": [im.world_to_camera().tolist() for im in images],\\n    }\\n    meta_path.write_text(json.dumps(meta), encoding=\\"utf-8\\")\\n    return load_prepared(out)\\n\\n\\ndef load_prepared(out: Path) -> SplatData:\\n    meta = json.loads((Path(out) / \\"cameras.json\\").read_text(encoding=\\"utf-8\\"))\\n    return SplatData(Path(out) / \\"frames\\", meta[\\"frames\\"], meta[\\"names\\"], meta[\\"width\\"], meta[\\"height\\"],\\n                     np.array(meta[\\"K\\"]), np.array(meta[\\"w2c\\"]))\\n\\n\\n_PLY_TYPES = {\\"float\\": \\"<f4\\", \\"float32\\": \\"<f4\\", \\"double\\": \\"<f8\\", \\"float64\\": \\"<f8\\", \\"uchar\\": \\"u1\\", \\"uint8\\": \\"u1\\",\\n              \\"char\\": \\"i1\\", \\"int8\\": \\"i1\\", \\"short\\": \\"<i2\\", \\"ushort\\": \\"<u2\\", \\"int\\": \\"<i4\\", \\"int32\\": \\"<i4\\",\\n              \\"uint\\": \\"<u4\\", \\"uint32\\": \\"<u4\\"}\\n\\n\\ndef load_ply(path: Path, device) -> tuple[torch.nn.ParameterDict, int]:\\n    \\"\\"\\"Standard 3DGS PLY (log scales, logit opacity, SH) -> gs_trainer params, sh_degree.\\"\\"\\"\\n    with open(path, \\"rb\\") as f:\\n        props, n, in_vertex, elements = [], 0, False, []\\n        while True:\\n            line = f.readline().decode(\\"ascii\\").strip()\\n            if line.startswith(\\"format\\"):\\n                assert \\"binary_little_endian\\" in line, f\\"Unsupported PLY format: {line}\\"\\n            elif line.startswith(\\"element\\"):\\n                _, name, count = line.split()\\n                elements.append(name)\\n                in_vertex = name == \\"vertex\\"\\n                if in_vertex:\\n                    n = int(count)\\n            elif line.startswith(\\"property\\") and in_vertex:\\n                _, typ, name = line.split()\\n                props.append((name, _PLY_TYPES[typ]))\\n            elif line == \\"end_header\\":\\n                break\\n        assert elements and elements[0] == \\"vertex\\", f\\"Expected vertex data first, got {elements}\\"\\n        arr = np.fromfile(f, dtype=np.dtype(props), count=n)\\n    names = arr.dtype.names\\n    cols = lambda prefix, k: np.stack([arr[f\\"{prefix}{i}\\"] for i in range(k)], axis=1).astype(np.float32)\\n    n_rest = sum(1 for p in names if p.startswith(\\"f_rest_\\"))\\n    k = n_rest // 3\\n    sh_degree = int(round(math.sqrt(k + 1))) - 1\\n    shN = cols(\\"f_rest_\\", n_rest).reshape(n, 3, k).transpose(0, 2, 1) if k else np.zeros((n, 0, 3), np.float32)\\n    t = lambda a: torch.nn.Parameter(torch.as_tensor(np.ascontiguousarray(a), dtype=torch.float32, device=device))\\n    params = torch.nn.ParameterDict({\\n        \\"means\\": t(np.stack([arr[\\"x\\"], arr[\\"y\\"], arr[\\"z\\"]], axis=1)),\\n        \\"scales\\": t(cols(\\"scale_\\", 3)),\\n        \\"quats\\": t(cols(\\"rot_\\", 4)),\\n        \\"opacities\\": t(arr[\\"opacity\\"].astype(np.float32)),\\n        \\"sh0\\": t(cols(\\"f_dc_\\", 3)[:, None, :]),\\n        \\"shN\\": t(shN),\\n    })\\n    return params, sh_degree\\n\\n\\n# ----------------------------------------------------------------------------- metrics\\n\\n\\ndef _psnr(a: torch.Tensor, b: torch.Tensor) -> float:\\n    return 10 * math.log10(1.0 / max(F.mse_loss(a, b).item(), 1e-12))\\n\\n\\ndef _scores(img: torch.Tensor, gt: torch.Tensor, net_lpips) -> dict:\\n    \\"\\"\\"img, gt: (H, W, 3) in [0, 1] on the same device.\\"\\"\\"\\n    return {\\n        \\"psnr\\": _psnr(img, gt),\\n        \\"ssim\\": ssim(img, gt).item(),\\n        \\"lpips\\": net_lpips(img.permute(2, 0, 1)[None] * 2 - 1, gt.permute(2, 0, 1)[None] * 2 - 1).item(),\\n    }\\n\\n\\ndef _w2c_t(data: SplatData, device) -> tuple[torch.Tensor, torch.Tensor]:\\n    return (torch.tensor(data.w2c, dtype=torch.float32, device=device),\\n            torch.tensor(data.K, dtype=torch.float32, device=device))\\n\\n\\n@torch.no_grad()\\ndef score_splat(params, sh_degree: int, data: SplatData, ids: list[int], net_lpips, device=\\"cuda\\") -> dict:\\n    w2c, K = _w2c_t(data, device)\\n    rows = []\\n    for i in ids:\\n        img, _ = render(params, w2c[i], K, data.width, data.height, sh_degree)\\n        rows.append(_scores(img, data.photo(i, device), net_lpips))\\n    return {k: float(np.mean([r[k] for r in rows])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n\\n\\n@torch.no_grad()\\ndef reproduce_check(params, sh_degree: int, data: SplatData, original_metrics: dict, device=\\"cuda\\") -> list[dict]:\\n    \\"\\"\\"Re-render the frames of the training run\'s own evaluation and compare PSNR.\\"\\"\\"\\n    w2c, K = _w2c_t(data, device)\\n    index = {f: i for i, f in enumerate(data.frames)}\\n    rows = []\\n    for v in original_metrics[\\"views\\"]:\\n        i = index[v[\\"frame\\"]]\\n        img, _ = render(params, w2c[i], K, data.width, data.height, sh_degree)\\n        rows.append({\\"frame\\": v[\\"frame\\"], \\"original\\": v[\\"psnr\\"], \\"now\\": _psnr(img, data.photo(i, device))})\\n    return rows\\n\\n\\n# ----------------------------------------------------------------------------- restoration\\n\\n\\ndef make_restore_fn(model, use_ref: bool, long_edge: int = 0):\\n    \\"\\"\\"fn(deg, ref) on (1, 3, H, W) tensors padded to multiples of 64.\\n\\n    ``long_edge`` > 0 runs the model at that size and resizes back (the model\\n    was trained around 1024 px).\\"\\"\\"\\n\\n    def fn(deg, ref):\\n        h, w = deg.shape[-2:]\\n        if long_edge and max(h, w) > long_edge:\\n            s = long_edge / max(h, w)\\n            size = (max(64, round(h * s / 64) * 64), max(64, round(w * s / 64) * 64))\\n            d = F.interpolate(deg, size=size, mode=\\"area\\")\\n            r = F.interpolate(ref, size=size, mode=\\"area\\") if use_ref else None\\n            out = model.restore(d, r)\\n            return F.interpolate(out, size=(h, w), mode=\\"bicubic\\", align_corners=False).clamp(0, 1)\\n        return model.restore(deg, ref if use_ref else None)\\n\\n    return fn\\n\\n\\ndef _to_u8(img: torch.Tensor) -> np.ndarray:\\n    return (img.float().clamp(0, 1).cpu().numpy() * 255 + 0.5).astype(np.uint8)\\n\\n\\ndef _label(img: Image.Image, text: str) -> Image.Image:\\n    from PIL import ImageFont\\n\\n    try:  # Pillow >= 10.1 can size the built-in font\\n        font = ImageFont.load_default(size=max(14, img.width // 28))\\n    except TypeError:\\n        font = ImageFont.load_default()\\n    d = ImageDraw.Draw(img)\\n    pad = max(4, img.width // 120)\\n    box = d.textbbox((pad, pad), text, font=font)\\n    d.rectangle((0, 0, box[2] + pad, box[3] + pad), fill=(0, 0, 0))\\n    d.text((pad, pad), text, fill=(255, 255, 255), font=font)\\n    return img\\n\\n\\ndef _worst_box(err: np.ndarray, size: int) -> tuple[int, int]:\\n    t = torch.from_numpy(err)[None, None]\\n    step = max(1, size // 4)\\n    pooled = F.avg_pool2d(t, size, stride=step)\\n    idx = int(pooled.flatten().argmax())\\n    nx = pooled.shape[-1]\\n    return (idx % nx) * step, (idx // nx) * step\\n\\n\\ndef comparison_image(tiles: dict[str, np.ndarray], out: Path, tile_width: int = 640, box_from: str = \\"render\\") -> None:\\n    \\"\\"\\"One column per image: full frame on top, 3x zoom below where ``tiles[box_from]``\\n    is furthest from the photo (red box).\\"\\"\\"\\n    gt, rnd = tiles[\\"foto\\"].astype(np.float32), tiles[box_from].astype(np.float32)\\n    h, w = gt.shape[:2]\\n    crop = max(48, min(h, w) // 4)\\n    x, y = _worst_box(np.abs(gt - rnd).mean(-1), crop)\\n    s = tile_width / w\\n    top_h = round(h * s)\\n    canvas = Image.new(\\"RGB\\", (tile_width * len(tiles), top_h + tile_width), (20, 20, 20))\\n    for j, (name, arr) in enumerate(tiles.items()):\\n        im = Image.fromarray(arr)\\n        small = im.resize((tile_width, top_h), Image.LANCZOS)\\n        ImageDraw.Draw(small).rectangle((x * s, y * s, (x + crop) * s, (y + crop) * s), outline=(255, 0, 0), width=3)\\n        zoom = im.crop((x, y, x + crop, y + crop)).resize((tile_width, tile_width), Image.LANCZOS)\\n        canvas.paste(_label(small, name), (j * tile_width, 0))\\n        canvas.paste(_label(zoom, f\\"{name} (zoom)\\"), (j * tile_width, top_h))\\n    canvas.save(out, quality=92)\\n\\n\\ndef evaluate_holdout(\\n    params,\\n    sh_degree: int,\\n    data: SplatData,\\n    holdout: list[int],\\n    train_ids: list[int],\\n    methods: dict,\\n    out_dir: Path,\\n    net_lpips,\\n    n_images: int = 12,\\n    restore_long_edge: int = 0,\\n    device=\\"cuda\\",\\n    log=print,\\n) -> dict:\\n    \\"\\"\\"methods: name -> (loader() -> Restorer, use_ref). Scores render and each method vs the held-out photos.\\"\\"\\"\\n    out_dir = Path(out_dir)\\n    img_dir = out_dir / \\"images\\"\\n    img_dir.mkdir(parents=True, exist_ok=True)\\n    res_path = out_dir / \\"per_frame.json\\"\\n    res = json.loads(res_path.read_text(encoding=\\"utf-8\\")) if res_path.exists() else {}\\n    w2c, K = _w2c_t(data, device)\\n    centers, forwards = data.centers, data.forwards\\n\\n    with torch.no_grad():\\n        renders = {}\\n        for i in holdout:\\n            img, _ = render(params, w2c[i], K, data.width, data.height, sh_degree)\\n            renders[i] = img.half().cpu()\\n            res.setdefault(str(i), {})[\\"render\\"] = _scores(img, data.photo(i, device), net_lpips)\\n    by_render = sorted(holdout, key=lambda i: res[str(i)][\\"render\\"][\\"psnr\\"])\\n    shown = sorted(set(by_render[: n_images // 2]) | set(subsample(holdout, n_images - n_images // 2)))\\n    log(f\\"[eval] {len(holdout)} held-out frames; render PSNR \\"\\n        f\\"{np.mean([res[str(i)][\'render\'][\'psnr\'] for i in holdout]):.2f}\\")\\n    refs = {i: nearest_reference(i, train_ids, centers, forwards) for i in holdout}\\n\\n    for name, (loader, use_ref) in methods.items():\\n        todo = [i for i in holdout if name not in res[str(i)] or (i in shown and not (img_dir / f\\"{name}_{i:06d}.jpg\\").exists())]\\n        if not todo:\\n            continue\\n        model = loader()\\n        fn = make_restore_fn(model, use_ref, restore_long_edge)\\n        with torch.no_grad():\\n            for n, i in enumerate(todo, 1):\\n                deg = renders[i].to(device).float()\\n                ref = data.photo(refs[i], device)\\n                out = restore_full(fn, deg, ref)\\n                res[str(i)][name] = _scores(out, data.photo(i, device), net_lpips)\\n                if i in shown:\\n                    Image.fromarray(_to_u8(out)).save(img_dir / f\\"{name}_{i:06d}.jpg\\", quality=95)\\n                if n % 20 == 0 or n == len(todo):\\n                    log(f\\"[eval] {name}: {n}/{len(todo)}\\")\\n        res_path.write_text(json.dumps(res), encoding=\\"utf-8\\")\\n        del model, fn\\n        torch.cuda.empty_cache()\\n    res_path.write_text(json.dumps(res), encoding=\\"utf-8\\")\\n\\n    for i in shown:\\n        tiles = {\\"foto\\": _to_u8(data.photo(i, \\"cpu\\")), \\"render\\": _to_u8(renders[i])}\\n        for name in methods:\\n            p = img_dir / f\\"{name}_{i:06d}.jpg\\"\\n            if p.exists():\\n                tiles[name] = np.asarray(Image.open(p).convert(\\"RGB\\"))\\n        tiles[\\"referans\\"] = _to_u8(data.photo(refs[i], \\"cpu\\"))\\n        comparison_image(tiles, out_dir / f\\"compare_{i:06d}.jpg\\")\\n\\n    summary = summarise(res, holdout, [\\"render\\", *methods])\\n    summary[\\"shown_frames\\"] = shown\\n    (out_dir / \\"summary.json\\").write_text(json.dumps(summary, indent=2), encoding=\\"utf-8\\")\\n    md = eval_markdown(summary)\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return summary\\n\\n\\ndef summarise(res: dict, ids: list[int], names: list[str]) -> dict:\\n    out = {\\"frames\\": len(ids), \\"methods\\": {}}\\n    for name in names:\\n        rows = [res[str(i)][name] for i in ids if name in res[str(i)]]\\n        if not rows:\\n            continue\\n        base = [res[str(i)][\\"render\\"] for i in ids if name in res[str(i)]]\\n        out[\\"methods\\"][name] = {\\n            \\"psnr\\": float(np.mean([r[\\"psnr\\"] for r in rows])),\\n            \\"ssim\\": float(np.mean([r[\\"ssim\\"] for r in rows])),\\n            \\"lpips\\": float(np.mean([r[\\"lpips\\"] for r in rows])),\\n            \\"lpips_wins\\": int(sum(r[\\"lpips\\"] < b[\\"lpips\\"] for r, b in zip(rows, base))),\\n            \\"psnr_wins\\": int(sum(r[\\"psnr\\"] > b[\\"psnr\\"] for r, b in zip(rows, base))),\\n            \\"n\\": len(rows),\\n        }\\n    worst = sorted(ids, key=lambda i: res[str(i)][\\"render\\"][\\"psnr\\"])[:8]\\n    out[\\"worst\\"] = [{\\"frame\\": i, **{n: res[str(i)][n] for n in names if n in res[str(i)]}} for i in worst]\\n    return out\\n\\n\\ndef eval_markdown(s: dict) -> str:\\n    r = s[\\"methods\\"][\\"render\\"]\\n    lines = [\\n        f\\"## Saklanan kareler: render vs Difix ({s[\'frames\']} kare, splat bu fotoğrafları hiç görmedi)\\",\\n        \\"\\",\\n        \\"| yöntem | PSNR ↑ | Δ | SSIM ↑ | LPIPS ↓ | Δ | LPIPS\'i iyileşen kare | PSNR\'ı iyileşen kare |\\",\\n        \\"|---|---|---|---|---|---|---|---|\\",\\n    ]\\n    for name, m in s[\\"methods\\"].items():\\n        lines.append(\\n            f\\"| {name} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - r[\'psnr\']:+.2f} | {m[\'ssim\']:.4f} | {m[\'lpips\']:.4f} | \\"\\n            f\\"{m[\'lpips\'] - r[\'lpips\']:+.4f} | {m[\'lpips_wins\']}/{m[\'n\']} | {m[\'psnr_wins\']}/{m[\'n\']} |\\"\\n        )\\n    names = [n for n in s[\\"methods\\"] if n != \\"render\\"]\\n    lines += [\\"\\", \\"**En kötü 8 kare (render PSNR\'a göre), PSNR / LPIPS:**\\", \\"\\",\\n              \\"| kare | render | \\" + \\" | \\".join(names) + \\" |\\", \\"|---|---\\" + \\"|---\\" * len(names) + \\"|\\"]\\n    for w in s[\\"worst\\"]:\\n        cells = [f\\"{w[\'render\'][\'psnr\']:.2f} / {w[\'render\'][\'lpips\']:.3f}\\"]\\n        cells += [f\\"{w[n][\'psnr\']:.2f} / {w[n][\'lpips\']:.3f}\\" if n in w else \\"-\\" for n in names]\\n        lines.append(f\\"| {w[\'frame\']} | \\" + \\" | \\".join(cells) + \\" |\\")\\n    lines += [\\"\\", \\"Okuma: LPIPS (algısal fark) düşüyorsa ve PSNR belirgin düşmüyorsa düzeltme gerçek. \\"\\n              \\"PSNR düşüp LPIPS iyileşiyorsa model keskin ama uydurma detay ekliyor olabilir; zoom\'lara bak.\\"]\\n    return \\"\\\\n\\".join(lines)\\n\\n\\n# ----------------------------------------------------------------------------- distill back\\n\\n\\ndef pseudo_poses(data: SplatData, train_ids: list[int], n: int, shift_frac: float) -> list[tuple[int, np.ndarray]]:\\n    \\"\\"\\"New viewpoints: training cameras moved sideways / up by ``shift_frac`` of the camera path size.\\n\\n    Returns (source training frame, w2c) pairs; the source photo is the restorer\'s reference.\\"\\"\\"\\n    centers = data.centers\\n    extent = float(np.linalg.norm(centers[train_ids].max(0) - centers[train_ids].min(0)))\\n    offset = shift_frac * extent\\n    dirs = [(0, 1.0), (0, -1.0), (1, 1.0), (1, -1.0)]  # camera right / left / down / up\\n    out = []\\n    for j, i in enumerate(subsample(train_ids, n)):\\n        axis, sign = dirs[j % len(dirs)]\\n        w2c = data.w2c[i].copy()\\n        R = w2c[:3, :3]\\n        c = centers[i] + sign * offset * R[axis]\\n        w2c[:3, 3] = -R @ c\\n        out.append((i, w2c))\\n    return out\\n\\n\\ndef distill_back(\\n    params,\\n    sh_degree: int,\\n    data: SplatData,\\n    holdout: list[int],\\n    train_ids: list[int],\\n    methods: dict,\\n    out_dir: Path,\\n    net_lpips,\\n    steps: int = 3000,\\n    n_pseudo: int = 120,\\n    shift_frac: float = 0.03,\\n    pseudo_share: float = 0.3,\\n    train_stride: int = 2,\\n    restore_long_edge: int = 0,\\n    device=\\"cuda\\",\\n    log=print,\\n) -> dict:\\n    \\"\\"\\"Fine-tune the splat: real photos only (control) vs photos + restored pseudo views.\\"\\"\\"\\n    out_dir = Path(out_dir)\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    res_path = out_dir / \\"results.json\\"\\n    results = json.loads(res_path.read_text(encoding=\\"utf-8\\")) if res_path.exists() else {}\\n    start_state = params_to_cpu_state(params)\\n    w2c_t, K = _w2c_t(data, device)\\n    scene_scale = 1.1 * float(np.linalg.norm(data.centers[train_ids] - data.centers[train_ids].mean(0), axis=1).max())\\n    cfg = TrainConfig(steps=steps, sh_degree=sh_degree, densify=False, lr_means=1.6e-5, lr_means_final_frac=0.1, seed=0)\\n    score = lambda p: score_splat(p, sh_degree, data, holdout, net_lpips, device)\\n\\n    if \\"start\\" not in results:\\n        results[\\"start\\"] = score(params)\\n        res_path.write_text(json.dumps(results, indent=2), encoding=\\"utf-8\\")\\n    ft_ids = train_ids[::train_stride]\\n    log(f\\"[distill] loading {len(ft_ids)} training photos (every {train_stride}.)\\")\\n    train_views = [View(data.photo_u8(i), w2c_t[i], K) for i in ft_ids]\\n\\n    if \\"control\\" not in results:\\n        p = train_gaussians(train_views, data.width, data.height, cfg, scene_scale=scene_scale,\\n                            params=params_from_state(start_state, device), log=log)\\n        results[\\"control\\"] = score(p)\\n        del p\\n        torch.cuda.empty_cache()\\n        res_path.write_text(json.dumps(results, indent=2), encoding=\\"utf-8\\")\\n        log(f\\"[distill] control: {results[\'control\']}\\")\\n\\n    poses = pseudo_poses(data, train_ids, n_pseudo, shift_frac)\\n    base = params_from_state(start_state, device)\\n    with torch.no_grad():\\n        renders = [render(base, torch.tensor(w, dtype=torch.float32, device=device), K,\\n                          data.width, data.height, sh_degree)[0].half().cpu() for _, w in poses]\\n    del base\\n    for name, (loader, use_ref) in methods.items():\\n        if name in results:\\n            continue\\n        model = loader()\\n        fn = make_restore_fn(model, use_ref, restore_long_edge)\\n        pseudo = []\\n        with torch.no_grad():\\n            for (src, w), rnd in zip(poses, renders):\\n                out = restore_full(fn, rnd.to(device).float(), data.photo(src, device))\\n                pseudo.append(torch.from_numpy(_to_u8(out)))\\n        del model, fn\\n        torch.cuda.empty_cache()\\n        ex_dir = out_dir / f\\"pseudo_{name}\\"\\n        ex_dir.mkdir(exist_ok=True)\\n        for k in subsample(list(range(len(poses))), 6):\\n            tiles = {\\"render (yeni kamera)\\": _to_u8(renders[k]), name: pseudo[k].numpy(),\\n                     \\"kaynak foto\\": _to_u8(data.photo(poses[k][0], \\"cpu\\"))}\\n            tw = 800\\n            row = [_label(Image.fromarray(a).resize((tw, round(tw * data.height / data.width)), Image.LANCZOS), t)\\n                   for t, a in tiles.items()]\\n            canvas = Image.new(\\"RGB\\", (tw * len(row), row[0].height))\\n            for j, im in enumerate(row):\\n                canvas.paste(im, (tw * j, 0))\\n            canvas.save(ex_dir / f\\"pseudo_{k:03d}.jpg\\", quality=90)\\n        w_pseudo = pseudo_share / (1 - pseudo_share) * len(train_views) / max(len(pseudo), 1)\\n        views = train_views + [View(img, torch.tensor(w, dtype=torch.float32, device=device), K, weight=w_pseudo)\\n                               for (_, w), img in zip(poses, pseudo)]\\n        p = train_gaussians(views, data.width, data.height, cfg, scene_scale=scene_scale,\\n                            params=params_from_state(start_state, device), log=log)\\n        results[name] = score(p)\\n        del p, pseudo, views\\n        torch.cuda.empty_cache()\\n        res_path.write_text(json.dumps(results, indent=2), encoding=\\"utf-8\\")\\n        log(f\\"[distill] {name}: {results[name]}\\")\\n\\n    meta = {\\"steps\\": steps, \\"n_pseudo\\": n_pseudo, \\"shift_frac\\": shift_frac, \\"pseudo_share\\": pseudo_share,\\n            \\"train_photos\\": len(ft_ids), \\"holdout\\": len(holdout)}\\n    md = distill_markdown(results, meta)\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return results\\n\\n\\ndef distill_markdown(results: dict, meta: dict) -> str:\\n    c = results.get(\\"control\\", results[\\"start\\"])\\n    lines = [\\n        f\\"## Splat\'e geri besleme ({meta[\'steps\']} adım ince ayar, {meta[\'n_pseudo\']} yeni bakış, \\"\\n        f\\"kaydırma %{meta[\'shift_frac\'] * 100:.0f})\\",\\n        \\"\\",\\n        f\\"Ölçüm: {meta[\'holdout\']} saklanan kare (hiçbir koşulda eğitimde kullanılmadı).\\",\\n        \\"\\",\\n        \\"| koşul | PSNR ↑ | Δ control | SSIM ↑ | LPIPS ↓ | Δ control |\\",\\n        \\"|---|---|---|---|---|---|\\",\\n    ]\\n    for name, m in results.items():\\n        lines.append(f\\"| {name} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - c[\'psnr\']:+.2f} | {m[\'ssim\']:.4f} | \\"\\n                     f\\"{m[\'lpips\']:.4f} | {m[\'lpips\'] - c[\'lpips\']:+.4f} |\\")\\n    lines += [\\"\\", \\"* `start`: mevcut splat. `control`: aynı ince ayar, sadece gerçek fotoğraflar.\\",\\n              \\"* Difix koşulu `control`\'ü geçiyorsa düzeltilmiş yeni bakışlar splat\'in kendisini iyileştirmiş demektir.\\"]\\n    return \\"\\\\n\\".join(lines)\\n", "lib/splat_restorer/pairs.py": "\\"\\"\\"Generate (degraded render, real photo, reference photo) pairs for one scene.\\n\\nFor each recipe a deliberately weak splat is trained on a sparse subset of the\\nframes; every non-training frame is rendered and saved next to its real photo.\\nThe reference is the real photo of the closest training camera.\\n\\nOutput is one tar per scene so Drive sees a single large atomic file:\\n\\n    index.json\\n    <recipe>/deg/<frame>.jpg   render of the weak splat\\n    <recipe>/gt/<frame>.jpg    real photo at the same pose\\n    <recipe>/ref/<frame>.jpg   real photo of a training camera (shared)\\n    eval/<frame>.jpg           (test scenes) never-trained frames for distill-back\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport shutil\\nimport tarfile\\nimport time\\nimport zlib\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom PIL import Image\\n\\nfrom .gs_trainer import TrainConfig, View, params_to_cpu_state, psnr, render, train_gaussians\\nfrom .scene import Scene, load_colmap_scene\\nfrom .splits import RECIPES, make_split, nearest_reference, subsample\\nfrom .workspace import Workspace, atomic_copy\\n\\nJPEG_QUALITY = 95\\n\\n\\ndef _save_jpg(arr: np.ndarray, path: Path) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    Image.fromarray((np.clip(arr, 0, 1) * 255 + 0.5).astype(np.uint8)).save(path, quality=JPEG_QUALITY)\\n\\n\\ndef _fname(i: int) -> str:\\n    return f\\"{i:05d}.jpg\\"\\n\\n\\ndef generate_scene_pairs(\\n    ws: Workspace,\\n    scene_name: str,\\n    scene_dir: Path,\\n    recipes: list[str],\\n    is_test: bool,\\n    max_pairs_per_recipe: int = 100,\\n    images_subdir: str = \\"images_4\\",\\n    device: str = \\"cuda\\",\\n    log=print,\\n) -> Path:\\n    t0 = time.time()\\n    scene = load_colmap_scene(scene_dir, images_subdir=images_subdir, name=scene_name)\\n    log(f\\"  {scene_name[:12]}: {scene.num_frames} frames {scene.width}x{scene.height}, {len(scene.points_xyz)} points\\")\\n    # uint8 keeps a 300-frame 960p scene at ~0.5 GB of RAM.\\n    photos_u8 = [(scene.load_image(i) * 255 + 0.5).astype(np.uint8) for i in range(scene.num_frames)]\\n    photo = lambda i: photos_u8[i].astype(np.float32) / 255.0\\n    centers, forwards = scene.camera_centers(), scene.view_dirs()\\n    Ks = torch.as_tensor(scene.Ks, device=device)\\n    w2c = torch.as_tensor(scene.w2c, device=device)\\n\\n    work = ws.local / \\"pairs_tmp\\" / scene_name\\n    shutil.rmtree(work, ignore_errors=True)\\n    work.mkdir(parents=True)\\n    index = {\\n        \\"scene\\": scene_name,\\n        \\"width\\": scene.width,\\n        \\"height\\": scene.height,\\n        \\"is_test\\": is_test,\\n        \\"cameras\\": {\\"K\\": scene.Ks.tolist(), \\"w2c\\": scene.w2c.tolist()},\\n        \\"recipes\\": {},\\n    }\\n\\n    for rname in recipes:\\n        recipe = RECIPES[rname]\\n        split = make_split(scene.num_frames, recipe)\\n        views = [View(torch.as_tensor(photo(i), device=device), w2c[i], Ks[i]) for i in split.train]\\n        cfg = TrainConfig(steps=recipe.steps, seed=zlib.crc32(scene_name.encode()) % 10_000)\\n        t1 = time.time()\\n        params = train_gaussians(\\n            views, scene.width, scene.height, cfg, init_xyz=scene.points_xyz, init_rgb=scene.points_rgb\\n        )\\n        train_s = time.time() - t1\\n\\n        pair_frames = subsample(split.pair, max_pairs_per_recipe)\\n        items = []\\n        with torch.no_grad():\\n            for i in pair_frames:\\n                img, _ = render(params, w2c[i], Ks[i], scene.width, scene.height, cfg.sh_degree)\\n                deg = img.float().cpu().numpy()\\n                ref = nearest_reference(i, split.train, centers, forwards)\\n                _save_jpg(deg, work / rname / \\"deg\\" / _fname(i))\\n                _save_jpg(photo(i), work / rname / \\"gt\\" / _fname(i))\\n                ref_path = work / rname / \\"ref\\" / _fname(ref)\\n                if not ref_path.exists():\\n                    _save_jpg(photo(ref), ref_path)\\n                items.append(\\n                    {\\n                        \\"frame\\": i,\\n                        \\"ref\\": ref,\\n                        \\"deg\\": f\\"{rname}/deg/{_fname(i)}\\",\\n                        \\"gt\\": f\\"{rname}/gt/{_fname(i)}\\",\\n                        \\"ref_path\\": f\\"{rname}/ref/{_fname(ref)}\\",\\n                        \\"psnr\\": round(psnr(img, torch.as_tensor(photo(i), device=device)), 3),\\n                    }\\n                )\\n        if is_test:\\n            # Distill-back needs every training photo and the splat itself.\\n            for i in split.train:\\n                p = work / rname / \\"ref\\" / _fname(i)\\n                if not p.exists():\\n                    _save_jpg(photo(i), p)\\n            out = ws.splat_path(scene_name, rname)\\n            out.parent.mkdir(parents=True, exist_ok=True)\\n            tmp = ws.local / \\"splat_tmp.pt\\"\\n            torch.save(params_to_cpu_state(params), tmp)\\n            atomic_copy(tmp, out)\\n            tmp.unlink(missing_ok=True)\\n\\n        mean_psnr = float(np.mean([it[\\"psnr\\"] for it in items])) if items else float(\\"nan\\")\\n        index[\\"recipes\\"][rname] = {\\n            \\"train\\": split.train,\\n            \\"pair\\": split.pair,\\n            \\"eval\\": split.eval,\\n            \\"steps\\": recipe.steps,\\n            \\"num_gaussians\\": int(len(params[\\"means\\"])),\\n            \\"train_seconds\\": round(train_s, 1),\\n            \\"mean_psnr\\": round(mean_psnr, 3),\\n            \\"items\\": items,\\n        }\\n        log(\\n            f\\"    {rname}: train {len(split.train)} views, {len(params[\'means\'])} gaussians, \\"\\n            f\\"{train_s:.0f}s, {len(items)} pairs, render PSNR {mean_psnr:.2f}\\"\\n        )\\n        del params, views\\n        torch.cuda.empty_cache()\\n\\n    if is_test:\\n        ev = make_split(scene.num_frames, RECIPES[recipes[0]]).eval\\n        for i in ev:\\n            _save_jpg(photo(i), work / \\"eval\\" / _fname(i))\\n\\n    (work / \\"index.json\\").write_text(json.dumps(index), encoding=\\"utf-8\\")\\n    local_tar = ws.local / f\\"{scene_name}.tar\\"\\n    with tarfile.open(local_tar, \\"w\\") as tar:\\n        tar.add(work, arcname=scene_name)\\n    atomic_copy(local_tar, ws.pair_tar(scene_name))\\n    local_tar.unlink(missing_ok=True)\\n    shutil.rmtree(work, ignore_errors=True)\\n    log(f\\"  {scene_name[:12]}: done in {time.time() - t0:.0f}s\\")\\n    return ws.pair_tar(scene_name)\\n\\n\\ndef extract_pairs(ws: Workspace, scenes: list[str]) -> Path:\\n    \\"\\"\\"Extract the tars of ``scenes`` to local disk (skips already extracted ones).\\"\\"\\"\\n    out = ws.local_pairs\\n    out.mkdir(parents=True, exist_ok=True)\\n    for s in scenes:\\n        if (out / s / \\"index.json\\").exists():\\n            continue\\n        tar_path = ws.pair_tar(s)\\n        if not tar_path.exists():\\n            raise FileNotFoundError(f\\"Pairs for scene {s} are missing: {tar_path}\\")\\n        with tarfile.open(tar_path) as tar:\\n            tar.extractall(out)\\n    return out\\n\\n\\ndef load_index(pairs_root: Path, scene: str) -> dict:\\n    return json.loads((pairs_root / scene / \\"index.json\\").read_text(encoding=\\"utf-8\\"))\\n", "lib/splat_restorer/restorer_model.py": "\\"\\"\\"Single-step diffusion restorer (Difix3D architecture).\\n\\nAdapted from NVIDIA Difix3D (https://github.com/nv-tlabs/Difix3D, src/model.py\\nand src/mv_unet.py) under the NVIDIA License in LICENSE-DIFIX3D.txt, which\\nallows non-commercial (research / evaluation) use only.\\n\\nChanges from the original:\\n* works with current diffusers / peft (no pinned 0.25 UNet copy): the\\n  reference-view attention is an attention *processor*, not a forked UNet;\\n* the text prompt is fixed and encoded once;\\n* x0 is computed directly from the scheduler\'s alphas (epsilon or v);\\n* checkpoints contain the full UNet so a saved model is self-contained.\\n\\nHow it works: the degraded image is VAE-encoded and treated as a latent at\\ntimestep ``t`` (default 199). The SD-Turbo UNet predicts the \\"noise\\", x0 is\\nrecovered in one step and decoded. VAE encoder activations are fed to the\\ndecoder through 1x1 skip convs so fine detail survives the round trip. With a\\nreference view, self-attention runs jointly over both views\' tokens.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport torch\\nimport torch.nn.functional as F\\n\\nBASE_MODEL = \\"stabilityai/sd-turbo\\"\\nPROMPT = \\"remove degradation\\"\\nVAE_LORA_TARGETS = (\\n    \\"conv1\\", \\"conv2\\", \\"conv_in\\", \\"conv_shortcut\\", \\"conv\\", \\"conv_out\\",\\n    \\"skip_conv_1\\", \\"skip_conv_2\\", \\"skip_conv_3\\", \\"skip_conv_4\\",\\n    \\"to_k\\", \\"to_q\\", \\"to_v\\", \\"to_out.0\\",\\n)\\n\\n\\ndef _vae_encoder_fwd(self, sample):\\n    sample = self.conv_in(sample)\\n    skips = []\\n    for down_block in self.down_blocks:\\n        skips.append(sample)\\n        sample = down_block(sample)\\n    sample = self.mid_block(sample)\\n    sample = self.conv_norm_out(sample)\\n    sample = self.conv_act(sample)\\n    sample = self.conv_out(sample)\\n    self.current_down_blocks = skips\\n    return sample\\n\\n\\ndef _vae_decoder_fwd(self, sample, latent_embeds=None):\\n    sample = self.conv_in(sample)\\n    upscale_dtype = next(iter(self.up_blocks.parameters())).dtype\\n    sample = self.mid_block(sample, latent_embeds)\\n    sample = sample.to(upscale_dtype)\\n    skip_convs = [self.skip_conv_1, self.skip_conv_2, self.skip_conv_3, self.skip_conv_4]\\n    skips = self.incoming_skip_acts[::-1]\\n    for idx, up_block in enumerate(self.up_blocks):\\n        if not self.ignore_skip:\\n            sample = sample + skip_convs[idx](skips[idx] * self.gamma)\\n        sample = up_block(sample, latent_embeds)\\n    if latent_embeds is None:\\n        sample = self.conv_norm_out(sample)\\n    else:\\n        sample = self.conv_norm_out(sample, latent_embeds)\\n    sample = self.conv_act(sample)\\n    return self.conv_out(sample)\\n\\n\\nclass MultiViewSelfAttnProcessor:\\n    \\"\\"\\"Self-attention over the tokens of all views of a sample.\\n\\n    The UNet sees a batch of (B * V) images; for attention the V views of each\\n    sample are concatenated along the token axis, which lets the degraded view\\n    borrow detail from the reference view.\\n    \\"\\"\\"\\n\\n    def __init__(self):\\n        self.num_views = 1\\n\\n    def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None, *args, **kwargs):\\n        input_ndim = hidden_states.ndim\\n        if input_ndim == 4:\\n            b0, c, h, w = hidden_states.shape\\n            hidden_states = hidden_states.view(b0, c, h * w).transpose(1, 2)\\n        bv, n, d = hidden_states.shape\\n        v = self.num_views\\n        x = hidden_states.reshape(bv // v, v * n, d) if v > 1 else hidden_states\\n        b = x.shape[0]\\n        q, k, val = attn.to_q(x), attn.to_k(x), attn.to_v(x)\\n        heads = attn.heads\\n        hd = q.shape[-1] // heads\\n        q, k, val = (t.view(b, -1, heads, hd).transpose(1, 2) for t in (q, k, val))\\n        out = F.scaled_dot_product_attention(q, k, val)\\n        out = out.transpose(1, 2).reshape(b, -1, heads * hd).to(q.dtype)\\n        out = attn.to_out[1](attn.to_out[0](out))\\n        if v > 1:\\n            out = out.reshape(bv, n, d)\\n        if input_ndim == 4:\\n            out = out.transpose(-1, -2).reshape(b0, c, h, w)\\n        if getattr(attn, \\"residual_connection\\", False):\\n            out = out + hidden_states\\n        return out / getattr(attn, \\"rescale_output_factor\\", 1.0)\\n\\n\\nclass Restorer(torch.nn.Module):\\n    def __init__(\\n        self,\\n        timestep: int = 199,\\n        lora_rank_vae: int = 4,\\n        device: str = \\"cuda\\",\\n        pretrained: bool = True,\\n        unet_overrides: dict | None = None,\\n        base_model: str = BASE_MODEL,\\n        pretrained_text_encoder: bool | None = None,\\n    ):\\n        \\"\\"\\"``pretrained=False`` builds random weights from the configs only, and\\n        ``unet_overrides`` can shrink that random UNet (tests on small GPUs).\\n        ``base_model`` is the Hugging Face repo the parts are loaded from.\\n        ``pretrained_text_encoder`` defaults to ``pretrained``.\\"\\"\\"\\n        super().__init__()\\n        from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel\\n        from diffusers.models.attention_processor import AttnProcessor2_0\\n        from peft import LoraConfig, inject_adapter_in_model\\n        from transformers import AutoTokenizer, CLIPTextConfig, CLIPTextModel\\n\\n        def load(cls, sub, overrides=None):\\n            if pretrained:\\n                return cls.from_pretrained(base_model, subfolder=sub)\\n            config = dict(cls.load_config(base_model, subfolder=sub))\\n            config.update(overrides or {})\\n            return cls.from_config(config)\\n\\n        tokenizer = AutoTokenizer.from_pretrained(base_model, subfolder=\\"tokenizer\\")\\n        if pretrained if pretrained_text_encoder is None else pretrained_text_encoder:\\n            text_encoder = CLIPTextModel.from_pretrained(base_model, subfolder=\\"text_encoder\\")\\n        else:\\n            text_encoder = CLIPTextModel(CLIPTextConfig.from_pretrained(base_model, subfolder=\\"text_encoder\\"))\\n        text_encoder = text_encoder.to(device)\\n        with torch.no_grad():\\n            ids = tokenizer(PROMPT, max_length=tokenizer.model_max_length, padding=\\"max_length\\",\\n                            truncation=True, return_tensors=\\"pt\\").input_ids.to(device)\\n            self.register_buffer(\\"prompt_embeds\\", text_encoder(ids)[0].float(), persistent=False)\\n        del text_encoder\\n\\n        sched = DDPMScheduler.from_pretrained(base_model, subfolder=\\"scheduler\\")\\n        self.prediction_type = sched.config.prediction_type\\n        self.register_buffer(\\"alphas_cumprod\\", sched.alphas_cumprod.float(), persistent=False)\\n        self.timestep = int(timestep)\\n\\n        vae = load(AutoencoderKL, \\"vae\\")\\n        vae.encoder.forward = _vae_encoder_fwd.__get__(vae.encoder, vae.encoder.__class__)\\n        vae.decoder.forward = _vae_decoder_fwd.__get__(vae.decoder, vae.decoder.__class__)\\n        dec = vae.decoder\\n        dec.skip_conv_1 = torch.nn.Conv2d(512, 512, 1, bias=False)\\n        dec.skip_conv_2 = torch.nn.Conv2d(256, 512, 1, bias=False)\\n        dec.skip_conv_3 = torch.nn.Conv2d(128, 512, 1, bias=False)\\n        dec.skip_conv_4 = torch.nn.Conv2d(128, 256, 1, bias=False)\\n        for conv in (dec.skip_conv_1, dec.skip_conv_2, dec.skip_conv_3, dec.skip_conv_4):\\n            torch.nn.init.constant_(conv.weight, 1e-5)\\n        dec.ignore_skip = False\\n        dec.gamma = 1.0\\n        targets = [name for name, _ in dec.named_modules() if name and any(name.endswith(t) for t in VAE_LORA_TARGETS)]\\n        inject_adapter_in_model(\\n            LoraConfig(r=lora_rank_vae, init_lora_weights=\\"gaussian\\", target_modules=targets), dec, adapter_name=\\"vae_skip\\"\\n        )\\n        vae.requires_grad_(False)\\n        self.vae = vae\\n\\n        unet = load(UNet2DConditionModel, \\"unet\\", unet_overrides)\\n        self.mv_processor = MultiViewSelfAttnProcessor()\\n        unet.set_attn_processor(\\n            {name: (self.mv_processor if name.endswith(\\"attn1.processor\\") else AttnProcessor2_0()) for name in unet.attn_processors}\\n        )\\n        self.unet = unet\\n        self.to(device)\\n\\n    @classmethod\\n    def from_difix(cls, repo: str = \\"nvidia/difix\\", device: str = \\"cuda\\") -> \\"Restorer\\":\\n        \\"\\"\\"NVIDIA\'s released Difix weights (\\"nvidia/difix\\" or \\"nvidia/difix_ref\\").\\n\\n        Their VAE (skip convs + \\"vae_skip\\" LoRA) and UNet have exactly this\\n        class\'s parameter names and shapes, so they load strictly.\\n        \\"\\"\\"\\n        from huggingface_hub import hf_hub_download\\n        from safetensors.torch import load_file\\n\\n        # Build VAE/UNet from the configs only: Difix\'s VAE file stores LoRA-wrapped\\n        # layers (\\".base_layer.\\"), which a plain AutoencoderKL.from_pretrained rejects.\\n        model = cls(device=device, base_model=repo, pretrained=False, pretrained_text_encoder=True)\\n        for part, module in ((\\"vae\\", model.vae), (\\"unet\\", model.unet)):\\n            path = hf_hub_download(repo, f\\"{part}/diffusion_pytorch_model.safetensors\\")\\n            module.load_state_dict(load_file(path), strict=True)\\n        model.set_eval()\\n        return model\\n\\n    # ----- trainable parameter groups -------------------------------------------------\\n    def vae_trainable_parameters(self):\\n        dec = self.vae.decoder\\n        params = [p for n, p in dec.named_parameters() if \\"lora_\\" in n]\\n        for conv in (dec.skip_conv_1, dec.skip_conv_2, dec.skip_conv_3, dec.skip_conv_4):\\n            params += [p for n, p in conv.named_parameters() if \\"lora_\\" not in n]\\n        return params\\n\\n    def set_train(self):\\n        self.unet.train()\\n        self.vae.train()\\n        self.unet.requires_grad_(True)\\n        self.vae.requires_grad_(False)\\n        for p in self.vae_trainable_parameters():\\n            p.requires_grad_(True)\\n\\n    def set_eval(self):\\n        self.unet.eval()\\n        self.vae.eval()\\n        self.requires_grad_(False)\\n\\n    # ----- forward ---------------------------------------------------------------------\\n    def forward(self, x: torch.Tensor, sample_latent: bool = True) -> torch.Tensor:\\n        \\"\\"\\"x: (B, V, 3, H, W) in [-1, 1]; view 0 is the image to restore,\\n        view 1 (optional) the reference. Returns (B, V, 3, H, W) in [-1, 1].\\"\\"\\"\\n        b, v = x.shape[:2]\\n        flat = x.flatten(0, 1)\\n        dist = self.vae.encode(flat).latent_dist\\n        z = (dist.sample() if sample_latent else dist.mode()) * self.vae.config.scaling_factor\\n        self.mv_processor.num_views = v\\n        t = torch.full((z.shape[0],), self.timestep, device=z.device, dtype=torch.long)\\n        cond = self.prompt_embeds.expand(z.shape[0], -1, -1).to(z.dtype)\\n        pred = self.unet(z, t, encoder_hidden_states=cond).sample\\n        a = self.alphas_cumprod[self.timestep].to(z.dtype)\\n        if self.prediction_type == \\"epsilon\\":\\n            z0 = (z - (1 - a).sqrt() * pred) / a.sqrt()\\n        elif self.prediction_type == \\"v_prediction\\":\\n            z0 = a.sqrt() * z - (1 - a).sqrt() * pred\\n        else:  # \\"sample\\"\\n            z0 = pred\\n        self.vae.decoder.incoming_skip_acts = self.vae.encoder.current_down_blocks\\n        out = self.vae.decode(z0 / self.vae.config.scaling_factor).sample.clamp(-1, 1)\\n        return out.view(b, v, *out.shape[1:])\\n\\n    @torch.no_grad()\\n    def restore(self, deg: torch.Tensor, ref: torch.Tensor | None = None) -> torch.Tensor:\\n        \\"\\"\\"deg/ref: (B, 3, H, W) in [0, 1], H and W multiples of 8. Returns [0, 1].\\"\\"\\"\\n        x = deg * 2 - 1\\n        if ref is not None:\\n            x = torch.stack([x, ref * 2 - 1], dim=1)\\n        else:\\n            x = x[:, None]\\n        # bf16 where the GPU supports it (A100/H100/L4); fp32 otherwise (T4), since\\n        # the SD VAE can overflow in fp16.\\n        # (is_bf16_supported() is also True on T4, where bf16 is only emulated and slow.)\\n        bf16 = x.is_cuda and torch.cuda.get_device_capability(x.device)[0] >= 8\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16, enabled=bf16):\\n            out = self.forward(x, sample_latent=False)[:, 0]\\n        return (out.float() * 0.5 + 0.5).clamp(0, 1)\\n\\n    # ----- persistence -----------------------------------------------------------------\\n    def trainable_state_dict(self) -> dict:\\n        vae_keys = {n for n, _ in self.vae.named_parameters() if \\"lora_\\" in n or \\"skip_conv\\" in n}\\n        return {\\n            \\"unet\\": self.unet.state_dict(),\\n            \\"vae\\": {k: v for k, v in self.vae.state_dict().items() if k in vae_keys},\\n            \\"timestep\\": self.timestep,\\n        }\\n\\n    def load_trainable_state_dict(self, sd: dict) -> None:\\n        self.unet.load_state_dict(sd[\\"unet\\"])\\n        missing = self.vae.load_state_dict(sd[\\"vae\\"], strict=False)\\n        bad = [k for k in missing.unexpected_keys]\\n        if bad:\\n            raise KeyError(f\\"Unexpected VAE keys in checkpoint: {bad[:5]}\\")\\n        self.timestep = int(sd.get(\\"timestep\\", self.timestep))\\n", "lib/splat_restorer/scene.py": "\\"\\"\\"Load a COLMAP scene (DL3DV gaussian_splat layout) into GPU-ready arrays.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom PIL import Image as PILImage\\n\\nfrom .colmap_io import read_sparse_model\\n\\n\\n@dataclass\\nclass Scene:\\n    name: str\\n    image_paths: list[Path]  # sorted by file name (= capture order for DL3DV)\\n    width: int\\n    height: int\\n    Ks: np.ndarray  # (F, 3, 3) at the loaded image resolution\\n    w2c: np.ndarray  # (F, 4, 4) world -> camera, normalised world\\n    points_xyz: np.ndarray  # (N, 3) normalised world\\n    points_rgb: np.ndarray  # (N, 3) float in [0, 1]\\n    scene_scale: float  # radius of the camera cloud after normalisation\\n\\n    @property\\n    def num_frames(self) -> int:\\n        return len(self.image_paths)\\n\\n    def camera_centers(self) -> np.ndarray:\\n        r = self.w2c[:, :3, :3]\\n        t = self.w2c[:, :3, 3]\\n        return -np.einsum(\\"fij,fi->fj\\", r, t)\\n\\n    def view_dirs(self) -> np.ndarray:\\n        # Camera +z (forward) in world coordinates = third row of R.\\n        return self.w2c[:, 2, :3]\\n\\n    def load_image(self, i: int) -> np.ndarray:\\n        \\"\\"\\"Return frame i as float32 HxWx3 in [0, 1] at the scene resolution.\\"\\"\\"\\n        img = PILImage.open(self.image_paths[i]).convert(\\"RGB\\")\\n        if img.size != (self.width, self.height):\\n            img = img.resize((self.width, self.height), PILImage.BICUBIC)\\n        return np.asarray(img, dtype=np.float32) / 255.0\\n\\n\\ndef _normalise(c2w_centers: np.ndarray) -> tuple[np.ndarray, float]:\\n    center = c2w_centers.mean(axis=0)\\n    radius = float(np.linalg.norm(c2w_centers - center, axis=1).max())\\n    scale = 1.0 / max(radius, 1e-8)\\n    return center, scale\\n\\n\\ndef load_colmap_scene(\\n    scene_dir: Path,\\n    images_subdir: str = \\"images_4\\",\\n    sparse_subdir: str = \\"sparse/0\\",\\n    max_points: int = 300_000,\\n    name: str | None = None,\\n) -> Scene:\\n    \\"\\"\\"Load a scene whose undistorted images live in ``scene_dir/images_subdir``.\\n\\n    COLMAP intrinsics are rescaled to the actual image size, so downsampled\\n    image folders (images_2/4/8) work with the full-resolution model.\\n    \\"\\"\\"\\n    scene_dir = Path(scene_dir)\\n    model = read_sparse_model(scene_dir / sparse_subdir)\\n    img_dir = scene_dir / images_subdir\\n    if not img_dir.is_dir():\\n        raise FileNotFoundError(f\\"Missing image folder {img_dir}\\")\\n\\n    by_name = {Path(im.name).name: im for im in model.images.values()}\\n    files = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in (\\".png\\", \\".jpg\\", \\".jpeg\\"))\\n    files = [p for p in files if p.name in by_name]\\n    if len(files) < 10:\\n        raise ValueError(f\\"{scene_dir}: only {len(files)} registered images found in {images_subdir}\\")\\n\\n    with PILImage.open(files[0]) as probe:\\n        width, height = probe.size\\n\\n    Ks, w2cs = [], []\\n    for p in files:\\n        im = by_name[p.name]\\n        cam = model.cameras[im.camera_id]\\n        if cam.has_distortion():\\n            raise ValueError(f\\"{scene_dir}: camera {cam.id} is distorted ({cam.model}); use undistorted images\\")\\n        fx, fy, cx, cy = cam.intrinsics()\\n        sx, sy = width / cam.width, height / cam.height\\n        Ks.append(np.array([[fx * sx, 0, cx * sx], [0, fy * sy, cy * sy], [0, 0, 1]], np.float64))\\n        w2cs.append(im.world_to_camera())\\n    Ks = np.stack(Ks)\\n    w2c = np.stack(w2cs)\\n\\n    # Normalise the world so cameras sit inside the unit sphere.\\n    centers = -np.einsum(\\"fij,fi->fj\\", w2c[:, :3, :3], w2c[:, :3, 3])\\n    center, scale = _normalise(centers)\\n    c2w = np.linalg.inv(w2c)\\n    c2w[:, :3, 3] = (c2w[:, :3, 3] - center) * scale\\n    w2c = np.linalg.inv(c2w)\\n    xyz = (model.points_xyz - center) * scale\\n    rgb = model.points_rgb.astype(np.float32) / 255.0\\n\\n    # Drop far outliers and subsample very dense clouds.\\n    keep = np.linalg.norm(xyz, axis=1) < 20.0\\n    xyz, rgb = xyz[keep], rgb[keep]\\n    if len(xyz) > max_points:\\n        idx = np.random.default_rng(0).choice(len(xyz), max_points, replace=False)\\n        xyz, rgb = xyz[idx], rgb[idx]\\n    if len(xyz) < 100:\\n        raise ValueError(f\\"{scene_dir}: sparse cloud too small ({len(xyz)} points)\\")\\n\\n    return Scene(\\n        name=name or scene_dir.name,\\n        image_paths=files,\\n        width=width,\\n        height=height,\\n        Ks=Ks.astype(np.float32),\\n        w2c=w2c.astype(np.float32),\\n        points_xyz=xyz.astype(np.float32),\\n        points_rgb=rgb,\\n        scene_scale=1.0,\\n    )\\n", "lib/splat_restorer/splits.py": "\\"\\"\\"Frame splits for degraded reconstructions.\\n\\nEvery scene reserves a fixed set of *eval* frames (every ``eval_every``-th\\nframe) that no recipe ever trains on, so the distill-back test can measure the\\nsplat on frames nobody has seen. Each recipe then picks its own sparse set of\\n*train* frames; everything else becomes a *pair* frame (render vs. photo).\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\n\\n\\n@dataclass(frozen=True)\\nclass Recipe:\\n    name: str\\n    train_stride: int  # keep every N-th non-eval frame for training\\n    steps: int  # optimisation steps for the degraded splat\\n\\n\\n# Two artefact regimes: very sparse (floaters, holes, smeared geometry) and\\n# moderately sparse (blur and view-dependent errors between views).\\nRECIPES: dict[str, Recipe] = {\\n    \\"sparse_hard\\": Recipe(\\"sparse_hard\\", train_stride=12, steps=4000),\\n    \\"sparse_mild\\": Recipe(\\"sparse_mild\\", train_stride=4, steps=4000),\\n}\\n\\n\\n@dataclass\\nclass FrameSplit:\\n    train: list[int]\\n    pair: list[int]\\n    eval: list[int]\\n\\n\\ndef eval_frames(num_frames: int, eval_every: int = 8) -> list[int]:\\n    return list(range(eval_every // 2, num_frames, eval_every))\\n\\n\\ndef make_split(num_frames: int, recipe: Recipe, eval_every: int = 8) -> FrameSplit:\\n    ev = eval_frames(num_frames, eval_every)\\n    ev_set = set(ev)\\n    candidates = [i for i in range(num_frames) if i not in ev_set]\\n    train = candidates[:: recipe.train_stride]\\n    # Always include the last candidate so the trajectory end is covered.\\n    if candidates and candidates[-1] not in train:\\n        train.append(candidates[-1])\\n    train_set = set(train)\\n    pair = [i for i in candidates if i not in train_set]\\n    return FrameSplit(train=sorted(train), pair=pair, eval=ev)\\n\\n\\ndef subsample(indices: list[int], max_count: int) -> list[int]:\\n    \\"\\"\\"Evenly spaced subset (deterministic) of at most max_count indices.\\"\\"\\"\\n    if len(indices) <= max_count:\\n        return list(indices)\\n    pos = np.linspace(0, len(indices) - 1, max_count).round().astype(int)\\n    return [indices[p] for p in sorted(set(pos.tolist()))]\\n\\n\\ndef nearest_reference(\\n    query: int,\\n    candidates: list[int],\\n    centers: np.ndarray,\\n    forwards: np.ndarray,\\n    angle_weight: float = 0.5,\\n) -> int:\\n    \\"\\"\\"Training frame whose camera is closest in position and viewing direction.\\"\\"\\"\\n    c = np.asarray(candidates)\\n    d_pos = np.linalg.norm(centers[c] - centers[query], axis=1)\\n    d_pos = d_pos / (d_pos.max() + 1e-8)\\n    cos = np.clip(forwards[c] @ forwards[query], -1.0, 1.0)\\n    d_ang = (1.0 - cos) / 2.0\\n    return int(c[np.argmin(d_pos + angle_weight * d_ang)])\\n", "lib/splat_restorer/stages.py": "\\"\\"\\"Notebook-facing stages. Every stage is resumable: re-running skips work\\nwhose durable output already exists on Drive.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport shutil\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nfrom . import dl3dv\\nfrom .pairs import extract_pairs, generate_scene_pairs\\nfrom .workspace import Workspace, atomic_write_json\\n\\n\\ndef log(msg: str) -> None:\\n    print(time.strftime(\\"%H:%M:%S\\"), msg, flush=True)\\n\\n\\n# ----------------------------------------------------------------------------- preflight\\ndef preflight(ws: Workspace) -> dict:\\n    import torch\\n\\n    info = {}\\n    assert torch.cuda.is_available(), \\"No CUDA GPU. Runtime -> Change runtime type -> GPU.\\"\\n    p = torch.cuda.get_device_properties(0)\\n    info[\\"gpu\\"] = f\\"{p.name}, {p.total_memory / 2**30:.0f} GB, sm_{p.major}{p.minor}\\"\\n    info[\\"torch\\"] = f\\"{torch.__version__} (CUDA {torch.version.cuda})\\"\\n    info[\\"bf16\\"] = torch.cuda.is_bf16_supported()\\n\\n    import gsplat\\n\\n    from .gs_trainer import render\\n\\n    # Tiny render: forces the gsplat CUDA build now rather than mid-run.\\n    params = {\\n        \\"means\\": torch.randn(64, 3, device=\\"cuda\\") * 0.1 + torch.tensor([0, 0, 2.0], device=\\"cuda\\"),\\n        \\"scales\\": torch.full((64, 3), -3.0, device=\\"cuda\\"),\\n        \\"quats\\": torch.tensor([[1.0, 0, 0, 0]], device=\\"cuda\\").repeat(64, 1),\\n        \\"opacities\\": torch.zeros(64, device=\\"cuda\\"),\\n        \\"sh0\\": torch.rand(64, 1, 3, device=\\"cuda\\"),\\n        \\"shN\\": torch.zeros(64, 15, 3, device=\\"cuda\\"),\\n    }\\n    K = torch.tensor([[100.0, 0, 64], [0, 100.0, 64], [0, 0, 1]], device=\\"cuda\\")\\n    img, _ = render(params, torch.eye(4, device=\\"cuda\\"), K, 128, 128, 3)\\n    assert img.shape == (128, 128, 3) and torch.isfinite(img).all()\\n    info[\\"gsplat\\"] = f\\"{gsplat.__version__} (render OK)\\"\\n\\n    import diffusers, peft, transformers\\n\\n    info[\\"diffusers\\"] = diffusers.__version__\\n    info[\\"peft\\"] = peft.__version__\\n    info[\\"transformers\\"] = transformers.__version__\\n\\n    from huggingface_hub import HfApi, auth_check\\n\\n    info[\\"hf_user\\"] = HfApi().whoami()[\\"name\\"]\\n    auth_check(dl3dv.REPO, repo_type=\\"dataset\\")\\n    info[\\"dl3dv_access\\"] = \\"OK\\"\\n\\n    info[\\"local_free_gb\\"] = round(shutil.disk_usage(ws.local).free / 2**30, 1)\\n    info[\\"drive_free_gb\\"] = round(shutil.disk_usage(ws.root).free / 2**30, 1)\\n    for k, v in info.items():\\n        log(f\\"  {k:14s} {v}\\")\\n    if info[\\"drive_free_gb\\"] < 30:\\n        log(\\"  WARNING: under 30 GB free on Drive; pairs + checkpoints peak around 25-30 GB.\\")\\n    return info\\n\\n\\n# ----------------------------------------------------------------------------- split\\ndef prepare_split(ws: Workspace, n_test: int, seed: int = 1234) -> dict:\\n    if ws.split_path.exists():\\n        split = json.loads(ws.split_path.read_text(encoding=\\"utf-8\\"))\\n        log(f\\"split: reusing {ws.split_path} ({len(split[\'test\'])} test, {len(split[\'train_pool\'])} train pool)\\")\\n        return split\\n    scenes = dl3dv.list_scenes()\\n    labels = dl3dv.scene_labels()\\n    split = dl3dv.make_split(scenes, n_test=n_test, seed=seed, labels=labels)\\n    atomic_write_json(ws.split_path, split)\\n    log(f\\"split: {len(scenes)} scenes -> {len(split[\'test\'])} test, {len(split[\'train_pool\'])} train pool\\")\\n    return split\\n\\n\\ndef done_scenes(ws: Workspace, scenes: list[str]) -> list[str]:\\n    return [s for s in scenes if ws.pair_tar(s).exists()]\\n\\n\\ndef train_scenes(ws: Workspace, split: dict, n: int) -> list[str]:\\n    \\"\\"\\"First n train-pool scenes whose pairs exist (failed scenes are skipped).\\"\\"\\"\\n    failed = set(_failures(ws))\\n    pool = [s for s in split[\\"train_pool\\"] if s not in failed]\\n    return done_scenes(ws, pool)[:n]\\n\\n\\ndef _failures(ws: Workspace) -> dict:\\n    p = ws.root / \\"failed_scenes.json\\"\\n    return json.loads(p.read_text(encoding=\\"utf-8\\")) if p.exists() else {}\\n\\n\\n# ----------------------------------------------------------------------------- pairs\\ndef generate_pairs(\\n    ws: Workspace,\\n    split: dict,\\n    n_train: int,\\n    recipes: list[str],\\n    max_pairs_per_recipe: int = 100,\\n    only_test: bool = False,\\n    time_budget_h: float | None = None,\\n) -> None:\\n    \\"\\"\\"Make pairs for all test scenes, then train-pool scenes until n_train succeed.\\n\\n    Downloads the next scene in a background thread while the GPU trains on\\n    the current one.\\n    \\"\\"\\"\\n    t_start = time.time()\\n    test = set(split[\\"test\\"])\\n    failed = _failures(ws)\\n    queue = [s for s in split[\\"test\\"] if s not in failed]\\n    if not only_test:\\n        queue += [s for s in split[\\"train_pool\\"] if s not in failed]\\n\\n    def need_more_train() -> bool:\\n        return len(done_scenes(ws, [s for s in split[\\"train_pool\\"] if s not in failed])) < n_train\\n\\n    def wanted(s: str) -> bool:\\n        return not ws.pair_tar(s).exists() and (s in test or need_more_train())\\n\\n    pending = [s for s in queue if wanted(s)]\\n    log(f\\"pairs: {len(done_scenes(ws, list(test)))}/{len(test)} test done, \\"\\n        f\\"{len(done_scenes(ws, split[\'train_pool\']))} train done (target {n_train}), {len(pending)} candidates queued\\")\\n\\n    prefetch: dict[str, dict] = {}\\n\\n    def start_download(s: str):\\n        slot = {\\"done\\": threading.Event(), \\"dir\\": None, \\"error\\": None}\\n\\n        def run():\\n            try:\\n                slot[\\"dir\\"] = dl3dv.download_scene(s, ws.raw_dir(s))\\n            except Exception as e:  # noqa: BLE001 - reported below\\n                slot[\\"error\\"] = e\\n            finally:\\n                slot[\\"done\\"].set()\\n\\n        threading.Thread(target=run, daemon=True).start()\\n        prefetch[s] = slot\\n\\n    order = [s for s in pending]\\n    for k, scene in enumerate(order):\\n        if not wanted(scene):\\n            continue\\n        if time_budget_h and (time.time() - t_start) / 3600 > time_budget_h:\\n            log(f\\"pairs: time budget of {time_budget_h} h reached, stopping (re-run to continue)\\")\\n            break\\n        if scene not in prefetch:\\n            start_download(scene)\\n        nxt = next((s for s in order[k + 1 :] if wanted(s)), None)\\n        if nxt and nxt not in prefetch:\\n            start_download(nxt)\\n        slot = prefetch.pop(scene)\\n        slot[\\"done\\"].wait()\\n        is_test = scene in test\\n        try:\\n            if slot[\\"error\\"]:\\n                raise slot[\\"error\\"]\\n            generate_scene_pairs(\\n                ws, scene, slot[\\"dir\\"], recipes, is_test=is_test, max_pairs_per_recipe=max_pairs_per_recipe, log=log\\n            )\\n        except Exception as e:  # noqa: BLE001 - one bad scene must not stop the run\\n            failed[scene] = f\\"{type(e).__name__}: {e}\\"\\n            atomic_write_json(ws.root / \\"failed_scenes.json\\", failed)\\n            log(f\\"  {scene[:12]} FAILED: {failed[scene]}\\")\\n            traceback.print_exc()\\n        finally:\\n            dl3dv.remove_download(ws.raw_dir(scene))\\n        done_train = len(done_scenes(ws, split[\\"train_pool\\"]))\\n        log(f\\"pairs progress: test {len(done_scenes(ws, list(test)))}/{len(test)}, train {done_train}/{n_train}, \\"\\n            f\\"elapsed {(time.time() - t_start) / 60:.0f} min\\")\\n        if not only_test and not need_more_train() and all(ws.pair_tar(s).exists() or s in failed for s in test):\\n            break\\n    # Do not leave half-downloaded prefetches around.\\n    for s in list(prefetch):\\n        prefetch[s][\\"done\\"].wait()\\n        dl3dv.remove_download(ws.raw_dir(s))\\n\\n\\ndef generate_one(ws: Workspace, scene: str, recipes: list[str], is_test: bool, max_pairs_per_recipe: int = 100) -> Path:\\n    if ws.pair_tar(scene).exists():\\n        log(f\\"pairs for {scene[:12]} already on Drive\\")\\n        return ws.pair_tar(scene)\\n    try:\\n        scene_dir = dl3dv.download_scene(scene, ws.raw_dir(scene))\\n        return generate_scene_pairs(ws, scene, scene_dir, recipes, is_test=is_test,\\n                                    max_pairs_per_recipe=max_pairs_per_recipe, log=log)\\n    finally:\\n        dl3dv.remove_download(ws.raw_dir(scene))\\n\\n\\ndef sanity_overfit(ws: Workspace, scene: str, steps: int = 300) -> bool:\\n    \\"\\"\\"Train the restorer briefly on ONE scene and check it beats its input there.\\n\\n    This proves the plumbing (data, model, loss, optimiser) works before any\\n    long run. Outputs stay on local disk.\\n    \\"\\"\\"\\n    import csv as _csv\\n\\n    from .train_restorer import RestorerConfig, train_restorer\\n\\n    root = extract(ws, [scene])\\n    local_ws = Workspace(root=ws.local / \\"sanity\\", local=ws.local / \\"sanity_local\\")\\n    shutil.rmtree(local_ws.runs_dir / \\"sanity\\", ignore_errors=True)\\n    cfg = RestorerConfig(\\n        run_name=\\"sanity\\", scenes=[scene], steps=steps, sample_every=100, log_every=10,\\n        local_ckpt_every=10**9, gram_warmup_steps=10**9, warmup_steps=50, num_workers=4,\\n    )\\n    train_restorer(local_ws, cfg, root, log=log)\\n    rows = [r for r in _csv.DictReader(open(local_ws.runs_dir / \\"sanity\\" / \\"log.csv\\")) if r[\\"val_psnr\\"]]\\n    last = rows[-1]\\n    gain = float(last[\\"val_psnr\\"]) - float(last[\\"val_psnr_input\\"])\\n    ok = gain > 0.5\\n    log(f\\"SANITY {\'PASSED\' if ok else \'FAILED\'}: val PSNR {float(last[\'val_psnr\']):.2f} vs input \\"\\n        f\\"{float(last[\'val_psnr_input\']):.2f} ({gain:+.2f} dB) after {steps} steps\\")\\n    (local_ws.runs_dir / \\"sanity\\" / \\"model_final.pt\\").unlink(missing_ok=True)\\n    return ok\\n\\n\\ndef extract(ws: Workspace, scenes: list[str]) -> Path:\\n    t = time.time()\\n    root = extract_pairs(ws, scenes)\\n    log(f\\"extracted {len(scenes)} scenes to {root} in {time.time() - t:.0f}s\\")\\n    return root\\n\\n\\n# ----------------------------------------------------------------------------- reports\\ndef show_markdown(path: Path) -> None:\\n    try:\\n        from IPython.display import Markdown, display\\n\\n        display(Markdown(path.read_text(encoding=\\"utf-8\\")))\\n    except Exception:  # noqa: BLE001 - plain text fallback outside notebooks\\n        print(path.read_text(encoding=\\"utf-8\\"))\\n\\n", "lib/splat_restorer/train_ab.py": "\\"\\"\\"A/B training variants of the app\'s static trainer on one captured scene.\\n\\nEvery variant trains the same Trainer4DGS setup the hybrid notebook uses\\n(\\"Spirula dataset + our trainer\\"), on the same frames and seed, and is scored\\non the same held-out photos. The trainer classes are passed in, so this module\\ndoes not depend on the app\'s source layout.\\n\\nVariants change one thing at a time:\\n\\n* ``lr_decay``: exponential decay of the Gaussian position learning rate\\n  (standard 3DGS; the app\'s trainer keeps it constant). Applied from the\\n  trainer\'s progress callback, so the trainer code is unchanged.\\n* ``clean``: training frames are undistorted at full resolution, area-averaged\\n  down (2x2 averaging halves sensor noise) and lightly denoised. Held-out\\n  photos are never trained on; they are scored both as captured and cleaned.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom PIL import Image\\n\\nfrom .colmap_io import read_sparse_model\\nfrom .own_splat import SplatData, _k_and_dist, _scores, _to_u8, _w2c_t, comparison_image, load_ply\\nfrom .gs_trainer import render\\n\\n\\n@dataclass\\nclass Variant:\\n    name: str\\n    steps: int = 30000\\n    cap: int = 1_000_000\\n    lr_decay: bool = False\\n    lr_final_frac: float = 0.01\\n    clean: bool = False\\n    note: str = \\"\\"\\n\\n\\nVARIANTS = {\\n    \\"baseline\\": Variant(\\"baseline\\", note=\\"28 Eylül eğitimiyle aynı ayarlar\\"),\\n    \\"lr_decay\\": Variant(\\"lr_decay\\", lr_decay=True, note=\\"konum öğrenme oranı 100x azalır\\"),\\n    \\"lr_decay_clean\\": Variant(\\"lr_decay_clean\\", lr_decay=True, clean=True, note=\\"+ gürültüsü azaltılmış eğitim kareleri\\"),\\n    \\"lr_decay_60k\\": Variant(\\"lr_decay_60k\\", steps=60000, lr_decay=True, note=\\"+ 60k adım\\"),\\n}\\n\\n\\n# ----------------------------------------------------------------------------- clean frames\\n\\n\\ndef clean_frames(source: Path, data: SplatData, out: Path, model_name: str = \\"0\\", nlm_h: float = 3.0,\\n                 workers: int = 8, log=print) -> Path:\\n    \\"\\"\\"Frames on the same cameras as ``data``, area-downsampled from full resolution and denoised.\\"\\"\\"\\n    import cv2\\n\\n    out = Path(out)\\n    frames = out / \\"frames\\"\\n    frames.mkdir(parents=True, exist_ok=True)\\n    model = read_sparse_model(Path(source) / \\"sparse\\" / model_name)\\n    cam = next(iter(model.cameras.values()))\\n    K, dist = _k_and_dist(cam)\\n    w, h = cam.width, cam.height\\n    sx, sy = w / data.width, h / data.height\\n    # Same output camera as data.K, expressed at full resolution (pixel-centre convention).\\n    k_full = data.K.copy()\\n    k_full[0, 0] *= sx; k_full[1, 1] *= sy\\n    k_full[0, 2] = (data.K[0, 2] + 0.5) * sx - 0.5\\n    k_full[1, 2] = (data.K[1, 2] + 0.5) * sy - 0.5\\n    mx, my = cv2.initUndistortRectifyMap(K, dist, None, k_full, (w, h), cv2.CV_32FC1)\\n\\n    def convert(i):\\n        dest = frames / data.frames[i]\\n        if dest.exists():\\n            return\\n        img = cv2.imread(str(Path(source) / \\"images\\" / data.names[i]))\\n        full = cv2.remap(img, mx, my, cv2.INTER_LINEAR)\\n        small = cv2.resize(full, (data.width, data.height), interpolation=cv2.INTER_AREA)\\n        if nlm_h > 0:\\n            small = cv2.fastNlMeansDenoisingColored(small, None, nlm_h, nlm_h, 7, 21)\\n        tmp = dest.with_suffix(\\".tmp.png\\")\\n        cv2.imwrite(str(tmp), small, [cv2.IMWRITE_PNG_COMPRESSION, 1])\\n        tmp.replace(dest)\\n\\n    todo = [i for i in range(data.num_frames) if not (frames / data.frames[i]).exists()]\\n    if todo:\\n        log(f\\"[clean] {len(todo)} frames: full-res undistort -> area downsample -> NLM h={nlm_h}\\")\\n        with ThreadPoolExecutor(workers) as pool:\\n            for n, _ in enumerate(pool.map(convert, todo), 1):\\n                if n % 200 == 0 or n == len(todo):\\n                    log(f\\"[clean] {n}/{len(todo)}\\")\\n    return frames\\n\\n\\n# ----------------------------------------------------------------------------- training\\n\\n\\ndef init_points(source: Path, model_name: str, seed: int, max_points: int = 150_000) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\\n    \\"\\"\\"COLMAP points, subsampled exactly like the hybrid notebook.\\"\\"\\"\\n    model = read_sparse_model(Path(source) / \\"sparse\\" / model_name)\\n    xyz, rgb = model.points_xyz, model.points_rgb\\n    ids = np.arange(len(xyz))\\n    if len(ids) > max_points:\\n        ids = np.sort(np.random.default_rng(seed).choice(len(xyz), max_points, replace=False))\\n    return xyz, rgb, ids\\n\\n\\ndef train_variant(\\n    v: Variant,\\n    data: SplatData,\\n    frames_dir: Path,\\n    xyz: np.ndarray,\\n    rgb: np.ndarray,\\n    init_ids: np.ndarray,\\n    holdout: list[int],\\n    work: Path,\\n    app,  # namespace with GaussianModel, DeformationField, Trainer4DGS, export_to_ply, scene_extent\\n    seed: int = 42,\\n    log_interval: int = 200,\\n    log=print,\\n) -> tuple[Path, dict]:\\n    \\"\\"\\"Train one variant with the app\'s trainer; returns (exported PLY, info).\\"\\"\\"\\n    torch.manual_seed(seed)\\n    np.random.seed(seed)\\n    gs = app.GaussianModel(torch.from_numpy(xyz[init_ids]).float(),\\n                           torch.from_numpy(rgb[init_ids].astype(\\"float32\\") / 255), fourier_K=0)\\n    deform = app.DeformationField(resolution=8, feat_dim=4, mlp_width=32, mlp_depth=1, num_time_freqs=2)\\n    for p in deform.parameters():\\n        p.requires_grad_(False)\\n    trainer = app.Trainer4DGS(\\n        gs, deform, scene_extent=app.scene_extent(xyz), max_gaussians=v.cap,\\n        density_end_iter=min(25000, v.steps // 2), deform_pos_mode=\\"mlp\\",\\n        lambda_deform_reg=0, lambda_smoothness=0, lambda_rigidity=0, lambda_fourier_reg=0, lambda_mask_motion=0,\\n    )\\n    lr0 = trainer.lr_specs[\\"means\\"]\\n    curve = []\\n\\n    def progress(it, total, loss, psnr, n):\\n        curve.append({\\"iter\\": it, \\"loss\\": loss, \\"train_psnr\\": psnr, \\"splats\\": n})\\n        if v.lr_decay:\\n            lr = lr0 * v.lr_final_frac ** (min(it, v.steps) / v.steps)\\n            for g in trainer.optimizer.param_groups:\\n                if any(p is trainer.gs.means for p in g[\\"params\\"]):\\n                    g[\\"lr\\"] = lr\\n\\n    t0 = time.time()\\n    trainer.train(\\n        [frames_dir / f for f in data.frames], torch.from_numpy(data.K).float(),\\n        [torch.from_numpy(w).float() for w in data.w2c],\\n        n_iters=v.steps, image_size=(data.width, data.height), static_mode=True, preload_to_ram=True,\\n        log_interval=log_interval, ckpt_dir=work / \\"ckpt\\", ckpt_interval=v.steps,\\n        sh_progressive_schedule=True, holdout_indices=holdout, progress_callback=progress,\\n    )\\n    torch.cuda.synchronize()\\n    seconds = time.time() - t0\\n    ply = app.export_to_ply(gs, None, work / \\"export\\", num_timestamps=1)[0]\\n    info = {\\"variant\\": asdict(v), \\"train_seconds\\": seconds, \\"splats\\": int(gs.num_points), \\"curve\\": curve}\\n    del trainer, gs, deform\\n    torch.cuda.empty_cache()\\n    return Path(ply), info\\n\\n\\n# ----------------------------------------------------------------------------- evaluation\\n\\n\\n@torch.no_grad()\\ndef evaluate_ply(ply: Path, data: SplatData, holdout: list[int], clean_dir: Path | None, net_lpips,\\n                 render_dir: Path, device=\\"cuda\\") -> dict:\\n    \\"\\"\\"Score held-out frames against the captured photo and (if given) its cleaned version.\\"\\"\\"\\n    params, deg = load_ply(ply, device)\\n    w2c, K = _w2c_t(data, device)\\n    render_dir.mkdir(parents=True, exist_ok=True)\\n    rows = {}\\n    for i in holdout:\\n        img, _ = render(params, w2c[i], K, data.width, data.height, deg)\\n        row = _scores(img, data.photo(i, device), net_lpips)\\n        if clean_dir is not None:\\n            clean = torch.from_numpy(np.asarray(Image.open(clean_dir / data.frames[i]).convert(\\"RGB\\")).copy())\\n            row[\\"clean\\"] = _scores(img, clean.to(device).float() / 255, net_lpips)\\n        rows[str(i)] = row\\n        Image.fromarray(_to_u8(img)).save(render_dir / f\\"{i:06d}.jpg\\", quality=92)\\n    del params\\n    torch.cuda.empty_cache()\\n    return rows\\n\\n\\ndef summarise(results: dict) -> dict:\\n    out = {}\\n    for name, r in results.items():\\n        rows = list(r[\\"frames\\"].values())\\n        m = {k: float(np.mean([x[k] for x in rows])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n        if rows and \\"clean\\" in rows[0]:\\n            m.update({f\\"clean_{k}\\": float(np.mean([x[\\"clean\\"][k] for x in rows])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")})\\n        m.update(splats=r[\\"info\\"][\\"splats\\"], minutes=r[\\"info\\"][\\"train_seconds\\"] / 60)\\n        out[name] = m\\n    return out\\n\\n\\ndef markdown(summary: dict, n_frames: int) -> str:\\n    base = summary.get(\\"baseline\\") or next(iter(summary.values()))\\n    has_clean = \\"clean_psnr\\" in base\\n    head = \\"| varyant | PSNR ↑ | Δ | SSIM ↑ | LPIPS ↓ | Δ |\\" + (\\" PSNR (temiz foto) | Δ |\\" if has_clean else \\"\\") + \\" splat | dk |\\"\\n    sep = \\"|---\\" * (head.count(\\"|\\") - 1) + \\"|\\"\\n    lines = [f\\"## Eğitim varyantları: {n_frames} saklanan kare (hiçbiri eğitimde kullanılmadı)\\", \\"\\", head, sep]\\n    for name, m in summary.items():\\n        row = (f\\"| {name} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - base[\'psnr\']:+.2f} | {m[\'ssim\']:.4f} | \\"\\n               f\\"{m[\'lpips\']:.4f} | {m[\'lpips\'] - base[\'lpips\']:+.4f} |\\")\\n        if has_clean:\\n            row += f\\" {m[\'clean_psnr\']:.2f} | {m[\'clean_psnr\'] - base[\'clean_psnr\']:+.2f} |\\"\\n        row += f\\" {m[\'splats\']:,} | {m[\'minutes\']:.0f} |\\"\\n        lines.append(row)\\n    lines += [\\"\\", \\"Δ: ilk satıra (baseline) göre. \'Temiz foto\' sütunu, sensör gürültüsü azaltılmış fotoğrafa karşı ölçer; \\"\\n              \\"gürültüsüz eğitilen varyant orijinal fotoğrafın gürültüsünü taklit etmediği için orada daha adil karşılaştırılır.\\"]\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef comparison_sheets(results: dict, data: SplatData, out_dir: Path, render_root: Path, n_worst: int = 4, n_even: int = 4) -> list[Path]:\\n    names = list(results)\\n    first = results[names[0]][\\"frames\\"]\\n    ids = sorted(int(i) for i in first)\\n    worst = sorted(ids, key=lambda i: first[str(i)][\\"psnr\\"])[:n_worst]\\n    even = [ids[k] for k in np.linspace(0, len(ids) - 1, n_even + 2).round().astype(int)[1:-1]]\\n    picks = list(dict.fromkeys(worst + even))\\n    paths = []\\n    for i in picks:\\n        tiles = {\\"foto\\": _to_u8(data.photo(i, \\"cpu\\"))}\\n        for n in names:\\n            tiles[n] = np.asarray(Image.open(render_root / n / \\"renders\\" / f\\"{i:06d}.jpg\\").convert(\\"RGB\\"))\\n        p = out_dir / f\\"compare_{i:06d}.jpg\\"\\n        comparison_image(tiles, p, tile_width=560, box_from=names[0])\\n        paths.append(p)\\n    return paths\\n\\n\\ndef save_json(path: Path, obj) -> None:\\n    tmp = path.with_suffix(\\".tmp\\")\\n    tmp.write_text(json.dumps(obj, indent=1), encoding=\\"utf-8\\")\\n    tmp.replace(path)\\n", "lib/splat_restorer/train_restorer.py": "\\"\\"\\"Train the diffusion restorer on extracted pairs. Resumable from Drive.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport json\\nimport math\\nimport random\\nimport time\\nfrom dataclasses import asdict, dataclass, field\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\n\\nfrom .data import PairDataset, collect_items, split_holdout\\nfrom .workspace import Workspace, atomic_copy, atomic_write_json\\n\\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\\nIMAGENET_STD = (0.229, 0.224, 0.225)\\nGRAM_LAYERS = {3: 1.0 / 2.6, 8: 1.0 / 4.8, 15: 1.0 / 3.7, 22: 1.0 / 5.6, 29: 10.0 / 1.5}\\n\\n\\n@dataclass\\nclass RestorerConfig:\\n    run_name: str\\n    scenes: list[str]\\n    use_ref: bool = True\\n    steps: int = 10_000\\n    batch_size: int = 4\\n    lr: float = 2e-5\\n    warmup_steps: int = 500\\n    weight_decay: float = 1e-2\\n    lambda_l2: float = 1.0\\n    lambda_lpips: float = 1.0\\n    lambda_gram: float = 1.0\\n    gram_warmup_steps: int = 2000\\n    crop: int = 512\\n    timestep: int = 199\\n    lora_rank_vae: int = 4\\n    grad_checkpointing: bool = False\\n    num_workers: int = 8\\n    log_every: int = 50\\n    sample_every: int = 500\\n    local_ckpt_every: int = 500\\n    drive_ckpt_every: int = 2000\\n    val_items: int = 64\\n    seed: int = 0\\n    recipes: list[str] = field(default_factory=list)\\n\\n\\ndef gram_loss(pred: torch.Tensor, target: torch.Tensor, vgg: torch.nn.Module) -> torch.Tensor:\\n    \\"\\"\\"Difix-style style loss on VGG16 feature Gram matrices (inputs in [0, 1]).\\"\\"\\"\\n    mean = torch.tensor(IMAGENET_MEAN, device=pred.device).view(1, 3, 1, 1)\\n    std = torch.tensor(IMAGENET_STD, device=pred.device).view(1, 3, 1, 1)\\n    x, y = (pred - mean) / std, (target - mean) / std\\n    total = pred.new_zeros(())\\n    for i, layer in enumerate(vgg):\\n        x, y = layer(x), layer(y)\\n        if i in GRAM_LAYERS:\\n            b, d, h, w = x.shape\\n            fx, fy = x.reshape(b, d, h * w).float(), y.reshape(b, d, h * w).float()\\n            gx = fx @ fx.transpose(1, 2)\\n            gy = fy @ fy.transpose(1, 2)\\n            total = total + GRAM_LAYERS[i] * ((gx - gy) ** 2).mean() / (d * h * w)\\n        if i >= max(GRAM_LAYERS):\\n            break\\n    return total\\n\\n\\ndef _infinite(loader):\\n    while True:\\n        for batch in loader:\\n            yield batch\\n\\n\\ndef _to_views(batch, use_ref: bool, device) -> tuple[torch.Tensor, torch.Tensor]:\\n    deg, gt, ref = (batch[k].to(device, non_blocking=True) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n    if use_ref:\\n        x = torch.stack([deg, ref], dim=1)\\n        y = torch.stack([gt, ref], dim=1)\\n    else:\\n        x, y = deg[:, None], gt[:, None]\\n    return x * 2 - 1, y * 2 - 1\\n\\n\\ndef _save_grid(path: Path, rows: list[list[torch.Tensor]]) -> None:\\n    from PIL import Image\\n\\n    lines = [torch.cat([t.clamp(0, 1) for t in row], dim=2) for row in rows]\\n    grid = torch.cat(lines, dim=1)\\n    arr = (grid.permute(1, 2, 0).cpu().numpy() * 255 + 0.5).astype(np.uint8)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    Image.fromarray(arr).save(path, quality=90)\\n\\n\\n@torch.no_grad()\\ndef validate(model, val_ds, use_ref: bool, net_lpips, device, max_items: int) -> dict:\\n    model.set_eval()\\n    l2s, lps, psnrs, base_psnrs = [], [], [], []\\n    for i in range(min(len(val_ds), max_items)):\\n        s = val_ds[i]\\n        deg, gt, ref = (s[k][None].to(device) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n        out = model.restore(deg, ref if use_ref else None)\\n        mse = F.mse_loss(out, gt).item()\\n        l2s.append(mse)\\n        psnrs.append(10 * math.log10(1 / max(mse, 1e-12)))\\n        base_psnrs.append(10 * math.log10(1 / max(F.mse_loss(deg, gt).item(), 1e-12)))\\n        lps.append(net_lpips(out * 2 - 1, gt * 2 - 1).mean().item())\\n    model.set_train()\\n    return {\\n        \\"val_psnr\\": float(np.mean(psnrs)),\\n        \\"val_psnr_input\\": float(np.mean(base_psnrs)),\\n        \\"val_lpips\\": float(np.mean(lps)),\\n        \\"val_l2\\": float(np.mean(l2s)),\\n    }\\n\\n\\ndef train_restorer(ws: Workspace, cfg: RestorerConfig, pairs_root: Path, device: str = \\"cuda\\", log=print) -> Path:\\n    import lpips\\n    import torchvision\\n\\n    from .restorer_model import Restorer\\n\\n    run_dir = ws.runs_dir / cfg.run_name\\n    run_dir.mkdir(parents=True, exist_ok=True)\\n    final_path = run_dir / \\"model_final.pt\\"\\n    if final_path.exists():\\n        log(f\\"[{cfg.run_name}] already finished -> {final_path}\\")\\n        return final_path\\n    local_dir = ws.local / \\"runs\\" / cfg.run_name\\n    local_dir.mkdir(parents=True, exist_ok=True)\\n    atomic_write_json(run_dir / \\"config.json\\", asdict(cfg))\\n\\n    random.seed(cfg.seed)\\n    np.random.seed(cfg.seed)\\n    torch.manual_seed(cfg.seed)\\n    torch.backends.cuda.matmul.allow_tf32 = True\\n    torch.backends.cudnn.allow_tf32 = True\\n\\n    items = collect_items(pairs_root, cfg.scenes, cfg.recipes or None)\\n    train_items, val_items = split_holdout(items, frac=0.02, seed=cfg.seed)\\n    log(f\\"[{cfg.run_name}] {len(cfg.scenes)} scenes, {len(train_items)} train pairs, {len(val_items)} val pairs, ref={cfg.use_ref}\\")\\n    train_ds = PairDataset(train_items, train=True, crop=cfg.crop)\\n    val_ds = PairDataset(val_items, train=False)\\n    loader = torch.utils.data.DataLoader(\\n        train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,\\n        pin_memory=True, drop_last=True, persistent_workers=cfg.num_workers > 0,\\n    )\\n\\n    model = Restorer(timestep=cfg.timestep, lora_rank_vae=cfg.lora_rank_vae, device=device)\\n    model.set_train()\\n    if cfg.grad_checkpointing:\\n        model.unet.enable_gradient_checkpointing()\\n    params = list(model.unet.parameters()) + model.vae_trainable_parameters()\\n    opt = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)\\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(cfg.warmup_steps, 1)))\\n\\n    net_lpips = lpips.LPIPS(net=\\"vgg\\", verbose=False).to(device).eval().requires_grad_(False)\\n    vgg = torchvision.models.vgg16(weights=\\"IMAGENET1K_V1\\").features[:30].to(device).eval().requires_grad_(False)\\n\\n    # Resume: prefer the newer of local / Drive checkpoints.\\n    step = 0\\n    candidates = [p for p in (local_dir / \\"ckpt.pt\\", run_dir / \\"ckpt.pt\\") if p.exists()]\\n    if candidates:\\n        ck_path = max(candidates, key=lambda p: torch.load(p, map_location=\\"cpu\\", weights_only=False, mmap=True)[\\"step\\"])\\n        ck = torch.load(ck_path, map_location=\\"cpu\\", weights_only=False)\\n        model.load_trainable_state_dict(ck[\\"model\\"])\\n        opt.load_state_dict(ck[\\"opt\\"])\\n        sched.load_state_dict(ck[\\"sched\\"])\\n        step = int(ck[\\"step\\"])\\n        log(f\\"[{cfg.run_name}] resumed from {ck_path} at step {step}\\")\\n        del ck\\n\\n    log_path = run_dir / \\"log.csv\\"\\n    new_log = not log_path.exists()\\n    log_f = open(log_path, \\"a\\", newline=\\"\\")\\n    writer = csv.writer(log_f)\\n    if new_log:\\n        writer.writerow([\\"step\\", \\"loss\\", \\"l2\\", \\"lpips\\", \\"gram\\", \\"lr\\", \\"sec_per_step\\", \\"val_psnr\\", \\"val_psnr_input\\", \\"val_lpips\\"])\\n\\n    def save_ckpt(to_drive: bool):\\n        state = {\\"model\\": model.trainable_state_dict(), \\"opt\\": opt.state_dict(), \\"sched\\": sched.state_dict(), \\"step\\": step}\\n        tmp = local_dir / \\"ckpt.pt.partial\\"\\n        torch.save(state, tmp)\\n        tmp.replace(local_dir / \\"ckpt.pt\\")\\n        if to_drive:\\n            atomic_copy(local_dir / \\"ckpt.pt\\", run_dir / \\"ckpt.pt\\")\\n\\n    it = _infinite(loader)\\n    t_last = time.time()\\n    while step < cfg.steps:\\n        batch = next(it)\\n        x, y = _to_views(batch, cfg.use_ref, device)\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n            pred = model(x)\\n        pred_f = pred.float().flatten(0, 1)\\n        y_f = y.flatten(0, 1)\\n        l2 = F.mse_loss(pred_f, y_f) * cfg.lambda_l2\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n            lp = net_lpips(pred_f, y_f).float().mean() * cfg.lambda_lpips\\n        loss = l2 + lp\\n        gram = torch.zeros((), device=device)\\n        if cfg.lambda_gram > 0 and step >= cfg.gram_warmup_steps:\\n            h, w = pred_f.shape[-2:]\\n            c = min(400, h, w)\\n            top, left = random.randint(0, h - c), random.randint(0, w - c)\\n            p_c = pred_f[..., top : top + c, left : left + c] * 0.5 + 0.5\\n            y_c = y_f[..., top : top + c, left : left + c] * 0.5 + 0.5\\n            with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n                gram = gram_loss(p_c, y_c, vgg) * cfg.lambda_gram\\n            loss = loss + gram\\n        opt.zero_grad(set_to_none=True)\\n        loss.backward()\\n        torch.nn.utils.clip_grad_norm_(params, 1.0)\\n        opt.step()\\n        sched.step()\\n        step += 1\\n\\n        if not math.isfinite(loss.item()):\\n            raise FloatingPointError(f\\"[{cfg.run_name}] non-finite loss at step {step}\\")\\n\\n        val = {}\\n        if step % cfg.sample_every == 0 or step == cfg.steps:\\n            val = validate(model, val_ds, cfg.use_ref, net_lpips, device, cfg.val_items)\\n            rows = []\\n            for i in range(min(4, len(val_ds))):\\n                s = val_ds[i]\\n                deg, gt, ref = (s[k][None].to(device) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n                model.set_eval()\\n                out = model.restore(deg, ref if cfg.use_ref else None)\\n                model.set_train()\\n                rows.append([deg[0], out[0], gt[0], ref[0]])\\n            _save_grid(run_dir / \\"samples\\" / f\\"step_{step:06d}.jpg\\", rows)\\n            log(\\n                f\\"[{cfg.run_name}] step {step}: val PSNR {val[\'val_psnr\']:.2f} \\"\\n                f\\"(input {val[\'val_psnr_input\']:.2f}), LPIPS {val[\'val_lpips\']:.4f}\\"\\n            )\\n        if step % cfg.log_every == 0:\\n            dt = (time.time() - t_last) / cfg.log_every\\n            t_last = time.time()\\n            writer.writerow([\\n                step, f\\"{loss.item():.5f}\\", f\\"{l2.item():.5f}\\", f\\"{lp.item():.5f}\\", f\\"{gram.item():.5f}\\",\\n                f\\"{sched.get_last_lr()[0]:.2e}\\", f\\"{dt:.3f}\\",\\n                val.get(\\"val_psnr\\", \\"\\"), val.get(\\"val_psnr_input\\", \\"\\"), val.get(\\"val_lpips\\", \\"\\"),\\n            ])\\n            log_f.flush()\\n            if step % (cfg.log_every * 10) == 0:\\n                log(f\\"[{cfg.run_name}] step {step}/{cfg.steps} loss {loss.item():.4f} ({dt:.2f}s/step)\\")\\n        if step % cfg.local_ckpt_every == 0 and step < cfg.steps:\\n            save_ckpt(to_drive=step % cfg.drive_ckpt_every == 0)\\n\\n    log_f.close()\\n    final = model.trainable_state_dict()\\n    final[\\"unet\\"] = {k: v.half() for k, v in final[\\"unet\\"].items()}\\n    final[\\"use_ref\\"] = cfg.use_ref\\n    final[\\"lora_rank_vae\\"] = cfg.lora_rank_vae\\n    tmp = local_dir / \\"model_final.pt\\"\\n    torch.save(final, tmp)\\n    atomic_copy(tmp, final_path)\\n    # The resume checkpoint is large; drop it once the final weights are safe.\\n    for p in (run_dir / \\"ckpt.pt\\", local_dir / \\"ckpt.pt\\"):\\n        p.unlink(missing_ok=True)\\n    atomic_write_json(run_dir / \\"DONE.json\\", {\\"steps\\": step, \\"finished\\": time.strftime(\\"%Y-%m-%d %H:%M:%S\\")})\\n    log(f\\"[{cfg.run_name}] finished -> {final_path}\\")\\n    return final_path\\n\\n\\ndef load_restorer(path: Path, device: str = \\"cuda\\"):\\n    from .restorer_model import Restorer\\n\\n    sd = torch.load(path, map_location=\\"cpu\\", weights_only=False)\\n    model = Restorer(timestep=int(sd.get(\\"timestep\\", 199)), lora_rank_vae=int(sd.get(\\"lora_rank_vae\\", 4)), device=device)\\n    sd[\\"unet\\"] = {k: v.float() for k, v in sd[\\"unet\\"].items()}\\n    model.load_trainable_state_dict(sd)\\n    model.set_eval()\\n    return model, bool(sd.get(\\"use_ref\\", False))\\n", "lib/splat_restorer/video.py": "\\"\\"\\"Run a restorer over every frame of a splat render video.\\n\\nThere is no ground-truth photo for a render video, so the outputs are for\\nlooking at (side-by-side video, zoomed comparison sheet) plus three rough\\nnumbers: how much the model changed each frame, how much sharper it got, and\\nwhether frame-to-frame flicker went up (per-frame restoration is not\\ntemporally consistent).\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport shutil\\nimport subprocess\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image, ImageDraw\\n\\n\\ndef probe(video: Path) -> dict:\\n    out = subprocess.run(\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-select_streams\\", \\"v:0\\", \\"-show_entries\\",\\n         \\"stream=width,height,r_frame_rate,nb_frames:format=duration\\", \\"-of\\", \\"json\\", str(video)],\\n        check=True, capture_output=True, text=True,\\n    ).stdout\\n    info = json.loads(out)\\n    s = info[\\"streams\\"][0]\\n    num, den = (int(x) for x in s[\\"r_frame_rate\\"].split(\\"/\\"))\\n    return {\\n        \\"width\\": int(s[\\"width\\"]),\\n        \\"height\\": int(s[\\"height\\"]),\\n        \\"fps\\": num / den if den else 30.0,\\n        \\"duration\\": float(info.get(\\"format\\", {}).get(\\"duration\\", 0) or 0),\\n    }\\n\\n\\ndef target_size(width: int, height: int, long_edge: int) -> tuple[int, int]:\\n    \\"\\"\\"Scale so the long edge is ``long_edge`` (never upscale); both sides multiples of 8.\\"\\"\\"\\n    s = min(1.0, long_edge / max(width, height))\\n    return max(8, round(width * s / 8) * 8), max(8, round(height * s / 8) * 8)\\n\\n\\ndef extract_frames(video: Path, out_dir: Path, size: tuple[int, int], stride: int = 1, max_seconds: float = 0) -> int:\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    vf = f\\"scale={size[0]}:{size[1]}:flags=lanczos\\"\\n    if stride > 1:\\n        vf = f\\"select=\'not(mod(n\\\\\\\\,{stride}))\',{vf}\\"\\n    cmd = [\\"ffmpeg\\", \\"-v\\", \\"error\\", \\"-y\\", \\"-i\\", str(video)]\\n    if max_seconds > 0:\\n        cmd += [\\"-t\\", str(max_seconds)]\\n    out = str(out_dir / \\"%05d.png\\")\\n    # -fps_mode needs ffmpeg 5.1+; Colab ships 4.4, which only knows -vsync.\\n    try:\\n        subprocess.run(cmd + [\\"-vf\\", vf, \\"-fps_mode\\", \\"vfr\\", out], check=True, capture_output=True)\\n    except subprocess.CalledProcessError:\\n        subprocess.run(cmd + [\\"-vf\\", vf, \\"-vsync\\", \\"vfr\\", out], check=True)\\n    return len(list(out_dir.glob(\\"*.png\\")))\\n\\n\\ndef _load(path: Path, device) -> torch.Tensor:\\n    arr = np.asarray(Image.open(path).convert(\\"RGB\\"), dtype=np.float32) / 255.0\\n    return torch.from_numpy(arr).permute(2, 0, 1)[None].to(device)\\n\\n\\ndef _save(t: torch.Tensor, path: Path) -> None:\\n    arr = (t[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255 + 0.5).astype(np.uint8)\\n    Image.fromarray(arr).save(path)\\n\\n\\ndef load_ref(path: Path, size: tuple[int, int], device) -> torch.Tensor:\\n    \\"\\"\\"Reference photo, centre-cropped to the frame aspect and resized to the frame size.\\"\\"\\"\\n    img = Image.open(path).convert(\\"RGB\\")\\n    w, h = size\\n    s = max(w / img.width, h / img.height)\\n    img = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)\\n    left, top = (img.width - w) // 2, (img.height - h) // 2\\n    img = img.crop((left, top, left + w, top + h))\\n    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)[None].to(device)\\n\\n\\ndef restore_frames(model, in_dir: Path, out_dir: Path, ref: torch.Tensor | None = None, device=\\"cuda\\", log=print) -> float:\\n    \\"\\"\\"Restore every frame not already in ``out_dir``. Returns seconds per frame.\\"\\"\\"\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    todo = [p for p in sorted(in_dir.glob(\\"*.png\\")) if not (out_dir / p.name).exists()]\\n    if not todo:\\n        return 0.0\\n    log(f\\"[video] restoring {len(todo)} frames\\")\\n    t0, n = time.time(), 0\\n    for i, p in enumerate(todo):\\n        x = _load(p, device)\\n        h, w = x.shape[-2:]\\n        # The UNet wants latent sides divisible by 8, i.e. image sides divisible by 64.\\n        ph, pw = (-h) % 64, (-w) % 64\\n        xp = F.pad(x, (0, pw, 0, ph), mode=\\"reflect\\") if ph or pw else x\\n        rp = F.pad(ref, (0, pw, 0, ph), mode=\\"reflect\\") if ref is not None and (ph or pw) else ref\\n        y = model.restore(xp, rp)[..., :h, :w]\\n        _save(y, out_dir / p.name)\\n        n += 1\\n        if i == 0 or (i + 1) % 25 == 0 or i + 1 == len(todo):\\n            log(f\\"[video] {i + 1}/{len(todo)}  {(time.time() - t0) / n:.2f} s/frame\\")\\n    return (time.time() - t0) / max(n, 1)\\n\\n\\ndef encode(frames_dir: Path, fps: float, out: Path) -> None:\\n    subprocess.run(\\n        [\\"ffmpeg\\", \\"-v\\", \\"error\\", \\"-y\\", \\"-framerate\\", f\\"{fps:.6f}\\", \\"-i\\", str(frames_dir / \\"%05d.png\\"),\\n         \\"-c:v\\", \\"libx264\\", \\"-crf\\", \\"16\\", \\"-preset\\", \\"medium\\", \\"-pix_fmt\\", \\"yuv420p\\", str(out)],\\n        check=True,\\n    )\\n\\n\\ndef _label(img: Image.Image, text: str) -> Image.Image:\\n    d = ImageDraw.Draw(img)\\n    pad = max(4, img.height // 120)\\n    box = d.textbbox((0, 0), text)\\n    d.rectangle((0, 0, box[2] + 2 * pad, box[3] + 2 * pad), fill=(0, 0, 0))\\n    d.text((pad, pad), text, fill=(255, 255, 255))\\n    return img\\n\\n\\ndef side_by_side(before_dir: Path, after_dir: Path, out_dir: Path) -> None:\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    for p in sorted(after_dir.glob(\\"*.png\\")):\\n        if (out_dir / p.name).exists():\\n            continue\\n        a = _label(Image.open(before_dir / p.name).convert(\\"RGB\\"), \\"ONCE (render)\\")\\n        b = _label(Image.open(p).convert(\\"RGB\\"), \\"SONRA (Difix)\\")\\n        canvas = Image.new(\\"RGB\\", (a.width + b.width, a.height))\\n        canvas.paste(a, (0, 0))\\n        canvas.paste(b, (a.width, 0))\\n        canvas.save(out_dir / p.name)\\n\\n\\ndef _most_changed_box(a: np.ndarray, b: np.ndarray, size: int) -> tuple[int, int]:\\n    diff = torch.from_numpy(np.abs(a - b).mean(-1))[None, None]\\n    pooled = F.avg_pool2d(diff, size, stride=max(1, size // 4))\\n    idx = int(pooled.flatten().argmax())\\n    ny, nx = pooled.shape[-2:]\\n    step = max(1, size // 4)\\n    return (idx % nx) * step, (idx // nx) * step\\n\\n\\ndef comparison_sheet(before_dir: Path, after_dir: Path, out: Path, rows: int = 4) -> None:\\n    \\"\\"\\"Rows of: before | after | zoomed before | zoomed after, zoomed where the model changed most.\\"\\"\\"\\n    names = sorted(p.name for p in after_dir.glob(\\"*.png\\"))\\n    picks = [names[i] for i in np.linspace(0, len(names) - 1, min(rows, len(names))).round().astype(int)]\\n    tiles = []\\n    for name in picks:\\n        a = Image.open(before_dir / name).convert(\\"RGB\\")\\n        b = Image.open(after_dir / name).convert(\\"RGB\\")\\n        crop = max(32, min(a.width, a.height) // 3)\\n        x, y = _most_changed_box(np.asarray(a, np.float32), np.asarray(b, np.float32), crop)\\n        za = a.crop((x, y, x + crop, y + crop)).resize((a.height, a.height), Image.NEAREST)\\n        zb = b.crop((x, y, x + crop, y + crop)).resize((a.height, a.height), Image.NEAREST)\\n        d = ImageDraw.Draw(a)\\n        d.rectangle((x, y, x + crop, y + crop), outline=(255, 0, 0), width=max(2, a.height // 200))\\n        row = [_label(a, \\"once\\"), _label(b, \\"sonra\\"), _label(za, \\"once (zoom)\\"), _label(zb, \\"sonra (zoom)\\")]\\n        canvas = Image.new(\\"RGB\\", (sum(t.width for t in row), a.height))\\n        xo = 0\\n        for t in row:\\n            canvas.paste(t, (xo, 0))\\n            xo += t.width\\n        tiles.append(canvas)\\n    sheet = Image.new(\\"RGB\\", (tiles[0].width, sum(t.height for t in tiles)))\\n    yo = 0\\n    for t in tiles:\\n        sheet.paste(t, (0, yo))\\n        yo += t.height\\n    if sheet.width > 3200:\\n        sheet = sheet.resize((3200, round(sheet.height * 3200 / sheet.width)), Image.LANCZOS)\\n    sheet.save(out, quality=92)\\n\\n\\ndef _gray(path: Path) -> np.ndarray:\\n    return np.asarray(Image.open(path).convert(\\"L\\"), dtype=np.float32) / 255.0\\n\\n\\ndef _laplacian_var(g: np.ndarray) -> float:\\n    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]\\n    return float(lap.var())\\n\\n\\ndef metrics(before_dir: Path, after_dir: Path) -> dict:\\n    names = sorted(p.name for p in after_dir.glob(\\"*.png\\"))\\n    change_psnr, sharp_ratio, flick_b, flick_a = [], [], [], []\\n    prev_b = prev_a = None\\n    for name in names:\\n        gb, ga = _gray(before_dir / name), _gray(after_dir / name)\\n        mse = float(((gb - ga) ** 2).mean())\\n        change_psnr.append(10 * np.log10(1.0 / max(mse, 1e-10)))\\n        sharp_ratio.append(_laplacian_var(ga) / max(_laplacian_var(gb), 1e-10))\\n        if prev_b is not None:\\n            flick_b.append(float(np.abs(gb - prev_b).mean()))\\n            flick_a.append(float(np.abs(ga - prev_a).mean()))\\n        prev_b, prev_a = gb, ga\\n    fb, fa = float(np.mean(flick_b)) if flick_b else 0.0, float(np.mean(flick_a)) if flick_a else 0.0\\n    return {\\n        \\"frames\\": len(names),\\n        \\"change_psnr_db\\": float(np.mean(change_psnr)),\\n        \\"sharpness_ratio\\": float(np.median(sharp_ratio)),\\n        \\"flicker_before\\": fb,\\n        \\"flicker_after\\": fa,\\n        \\"flicker_ratio\\": fa / fb if fb > 0 else float(\\"nan\\"),\\n    }\\n\\n\\ndef summary_markdown(m: dict, info: dict) -> str:\\n    fr = m[\\"flicker_ratio\\"]\\n    return \\"\\\\n\\".join([\\n        f\\"## Difix video testi: {info[\'video\']}\\",\\n        \\"\\",\\n        f\\"* Model: `{info[\'repo\']}`\\" + (f\\", referans: `{info[\'ref\']}`\\" if info.get(\\"ref\\") else \\"\\"),\\n        f\\"* İşlenen kare: {m[\'frames\']} ({info[\'size\'][0]}x{info[\'size\'][1]}), {info[\'sec_per_frame\']:.2f} s/kare\\",\\n        \\"\\",\\n        \\"| ölçüm | değer | nasıl okunur |\\",\\n        \\"|---|---|---|\\",\\n        f\\"| değişim (önce↔sonra PSNR) | {m[\'change_psnr_db\']:.1f} dB | 40+ dB: neredeyse dokunmamış, 25-35: belirgin düzeltme, <22: çok şey uydurmuş olabilir |\\",\\n        f\\"| keskinlik oranı | {m[\'sharpness_ratio\']:.2f}x | >1 daha keskin; tek başına kalite demek değil (gürültü de keskinlik sayılır) |\\",\\n        f\\"| titreme oranı | {fr:.2f}x | ~1 iyi; 1.2+ ise kareler arası titreme artmış (kare kare işlemenin bilinen yan etkisi) |\\",\\n        \\"\\",\\n        \\"Gerçek fotoğraf olmadığı için bu sayılar doğruluk ölçmez; asıl karar `side_by_side.mp4` ve \\"\\n        \\"`compare.jpg`\'deki zoom\'lara bakarak verilir (kırmızı kutu: modelin en çok değiştirdiği bölge).\\",\\n    ])\\n\\n\\ndef run(\\n    video: Path,\\n    out_dir: Path,\\n    model,\\n    repo: str,\\n    work_dir: Path = Path(\\"/content/difix_video\\"),\\n    ref_image: Path | None = None,\\n    long_edge: int = 1024,\\n    stride: int = 1,\\n    max_seconds: float = 0,\\n    device: str = \\"cuda\\",\\n    log=print,\\n) -> dict:\\n    video, out_dir = Path(video), Path(out_dir)\\n    info = probe(video)\\n    size = target_size(info[\\"width\\"], info[\\"height\\"], long_edge)\\n    work = Path(work_dir) / f\\"{video.stem}_{size[0]}x{size[1]}_s{stride}_t{max_seconds:g}\\"\\n    before, after = work / \\"before\\", work / f\\"after_{repo.split(\'/\')[-1]}\\"\\n    if not any(before.glob(\\"*.png\\")):\\n        n = extract_frames(video, before, size, stride, max_seconds)\\n        log(f\\"[video] {info[\'width\']}x{info[\'height\']} @ {info[\'fps\']:.2f} fps -> {n} frames at {size[0]}x{size[1]}\\")\\n    ref = load_ref(Path(ref_image), size, device) if ref_image else None\\n    spf = restore_frames(model, before, after, ref=ref, device=device, log=log)\\n    timing = after.parent / f\\"{after.name}_timing.json\\"\\n    if spf:\\n        timing.write_text(json.dumps({\\"sec_per_frame\\": spf}), encoding=\\"utf-8\\")\\n    elif timing.exists():\\n        spf = json.loads(timing.read_text(encoding=\\"utf-8\\"))[\\"sec_per_frame\\"]\\n\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    fps = info[\\"fps\\"] / stride\\n    encode(after, fps, out_dir / \\"restored.mp4\\")\\n    sbs = work / \\"side_by_side\\"\\n    side_by_side(before, after, sbs)\\n    encode(sbs, fps, out_dir / \\"side_by_side.mp4\\")\\n    comparison_sheet(before, after, out_dir / \\"compare.jpg\\")\\n    m = metrics(before, after)\\n    meta = {\\"video\\": video.name, \\"repo\\": repo, \\"ref\\": Path(ref_image).name if ref_image else None,\\n            \\"size\\": size, \\"stride\\": stride, \\"sec_per_frame\\": spf, **m}\\n    (out_dir / \\"metrics.json\\").write_text(json.dumps(meta, indent=2), encoding=\\"utf-8\\")\\n    md = summary_markdown(m, meta)\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return {\\"out_dir\\": out_dir, \\"work\\": work, \\"metrics\\": meta}\\n\\n\\ndef cleanup(work: Path) -> None:\\n    shutil.rmtree(work, ignore_errors=True)\\n", "lib/splat_restorer/workspace.py": "\\"\\"\\"Folder layout. Durable outputs live on Drive; scratch work on local disk.\\n\\nDrive (``root``)::\\n\\n    split.json                    scene split (written once, then reused)\\n    pairs/<scene>.tar             all pairs of one scene (atomic, = done marker)\\n    splats/<scene>/<recipe>.pt    degraded splats of *test* scenes\\n    runs/<run>/...                restorer / baseline checkpoints and logs\\n    results/...                   evaluation tables, contact sheets\\n\\nLocal (``local``)::\\n\\n    raw/<scene>/                  downloaded scene, deleted after use\\n    pairs/                        extracted pair tars for training / eval\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nimport shutil\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\n\\n@dataclass\\nclass Workspace:\\n    root: Path\\n    local: Path\\n\\n    def __post_init__(self):\\n        self.root = Path(self.root)\\n        self.local = Path(self.local)\\n        for d in (self.root, self.pairs_dir, self.splats_dir, self.runs_dir, self.results_dir, self.local):\\n            d.mkdir(parents=True, exist_ok=True)\\n\\n    @property\\n    def split_path(self) -> Path:\\n        return self.root / \\"split.json\\"\\n\\n    @property\\n    def pairs_dir(self) -> Path:\\n        return self.root / \\"pairs\\"\\n\\n    @property\\n    def splats_dir(self) -> Path:\\n        return self.root / \\"splats\\"\\n\\n    @property\\n    def runs_dir(self) -> Path:\\n        return self.root / \\"runs\\"\\n\\n    @property\\n    def results_dir(self) -> Path:\\n        return self.root / \\"results\\"\\n\\n    def pair_tar(self, scene: str) -> Path:\\n        return self.pairs_dir / f\\"{scene}.tar\\"\\n\\n    def raw_dir(self, scene: str) -> Path:\\n        return self.local / \\"raw\\" / scene\\n\\n    @property\\n    def local_pairs(self) -> Path:\\n        return self.local / \\"pairs\\"\\n\\n    def splat_path(self, scene: str, recipe: str) -> Path:\\n        return self.splats_dir / scene / f\\"{recipe}.pt\\"\\n\\n\\ndef atomic_copy(src: Path, dst: Path) -> None:\\n    \\"\\"\\"Copy to ``dst`` via a temp name so a crash never leaves a half file.\\"\\"\\"\\n    dst.parent.mkdir(parents=True, exist_ok=True)\\n    tmp = dst.with_name(dst.name + \\".partial\\")\\n    shutil.copyfile(src, tmp)\\n    os.replace(tmp, dst)\\n\\n\\ndef atomic_write_json(path: Path, data) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    tmp = path.with_name(path.name + \\".partial\\")\\n    tmp.write_text(json.dumps(data, indent=2), encoding=\\"utf-8\\")\\n    os.replace(tmp, path)\\n"}')
root = Path("/content/ab_src")
for name, text in SOURCES.items():
    p = root / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
for sub in ("app", "lib"):
    if str(root / sub) not in sys.path:
        sys.path.insert(0, str(root / sub))

from backend.model.gaussian_model import GaussianModel
from backend.model.deformation import DeformationField
from backend.model.trainer import Trainer4DGS
from backend.model.renderer import render_view
from backend.export.to_splat import export_to_ply
from backend.preprocess.parse_colmap import scene_extent
from splat_restorer import own_splat, train_ab
app = types.SimpleNamespace(GaussianModel=GaussianModel, DeformationField=DeformationField, Trainer4DGS=Trainer4DGS,
                            export_to_ply=export_to_ply, scene_extent=scene_extent)
print(f"{len(SOURCES)} dosya yazıldı; app eğiticisi feature/scene-composer @ 163d96c")

In [ ]:
# @title Drive + veri (+ temiz kareler)
import shutil, zipfile, psutil
from google.colab import drive
drive.mount("/content/drive")
out_dir = Path(OUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)

raw = Path("/content/own_dataset")
if not (raw / ".unzipped").exists():
    print("Zip açılıyor...")
    shutil.rmtree(raw, ignore_errors=True)
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(raw)
    (raw / ".unzipped").touch()
cams_bin = next(raw.rglob("sparse/*/cameras.bin"))
src, model_name = cams_bin.parents[2], cams_bin.parent.name
data = own_splat.prepare_dataset(src, Path(f"/content/own_prepared_{LONG_EDGE}"), long_edge=LONG_EDGE,
                                 model_name=model_name, workers=os.cpu_count() or 4)
clean_dir = train_ab.clean_frames(src, data, Path(f"/content/own_clean_{LONG_EDGE}_h{NLM_H:g}"), model_name, nlm_h=NLM_H,
                                  workers=os.cpu_count() or 4)
holdout = list(range(0, data.num_frames, HOLDOUT_EVERY))
xyz, rgb, init_ids = train_ab.init_points(src, model_name, SEED)
need = data.num_frames * data.width * data.height * 3 / 2**30
print(f"{data.num_frames} kare {data.width}x{data.height}, saklanan {len(holdout)}; {len(xyz):,} nokta")
print(f"Eğitim RAM önbelleği ~{need:.1f} GiB; boş RAM {psutil.virtual_memory().available / 2**30:.1f} GiB")
assert need < psutil.virtual_memory().available / 2**30 * 0.6, "RAM yetersiz: Runtime → Change runtime type → High-RAM"

In [ ]:
# @title GPU kontrolü (ilk seferde gsplat derlenir, ~3-5 dk)
import time, torch
assert torch.cuda.is_available(), "GPU yok: Runtime → Change runtime type → GPU"
t0 = time.time()
m = torch.tensor([[0., 0., 2.]], device="cuda", requires_grad=True)
r, _, _ = render_view(m, torch.tensor([[1., 0, 0, 0]], device="cuda"), torch.full((1, 3), .1, device="cuda"),
                      torch.tensor([.5], device="cuda"), torch.full((1, 3), .5, device="cuda"),
                      torch.tensor([[50., 0, 32], [0, 50., 32], [0, 0, 1.]], device="cuda"), torch.eye(4, device="cuda"),
                      64, 64, sh_degree=0)
r.sum().backward()
assert m.grad is not None and torch.isfinite(m.grad).all()
print(f"GPU render/backward OK ({time.time() - t0:.0f} s, ilk derleme dahil)")

In [ ]:
# @title Eğit + ölç (varyant başına ~10-30 dk)
import dataclasses, lpips
from IPython.display import Markdown, display
net_lpips = lpips.LPIPS(net="alex", verbose=False).cuda().eval()
results = {}
for name in VARIANTS:
    res_path = out_dir / name / "result.json"
    if res_path.exists():
        results[name] = json.loads(res_path.read_text())
        print(f"{name}: daha önce bitmiş, atlandı")
        continue
    v = train_ab.VARIANTS[name]
    if STEPS_OVERRIDE > 0:
        v = dataclasses.replace(v, steps=STEPS_OVERRIDE * (2 if v.steps > 30000 else 1))
    print(f"\n===== {name}: {v.note} ({v.steps:,} adım)")
    work = Path("/content/ab_work") / name
    shutil.rmtree(work, ignore_errors=True)
    frames_dir = clean_dir if v.clean else data.frames_dir
    ply, info = train_ab.train_variant(v, data, frames_dir, xyz, rgb, init_ids, holdout, work, app, seed=SEED)
    (out_dir / name).mkdir(parents=True, exist_ok=True)
    shutil.copy2(ply, out_dir / name / "final.ply")
    frames = train_ab.evaluate_ply(ply, data, holdout, clean_dir, net_lpips, out_dir / name / "renders")
    results[name] = {"info": info, "frames": frames}
    train_ab.save_json(res_path, results[name])
    s = train_ab.summarise({name: results[name]})[name]
    print(f"{name}: PSNR {s['psnr']:.2f}  LPIPS {s['lpips']:.4f}  temiz-foto PSNR {s['clean_psnr']:.2f}  "
          f"{s['splats']:,} splat  {s['minutes']:.0f} dk")

In [ ]:
# @title Sonuçlar
from IPython.display import Image as IPImage
summary = train_ab.summarise(results)
md_text = train_ab.markdown(summary, len(holdout))
(out_dir / "summary.md").write_text(md_text, encoding="utf-8")
train_ab.save_json(out_dir / "summary.json", summary)
display(Markdown(md_text))
sheets = train_ab.comparison_sheets(results, data, out_dir, out_dir)
print("Sütunlar: foto | " + " | ".join(results) + "; alt sıra: kırmızı kutunun zoom'u")
for p in sheets[:6]:
    display(IPImage(filename=str(p), width=1600))

## Nasıl okunur?

* **PSNR / LPIPS** saklanan fotoğraflara karşıdır; **Δ** baseline'a göre farktır. +0,3 dB ve üstü belirgin bir iyileşmedir.
* **Temiz foto PSNR:** fotoğrafların gürültüsü azaltılmış hâline karşı. `lr_decay_clean` orijinal fotoğrafa karşı
  gürültüyü taklit etmediği için orada geride görünebilir; bu sütunda adil karşılaştırılır.
* `baseline` ~25,3 dB değilse kurulumda bir fark var demektir; diğer sonuçlara güvenmeden önce bana haber ver.

Bana `summary.md` ve birkaç `compare_*.jpg` getirmen yeterli (Drive'da `OUT_DIR` altında).